In [1]:
# ================================================================
# T1.4c — BLACK-BASELINE SENSITIVITY ANALYSIS
# ================================================================
#
# Scientific question:
# Does the observed Integrated Gradients enrichment trend depend
# on the Gaussian-blur baseline, or does it persist when IG uses
# a black-image baseline?
#
# LOCKED PRINCIPLE:
# Everything in the production XAI pipeline remains unchanged
# except the IG baseline.
#
# This notebook MUST use the actual production/T1.4a pipeline.
# It MUST NOT depend on t1_4_common.py.
#
# Design:
#   30 production models
#   6 rho levels × 5 replicates
#   15 frozen XAI images per model
#   2 baselines:
#       1. Production Gaussian-blur baseline
#       2. Black-image baseline
#
# Primary sensitivity comparison:
#   - paired enrichment: Gaussian vs Black
#   - model-level rho/enrichment association
#   - direction and magnitude of rho association
#   - completeness behavior
#
# Status:
#   Exploratory / sensitivity analysis.
#   Does NOT replace the locked primary Gaussian-baseline endpoint.
# ================================================================

T14C_PROTOCOL = "T1.4c_black_baseline_sensitivity_v1"

MASTER_SEED = 20260925
SUBSAMPLE_SEED = 20260924

N_MODELS = 30
N_IMAGES_PER_MODEL = 15

RHO_LEVELS = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
N_REPLICATES = 5

assert N_MODELS == len(RHO_LEVELS) * N_REPLICATES

print("=" * 70)
print("T1.4c — BLACK-BASELINE SENSITIVITY ANALYSIS")
print("=" * 70)

print(f"Protocol          : {T14C_PROTOCOL}")
print(f"Master seed       : {MASTER_SEED}")
print(f"Subsample seed    : {SUBSAMPLE_SEED}")
print(f"Models            : {N_MODELS}")
print(f"Rho levels        : {RHO_LEVELS}")
print(f"Replicates/rho    : {N_REPLICATES}")
print(f"Images/model      : {N_IMAGES_PER_MODEL}")
print(f"Model-image pairs : {N_MODELS * N_IMAGES_PER_MODEL}")
print(f"IG evaluations    : {N_MODELS * N_IMAGES_PER_MODEL * 2}")

print("=" * 70)
print("PROTOCOL LOCK INITIALIZED")
print("=" * 70)

T1.4c — BLACK-BASELINE SENSITIVITY ANALYSIS
Protocol          : T1.4c_black_baseline_sensitivity_v1
Master seed       : 20260925
Subsample seed    : 20260924
Models            : 30
Rho levels        : [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
Replicates/rho    : 5
Images/model      : 15
Model-image pairs : 450
IG evaluations    : 900
PROTOCOL LOCK INITIALIZED


In [2]:
# ================================================================
# CELL 2 — PRODUCTION PROJECT DISCOVERY
# ================================================================
#
# Purpose:
#   Locate the actual frozen production/T1.4a implementation,
#   checkpoints, XAI subset, masks, and analysis files.
#
# IMPORTANT:
#   No files are modified.
#   No models are loaded.
#   No GPU computation is performed.
# ================================================================

from pathlib import Path
import os

print("=" * 70)
print("CELL 2 — PRODUCTION PROJECT DISCOVERY")
print("=" * 70)

# ---------------------------------------------------------------
# 1. Kaggle input roots
# ---------------------------------------------------------------

INPUT_ROOT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working")

print("\nInput root :", INPUT_ROOT)
print("Work root  :", WORK_ROOT)

assert INPUT_ROOT.exists(), "Kaggle input directory not found."
assert WORK_ROOT.exists(), "Kaggle working directory not found."

# ---------------------------------------------------------------
# 2. Discover Python files
# ---------------------------------------------------------------

py_files = sorted(INPUT_ROOT.rglob("*.py"))

print(f"\nPython files found: {len(py_files)}")

for p in py_files[:100]:
    print("  ", p)

if len(py_files) > 100:
    print(f"  ... {len(py_files) - 100} more")

# ---------------------------------------------------------------
# 3. Discover likely production / T1.4 files
# ---------------------------------------------------------------

keywords = [
    "analysis_pipeline",
    "train",
    "xai",
    "ig",
    "parameter",
    "random",
    "hair",
    "mask",
    "generator",
    "model",
]

candidate_files = []

for p in py_files:
    name = p.name.lower()
    if any(k in name for k in keywords):
        candidate_files.append(p)

print("\n" + "=" * 70)
print("LIKELY RELEVANT PYTHON FILES")
print("=" * 70)

for p in candidate_files:
    print("  ", p)

# ---------------------------------------------------------------
# 4. Discover CSV / JSON / NPZ / checkpoint-like files
# ---------------------------------------------------------------

extensions = {
    ".csv",
    ".json",
    ".npz",
    ".pt",
    ".pth",
    ".ckpt",
    ".yaml",
    ".yml",
}

data_files = []

for p in INPUT_ROOT.rglob("*"):
    if p.is_file() and p.suffix.lower() in extensions:
        data_files.append(p)

print("\n" + "=" * 70)
print("DATA / CHECKPOINT FILES")
print("=" * 70)

print(f"Found: {len(data_files)}")

for p in sorted(data_files)[:150]:
    print("  ", p)

if len(data_files) > 150:
    print(f"  ... {len(data_files) - 150} more")

# ---------------------------------------------------------------
# 5. Explicitly locate the frozen primary endpoint table
# ---------------------------------------------------------------

endpoint_candidates = list(
    INPUT_ROOT.rglob("model_endpoints_final_checkpoint.csv")
)

print("\n" + "=" * 70)
print("FROZEN PRIMARY ENDPOINT TABLE")
print("=" * 70)

if endpoint_candidates:
    for p in endpoint_candidates:
        print("  FOUND:", p)
else:
    print("  NOT FOUND")

# ---------------------------------------------------------------
# 6. Explicitly locate T1.4a-related files
# ---------------------------------------------------------------

t14a_candidates = []

for p in INPUT_ROOT.rglob("*"):
    if p.is_file():
        name = p.name.lower()
        path_str = str(p).lower()

        if (
            "t1_4a" in name
            or "t1_4a" in path_str
            or "parameter_random" in name
            or "randomization" in name
        ):
            t14a_candidates.append(p)

print("\n" + "=" * 70)
print("T1.4a / PARAMETER-RANDOMIZATION FILES")
print("=" * 70)

for p in sorted(set(t14a_candidates)):
    print("  ", p)

# ---------------------------------------------------------------
# 7. Do NOT import anything yet.
# ---------------------------------------------------------------

print("\n" + "=" * 70)
print("DISCOVERY COMPLETE")
print("=" * 70)

print("""
No production code has been imported.
No checkpoint has been loaded.
No model has been modified.
No IG computation has been performed.

Next step:
we will inspect the discovered production/T1.4a files
and identify the exact implementation that must be reused.
""")

CELL 2 — PRODUCTION PROJECT DISCOVERY

Input root : /kaggle/input
Work root  : /kaggle/working

Python files found: 16
   /kaggle/input/datasets/tejasseshadriiiii/isic-final-project/step1J_generator_recovery_lock/recovered_original_hair_generator.py
   /kaggle/input/datasets/tejasseshadriiiii/isic-final-project/step2Q_solver_implementation_v2/deterministic_evaluation_solver.py
   /kaggle/input/datasets/tejasseshadriiiii/isic-final-project/step2W_analysis_code_freeze/analysis_pipeline.py
   /kaggle/input/datasets/tejasseshadriiiii/isic-final-project/step2X_training_code_freeze/train_one_model.py
   /kaggle/input/notebooks/tejasseshadriiiii/isic-kaggle-analysis/isic_final_project/step1J_generator_recovery_lock/recovered_original_hair_generator.py
   /kaggle/input/notebooks/tejasseshadriiiii/isic-kaggle-analysis/isic_final_project/step2Q_solver_implementation_v2/deterministic_evaluation_solver.py
   /kaggle/input/notebooks/tejasseshadriiiii/isic-kaggle-analysis/isic_final_project/step2W_a

In [3]:
# ================================================================
# CELL 3 — INSPECT AND FINGERPRINT FROZEN PRODUCTION CODE
# ================================================================
#
# Purpose:
#   Verify the exact production analysis implementation before
#   importing or modifying anything.
#
# IMPORTANT:
#   We do NOT use t1_4_common.py.
#   We do NOT modify analysis_pipeline.py.
#   We do NOT load a model.
#   We do NOT run Integrated Gradients.
# ================================================================

from pathlib import Path
import hashlib
import inspect
import ast

print("=" * 70)
print("CELL 3 — FROZEN PRODUCTION CODE INSPECTION")
print("=" * 70)

# ---------------------------------------------------------------
# 1. Exact production paths discovered in Cell 2
# ---------------------------------------------------------------

PROJECT_ROOT = Path(
    "/kaggle/input/notebooks/tejasseshadriiiii/"
    "isic-kaggle-analysis/isic_final_project"
)

ANALYSIS_CODE = (
    PROJECT_ROOT
    / "step2W_analysis_code_freeze"
    / "analysis_pipeline.py"
)

TRAINING_CODE = (
    PROJECT_ROOT
    / "step2X_training_code_freeze"
    / "train_one_model.py"
)

GENERATOR_CODE = (
    PROJECT_ROOT
    / "step1J_generator_recovery_lock"
    / "recovered_original_hair_generator.py"
)

XAI_SUBSET = (
    PROJECT_ROOT
    / "step2U_xai_subset_lock"
    / "LOCKED_XAI_SUBSET.csv"
)

ENDPOINT_TABLE = (
    Path("/kaggle/input/notebooks/tejasseshadriiiii/")
    / "isic-kaggle-analysis"
    / "results"
    / "model_endpoints_final_checkpoint.csv"
)

for path in [
    ANALYSIS_CODE,
    TRAINING_CODE,
    GENERATOR_CODE,
    XAI_SUBSET,
    ENDPOINT_TABLE,
]:
    print(f"{'FOUND' if path.exists() else 'MISSING'} : {path}")

assert ANALYSIS_CODE.exists()
assert TRAINING_CODE.exists()
assert GENERATOR_CODE.exists()
assert XAI_SUBSET.exists()
assert ENDPOINT_TABLE.exists()

# ---------------------------------------------------------------
# 2. SHA-256 helper
# ---------------------------------------------------------------

def sha256_file(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)

    return h.hexdigest()

analysis_hash = sha256_file(ANALYSIS_CODE)
training_hash = sha256_file(TRAINING_CODE)
generator_hash = sha256_file(GENERATOR_CODE)
subset_hash = sha256_file(XAI_SUBSET)

print("\n" + "=" * 70)
print("FILE FINGERPRINTS")
print("=" * 70)

print("analysis_pipeline.py SHA256:")
print(analysis_hash)

print("\ntrain_one_model.py SHA256:")
print(training_hash)

print("\nrecovered_original_hair_generator.py SHA256:")
print(generator_hash)

print("\nLOCKED_XAI_SUBSET.csv SHA256:")
print(subset_hash)

# ---------------------------------------------------------------
# 3. Parse analysis_pipeline.py without executing it
# ---------------------------------------------------------------

source = ANALYSIS_CODE.read_text(encoding="utf-8")

tree = ast.parse(source)

function_names = [
    node.name
    for node in tree.body
    if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef))
]

print("\n" + "=" * 70)
print("FUNCTIONS IN FROZEN analysis_pipeline.py")
print("=" * 70)

for name in function_names:
    print("  ", name)

assert "ig_enrichment" in function_names, (
    "Frozen analysis_pipeline.py does not contain ig_enrichment()."
)

# ---------------------------------------------------------------
# 4. Extract exact ig_enrichment source
# ---------------------------------------------------------------

ig_node = next(
    node
    for node in tree.body
    if isinstance(node, ast.FunctionDef)
    and node.name == "ig_enrichment"
)

ig_source = ast.get_source_segment(source, ig_node)

print("\n" + "=" * 70)
print("EXACT FROZEN ig_enrichment()")
print("=" * 70)

print(ig_source)

# ---------------------------------------------------------------
# 5. Identify important production constants
# ---------------------------------------------------------------

important_names = [
    "IG_STEPS",
    "IG_RETRY_STEPS",
    "IG_SIGMA",
    "IG_TOL_REL",
    "IG_TAU_ABS",
    "XAI_EPS",
    "_MEAN",
    "_STD",
]

print("\n" + "=" * 70)
print("PRODUCTION CONSTANTS FOUND IN SOURCE")
print("=" * 70)

for name in important_names:
    found = False

    for node in tree.body:
        if isinstance(node, ast.Assign):
            for target in node.targets:
                if isinstance(target, ast.Name) and target.id == name:
                    value = ast.get_source_segment(source, node.value)
                    print(f"{name} = {value}")
                    found = True

    if not found:
        print(f"{name} = NOT FOUND AT MODULE LEVEL")

# ---------------------------------------------------------------
# 6. Search source for the key production pipeline components
# ---------------------------------------------------------------

required_terms = [
    "Gaussian",
    "gaussian_blur",
    "completeness",
    "enrichment",
    "hair",
    "mask",
]

print("\n" + "=" * 70)
print("KEY TERMS IN analysis_pipeline.py")
print("=" * 70)

for term in required_terms:
    print(
        f"{term:20s}: "
        f"{'FOUND' if term.lower() in source.lower() else 'NOT FOUND'}"
    )

# ---------------------------------------------------------------
# 7. Final status
# ---------------------------------------------------------------

print("\n" + "=" * 70)
print("CELL 3 STATUS")
print("=" * 70)

print("Frozen production analysis code located : PASS")
print("Frozen production IG function located  : PASS")
print("Production code fingerprinted          : PASS")
print("No model loaded                         : YES")
print("No IG computation performed             : YES")
print("t1_4_common.py used                    : NO")

print("\nREADY FOR CELL 4")

CELL 3 — FROZEN PRODUCTION CODE INSPECTION
FOUND : /kaggle/input/notebooks/tejasseshadriiiii/isic-kaggle-analysis/isic_final_project/step2W_analysis_code_freeze/analysis_pipeline.py
FOUND : /kaggle/input/notebooks/tejasseshadriiiii/isic-kaggle-analysis/isic_final_project/step2X_training_code_freeze/train_one_model.py
FOUND : /kaggle/input/notebooks/tejasseshadriiiii/isic-kaggle-analysis/isic_final_project/step1J_generator_recovery_lock/recovered_original_hair_generator.py
FOUND : /kaggle/input/notebooks/tejasseshadriiiii/isic-kaggle-analysis/isic_final_project/step2U_xai_subset_lock/LOCKED_XAI_SUBSET.csv
FOUND : /kaggle/input/notebooks/tejasseshadriiiii/isic-kaggle-analysis/results/model_endpoints_final_checkpoint.csv

FILE FINGERPRINTS
analysis_pipeline.py SHA256:
bcba1d25782f5770fbe629b1d0281f87a4bfde29b343c731bec7cacdb87bcea5

train_one_model.py SHA256:
1ed04738923f842bc70a688913163df557049099d11f612f4d5fec51b7735d68

recovered_original_hair_generator.py SHA256:
32be8092c85458bed31a

In [4]:
# ================================================================
# CELL 4 — RESOLVE BLACK BASELINE + LOCKED SAMPLE STRUCTURE
# ================================================================
from pathlib import Path
import pandas as pd
import numpy as np
import ast
import re

print("=" * 70)
print("CELL 4 — BASELINE SEMANTICS + SAMPLE STRUCTURE AUDIT")
print("=" * 70)

# ----------------------------------------------------------------
# 1. Load frozen production analysis source
# ----------------------------------------------------------------
analysis_source = ANALYSIS_CODE.read_text(encoding="utf-8")

# ----------------------------------------------------------------
# 2. Search the frozen project for explicit black-baseline logic
# ----------------------------------------------------------------
PROJECT_SEARCH_ROOTS = [
    PROJECT_ROOT / "step2W_analysis_code_freeze",
    PROJECT_ROOT / "step2U_xai_subset_lock",
    PROJECT_ROOT / "step1J_generator_recovery_lock",
]

search_terms = [
    "black",
    "baseline",
    "zeros",
    "zero baseline",
    "black_baseline",
    "black baseline",
    "torch.zeros",
    "np.zeros",
    "0.0",
]

print("\n" + "=" * 70)
print("EXPLICIT BLACK-BASELINE SEARCH")
print("=" * 70)

matched_files = []

for root in PROJECT_SEARCH_ROOTS:
    if not root.exists():
        continue

    for path in root.rglob("*"):
        if not path.is_file():
            continue

        # Restrict to text-like project files
        if path.suffix.lower() not in {
            ".py", ".md", ".txt", ".json", ".csv"
        }:
            continue

        try:
            txt = path.read_text(encoding="utf-8", errors="ignore")
        except Exception:
            continue

        lower = txt.lower()

        hits = []
        for term in search_terms:
            if term.lower() in lower:
                hits.append(term)

        if hits:
            matched_files.append((path, sorted(set(hits))))

for path, hits in matched_files:
    print(f"\nFILE: {path}")
    print("MATCHES:", ", ".join(hits))

print(f"\nFiles with relevant baseline/black terms: {len(matched_files)}")

# ----------------------------------------------------------------
# 3. Print context around explicit black-baseline references
# ----------------------------------------------------------------
print("\n" + "=" * 70)
print("BLACK-BASELINE SOURCE CONTEXT")
print("=" * 70)

context_count = 0

for path, hits in matched_files:
    try:
        txt = path.read_text(encoding="utf-8", errors="ignore")
    except Exception:
        continue

    lines = txt.splitlines()

    for i, line in enumerate(lines):
        low = line.lower()

        if (
            "black baseline" in low
            or "black_baseline" in low
            or "black" in low and "baseline" in low
            or "torch.zeros" in low
            or "np.zeros" in low
        ):
            start = max(0, i - 4)
            end = min(len(lines), i + 5)

            print("\n" + "-" * 70)
            print(f"{path}  [lines {start + 1}-{end}]")
            for j in range(start, end):
                print(f"{j + 1:5d}: {lines[j]}")

            context_count += 1

            # Prevent an enormous dump if there are many generic
            # zero/black references.
            if context_count >= 30:
                print("\n[Context display capped at 30 matches]")
                break

    if context_count >= 30:
        break

print(f"\nDisplayed baseline contexts: {context_count}")

# ----------------------------------------------------------------
# 4. Inspect locked XAI subset
# ----------------------------------------------------------------
print("\n" + "=" * 70)
print("LOCKED XAI SUBSET STRUCTURE")
print("=" * 70)

xai_df = pd.read_csv(XAI_SUBSET)

print("Rows:", len(xai_df))
print("Columns:")
for c in xai_df.columns:
    print("  ", c)

print("\nFirst 5 rows:")
print(xai_df.head().to_string(index=False))

print("\nDtypes:")
print(xai_df.dtypes.to_string())

# Candidate identity / metadata columns
candidate_cols = [
    "image_id",
    "lesion_id",
    "image",
    "filename",
    "severity",
    "rho",
    "label",
    "target",
    "split",
]

print("\nCandidate metadata summaries:")
for c in candidate_cols:
    if c in xai_df.columns:
        vals = xai_df[c]
        print(f"\n{c}:")
        print("  dtype:", vals.dtype)
        print("  unique:", vals.nunique(dropna=False))
        if vals.nunique(dropna=False) <= 20:
            print("  values:", vals.value_counts(dropna=False).to_dict())

# ----------------------------------------------------------------
# 5. Inspect frozen primary endpoint table
# ----------------------------------------------------------------
print("\n" + "=" * 70)
print("PRIMARY ENDPOINT TABLE STRUCTURE")
print("=" * 70)

endpoint_df = pd.read_csv(ENDPOINT_TABLE)

print("Rows:", len(endpoint_df))
print("Columns:")
for c in endpoint_df.columns:
    print("  ", c)

print("\nFirst 5 rows:")
print(endpoint_df.head().to_string(index=False))

print("\nDtypes:")
print(endpoint_df.dtypes.to_string())

for c in ["model", "rho", "replicate", "checkpoint"]:
    if c in endpoint_df.columns:
        vals = endpoint_df[c]
        print(
            f"{c:12s} | "
            f"unique={vals.nunique(dropna=False):3d} | "
            f"missing={int(vals.isna().sum()):3d}"
        )

# ----------------------------------------------------------------
# 6. Check primary design balance
# ----------------------------------------------------------------
print("\n" + "=" * 70)
print("PRIMARY DESIGN BALANCE")
print("=" * 70)

required_endpoint_cols = [
    "model",
    "rho",
    "replicate",
]

missing_required = [
    c for c in required_endpoint_cols
    if c not in endpoint_df.columns
]

print("Required metadata missing:", missing_required)

if not missing_required:
    print("\nModels per rho:")
    print(
        endpoint_df.groupby("rho")["model"]
        .nunique()
        .to_string()
    )

    print("\nRows per rho:")
    print(
        endpoint_df.groupby("rho")
        .size()
        .to_string()
    )

    print("\nReplicates per rho:")
    print(
        endpoint_df.groupby("rho")["replicate"]
        .nunique()
        .to_string()
    )

# ----------------------------------------------------------------
# 7. Look specifically for the primary endpoint columns
# ----------------------------------------------------------------
print("\n" + "=" * 70)
print("REQUIRED T1.4c-RELATED ENDPOINTS")
print("=" * 70)

for c in [
    "delta_z_hair",
    "D",
    "D_AUROC",
    "xai_enrichment",
]:
    if c in endpoint_df.columns:
        print(
            f"{c:20s}: FOUND | "
            f"non-null={endpoint_df[c].notna().sum()} | "
            f"unique={endpoint_df[c].nunique(dropna=True)}"
        )
    else:
        print(f"{c:20s}: NOT FOUND")

# ----------------------------------------------------------------
# 8. Check whether the locked subset is identical/reusable across
#    model/rho conditions based on image identity.
# ----------------------------------------------------------------
print("\n" + "=" * 70)
print("LOCKED SUBSET IDENTITY CHECK")
print("=" * 70)

identity_candidates = [
    c for c in ["image_id", "lesion_id", "image", "filename"]
    if c in xai_df.columns
]

if identity_candidates:
    identity_col = identity_candidates[0]

    print("Selected identity column:", identity_col)
    print("Unique identities:", xai_df[identity_col].nunique())

    if "rho" in xai_df.columns:
        print("\nIdentity counts by rho:")
        print(
            xai_df.groupby("rho")[identity_col]
            .nunique()
            .to_string()
        )

    print("\nDuplicate identity rows:")
    dup_count = int(
        xai_df.duplicated(subset=[identity_col]).sum()
    )
    print(dup_count)
else:
    print("No obvious image identity column found.")

# ----------------------------------------------------------------
# 9. Explicit protocol status — DO NOT make a decision here.
# ----------------------------------------------------------------
print("\n" + "=" * 70)
print("CELL 4 STATUS")
print("=" * 70)

print("Frozen analysis source inspected        : PASS")
print("Black-baseline references searched      : PASS")
print("Locked XAI subset inspected             : PASS")
print("Primary endpoint table inspected        : PASS")
print("Primary balance inspected               : PASS")
print("No IG computation                       : YES")
print("No model loading                        : YES")
print("No baseline decision guessed             : YES")

print("\nIMPORTANT:")
print("The black-baseline representation will be LOCKED only after")
print("reviewing the printed source context above.")
print("\nREADY FOR CELL 4 OUTPUT REVIEW")

CELL 4 — BASELINE SEMANTICS + SAMPLE STRUCTURE AUDIT

EXPLICIT BLACK-BASELINE SEARCH

FILE: /kaggle/input/notebooks/tejasseshadriiiii/isic-kaggle-analysis/isic_final_project/step2W_analysis_code_freeze/analysis_pipeline.py
MATCHES: 0.0, baseline, torch.zeros, zeros

FILE: /kaggle/input/notebooks/tejasseshadriiiii/isic-kaggle-analysis/isic_final_project/step1J_generator_recovery_lock/recovered_original_hair_generator.py
MATCHES: 0.0

Files with relevant baseline/black terms: 2

BLACK-BASELINE SOURCE CONTEXT

----------------------------------------------------------------------
/kaggle/input/notebooks/tejasseshadriiiii/isic-kaggle-analysis/isic_final_project/step2W_analysis_code_freeze/analysis_pipeline.py  [lines 211-219]
  211:         o = model(t)
  212:         return o[:, 1] - o[:, 0]
  213: 
  214:     def run(n):
  215:         acc = torch.zeros_like(xn)
  216:         alphas = torch.arange(1, n + 1, device=device, dtype=torch.float32) / n
  217:         for s in range(0, n, chun

In [5]:
# ================================================================
# CELL 5 — RESOLVE FROZEN MODEL / IMAGE / MASK LOADING
# ================================================================

from pathlib import Path
import pandas as pd
import numpy as np
import re
import ast

print("=" * 70)
print("CELL 5 — FROZEN MODEL / IMAGE / MASK LOADING AUDIT")
print("=" * 70)

# ----------------------------------------------------------------
# 1. Load frozen training source
# ----------------------------------------------------------------

training_source = TRAINING_CODE.read_text(encoding="utf-8")

print("\n" + "=" * 70)
print("TRAINING SOURCE — RELEVANT DEFINITIONS")
print("=" * 70)

training_tree = ast.parse(training_source)

training_functions = [
    node.name
    for node in training_tree.body
    if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef))
]

print("Functions:")
for name in training_functions:
    print("  ", name)

# Search for checkpoint/model/data-related definitions
keywords = [
    "checkpoint",
    "model",
    "load",
    "image",
    "mask",
    "dataset",
    "state_dict",
    "resnet",
]

print("\nRelevant source lines:")

shown = 0

for i, line in enumerate(training_source.splitlines(), start=1):
    low = line.lower()

    if any(k in low for k in keywords):
        print(f"{i:5d}: {line}")
        shown += 1

        if shown >= 120:
            print("\n[Output capped at 120 relevant lines]")
            break

# ----------------------------------------------------------------
# 2. Inspect the frozen generator source
# ----------------------------------------------------------------

generator_source = GENERATOR_CODE.read_text(encoding="utf-8")

print("\n" + "=" * 70)
print("FROZEN GENERATOR — PUBLIC FUNCTIONS")
print("=" * 70)

generator_tree = ast.parse(generator_source)

generator_functions = [
    node.name
    for node in generator_tree.body
    if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef))
]

for name in generator_functions:
    print("  ", name)

# ----------------------------------------------------------------
# 3. Locate explicit hair-generator functions
# ----------------------------------------------------------------

print("\n" + "=" * 70)
print("HAIR GENERATOR DEFINITIONS")
print("=" * 70)

for target_name in [
    "generate_hair_mask",
    "apply_hair",
]:
    found_nodes = [
        node
        for node in generator_tree.body
        if isinstance(node, ast.FunctionDef)
        and node.name == target_name
    ]

    if found_nodes:
        node = found_nodes[0]
        print(f"\nFOUND: {target_name}")
        print(ast.get_source_segment(generator_source, node))
    else:
        print(f"\nNOT FOUND: {target_name}")

# ----------------------------------------------------------------
# 4. Inspect endpoint model names
# ----------------------------------------------------------------

print("\n" + "=" * 70)
print("FROZEN PRIMARY MODEL NAMES")
print("=" * 70)

endpoint_df = pd.read_csv(ENDPOINT_TABLE)

model_names = endpoint_df["model"].astype(str).tolist()

print("Number of endpoint models:", len(model_names))
print("Unique model names:", endpoint_df["model"].nunique())

for name in model_names:
    print("  ", name)

# ----------------------------------------------------------------
# 5. Search Kaggle input tree for checkpoint candidates
# ----------------------------------------------------------------

print("\n" + "=" * 70)
print("CHECKPOINT DISCOVERY")
print("=" * 70)

SEARCH_ROOT = Path("/kaggle/input")

checkpoint_extensions = {
    ".pt",
    ".pth",
    ".ckpt",
    ".bin",
}

checkpoint_candidates = []

for path in SEARCH_ROOT.rglob("*"):
    if not path.is_file():
        continue

    if path.suffix.lower() in checkpoint_extensions:
        checkpoint_candidates.append(path)

print("Checkpoint files found:", len(checkpoint_candidates))

for path in checkpoint_candidates[:200]:
    print("  ", path)

if len(checkpoint_candidates) > 200:
    print("\n[Checkpoint display capped at 200 files]")

# ----------------------------------------------------------------
# 6. Search for model names inside discovered paths
# ----------------------------------------------------------------

print("\n" + "=" * 70)
print("MODEL-NAME ↔ CHECKPOINT PATH MATCHING")
print("=" * 70)

for model_name in model_names:

    matches = [
        path
        for path in checkpoint_candidates
        if model_name.lower() in str(path).lower()
    ]

    print(f"\n{model_name}")
    print("  matches:", len(matches))

    for path in matches[:20]:
        print("   ", path)

# ----------------------------------------------------------------
# 7. Inspect possible image/data roots
# ----------------------------------------------------------------

print("\n" + "=" * 70)
print("IMAGE / DATA ROOT DISCOVERY")
print("=" * 70)

data_keywords = [
    "image",
    "images",
    "isic",
    "validation",
    "val",
    "eval",
    "clean",
    "hair",
    "mask",
]

candidate_dirs = []

for path in SEARCH_ROOT.rglob("*"):
    if not path.is_dir():
        continue

    name = path.name.lower()

    if any(k in name for k in data_keywords):
        candidate_dirs.append(path)

print("Candidate directories:", len(candidate_dirs))

for path in candidate_dirs[:200]:
    print("  ", path)

if len(candidate_dirs) > 200:
    print("\n[Directory display capped at 200]")

# ----------------------------------------------------------------
# 8. Search generator source for data/image dependencies
# ----------------------------------------------------------------

print("\n" + "=" * 70)
print("GENERATOR DATA DEPENDENCIES")
print("=" * 70)

for i, line in enumerate(generator_source.splitlines(), start=1):

    low = line.lower()

    if any(
        term in low
        for term in [
            "image_path",
            "image_root",
            "load_image",
            "read_image",
            "imread",
            "pil",
            "cv2",
            "mask",
            "224",
            "resize",
        ]
    ):
        print(f"{i:5d}: {line}")

# ----------------------------------------------------------------
# 9. Inspect frozen analysis source for model/data loading helpers
# ----------------------------------------------------------------

print("\n" + "=" * 70)
print("ANALYSIS PIPELINE — LOADING REFERENCES")
print("=" * 70)

for i, line in enumerate(analysis_source.splitlines(), start=1):

    low = line.lower()

    if any(
        term in low
        for term in [
            "torch.load",
            "load_state_dict",
            "resnet18",
            "imagenet",
            "dataset",
            "image",
            "mask",
            "checkpoint",
        ]
    ):
        print(f"{i:5d}: {line}")

# ----------------------------------------------------------------
# 10. No execution — audit only
# ----------------------------------------------------------------

print("\n" + "=" * 70)
print("CELL 5 STATUS")
print("=" * 70)

print("Training source inspected              : PASS")
print("Generator source inspected             : PASS")
print("Model names extracted                  : PASS")
print("Checkpoint candidates discovered       : PASS")
print("Image/data candidates discovered       : PASS")
print("No checkpoint loaded                   : YES")
print("No image loaded                        : YES")
print("No mask generated                     : YES")
print("No IG computation                     : YES")
print("No GPU computation                    : YES")

print("\nREADY FOR CELL 5 OUTPUT REVIEW")

CELL 5 — FROZEN MODEL / IMAGE / MASK LOADING AUDIT

TRAINING SOURCE — RELEVANT DEFINITIONS
Functions:
   derive_seed
   sample_rng
   augment_uint8
   to_chw_float
   sha_file
   make_dataset_class
   build_model
   gap_equivalence
   balanced_accuracy
   run
   main

Relevant source lines:
    2: """Frozen training script - ONE model per process (ISIC 2019 synthetic-hair dose-response study).
    4: Locked recipe: ImageNet-pretrained ResNet-18 (IMAGENET1K_V1), 2-output head, 224x224, ImageNet normalisation,
    6: CrossEntropyLoss, float32, deterministic algorithms, no early stopping, primary checkpoint = final epoch 15.
   10: os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")   # must precede CUDA initialisation -> fresh process per model
   19: PARAMS_2CLASS = 11_177_538            # ResNet-18 (11,689,512 with the 1000-class head) with a 2-output head
   34:     from PIL import Image
   35:     img = Image.fromarray(arr)
   37:         img = img.transpose(Image.Transpose.F

In [6]:
# ================================================================
# CELL 6 — CHECKPOINT + CACHE IDENTITY AUDIT
# ================================================================

from pathlib import Path
import hashlib
import pandas as pd
import numpy as np

print("=" * 70)
print("CELL 6 — CHECKPOINT + CACHE IDENTITY AUDIT")
print("=" * 70)

# ----------------------------------------------------------------
# 1. Define the two discovered checkpoint roots
# ----------------------------------------------------------------

CKPT_ROOT_A = Path(
    "/kaggle/input/notebooks/tejasseshadriiiii/"
    "ml-open-lab-30m-trained/training_runs"
)

CKPT_ROOT_B = Path(
    "/kaggle/input/notebooks/tejasseshadriiiii/"
    "isic-kaggle-analysis/trained_models_readonly/training_runs"
)

print("\nCheckpoint root A:")
print(CKPT_ROOT_A)

print("\nCheckpoint root B:")
print(CKPT_ROOT_B)

assert CKPT_ROOT_A.exists()
assert CKPT_ROOT_B.exists()

# ----------------------------------------------------------------
# 2. SHA256 helper
# ----------------------------------------------------------------

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

# ----------------------------------------------------------------
# 3. Compare final_epoch15.pt for all 30 models
# ----------------------------------------------------------------

print("\n" + "=" * 70)
print("CHECKPOINT BYTE-IDENTITY COMPARISON")
print("=" * 70)

comparison_rows = []

for model_name in sorted(model_names):

    pA = CKPT_ROOT_A / model_name / "final_epoch15.pt"
    pB = CKPT_ROOT_B / model_name / "final_epoch15.pt"

    exists_A = pA.exists()
    exists_B = pB.exists()

    sha_A = sha256_file(pA) if exists_A else None
    sha_B = sha256_file(pB) if exists_B else None

    identical = (
        exists_A
        and exists_B
        and sha_A == sha_B
    )

    comparison_rows.append({
        "model": model_name,
        "A_exists": exists_A,
        "B_exists": exists_B,
        "sha_A": sha_A,
        "sha_B": sha_B,
        "identical": identical,
    })

    print(
        f"{model_name:18s} | "
        f"A={exists_A} | "
        f"B={exists_B} | "
        f"IDENTICAL={identical}"
    )

comparison_df = pd.DataFrame(comparison_rows)

print("\nSummary:")
print(
    "Both present:",
    int((comparison_df["A_exists"] & comparison_df["B_exists"]).sum())
)
print(
    "Byte-identical:",
    int(comparison_df["identical"].sum())
)

# ----------------------------------------------------------------
# 4. Discover production image/cache files
# ----------------------------------------------------------------

print("\n" + "=" * 70)
print("PRODUCTION CACHE DISCOVERY")
print("=" * 70)

PROJECT_DATA_ROOT = Path(
    "/kaggle/input/datasets/tejasseshadriiiii/"
    "isic-final-project"
)

CACHE_FILES = []

for path in PROJECT_DATA_ROOT.rglob("*"):
    if path.is_file():
        name = path.name.lower()

        if (
            name in {
                "clean_224.npy",
                "hair_224.npy",
                "index.csv",
                "images.csv",
                "evaluation_index.csv",
                "eval_index.csv",
            }
            or "cache" in str(path).lower()
        ):
            CACHE_FILES.append(path)

print("Candidate cache/index files:", len(CACHE_FILES))

for path in CACHE_FILES[:200]:
    print("  ", path)

# ----------------------------------------------------------------
# 5. Explicitly locate clean_224.npy / hair_224.npy
# ----------------------------------------------------------------

print("\n" + "=" * 70)
print("EXACT IMAGE CACHE FILES")
print("=" * 70)

clean_candidates = [
    p for p in PROJECT_DATA_ROOT.rglob("clean_224.npy")
]

hair_candidates = [
    p for p in PROJECT_DATA_ROOT.rglob("hair_224.npy")
]

print("clean_224.npy candidates:", len(clean_candidates))
for p in clean_candidates:
    print("  ", p)

print("\nhair_224.npy candidates:", len(hair_candidates))
for p in hair_candidates:
    print("  ", p)

# ----------------------------------------------------------------
# 6. Inspect cache shapes WITHOUT loading the full arrays
# ----------------------------------------------------------------

print("\n" + "=" * 70)
print("CACHE ARRAY METADATA")
print("=" * 70)

for label, candidates in [
    ("clean_224", clean_candidates),
    ("hair_224", hair_candidates),
]:

    for p in candidates:
        arr = np.load(p, mmap_mode="r")

        print(f"\n{label}:")
        print("  path  :", p)
        print("  shape :", arr.shape)
        print("  dtype :", arr.dtype)
        print("  ndim  :", arr.ndim)

        if arr.ndim >= 4:
            print("  first image shape:", arr[0].shape)
            print("  first image min :", float(arr[0].min()))
            print("  first image max :", float(arr[0].max()))

# ----------------------------------------------------------------
# 7. Discover candidate evaluation index files
# ----------------------------------------------------------------

print("\n" + "=" * 70)
print("EVALUATION INDEX / MANIFEST DISCOVERY")
print("=" * 70)

index_candidates = []

for path in PROJECT_DATA_ROOT.rglob("*"):
    if not path.is_file():
        continue

    name = path.name.lower()

    if (
        path.suffix.lower() == ".csv"
        and any(
            token in name
            for token in [
                "index",
                "manifest",
                "population",
                "evaluation",
                "eval",
                "clean",
            ]
        )
    ):
        index_candidates.append(path)

print("Candidate CSV index/manifest files:", len(index_candidates))

for p in index_candidates[:200]:
    print("  ", p)

# ----------------------------------------------------------------
# 8. Inspect CSVs containing an 'image' column
# ----------------------------------------------------------------

print("\n" + "=" * 70)
print("IMAGE-INDEX COMPATIBILITY CHECK")
print("=" * 70)

compatible_tables = []

for p in index_candidates:

    try:
        df = pd.read_csv(p, nrows=10)
    except Exception:
        continue

    cols_lower = {c.lower(): c for c in df.columns}

    if "image" in cols_lower:
        image_col = cols_lower["image"]

        compatible_tables.append((p, image_col))

        print("\nFILE:", p)
        print("Image column:", image_col)
        print("Columns:", list(df.columns))

# ----------------------------------------------------------------
# 9. Check the locked 500 images against candidate indexes
# ----------------------------------------------------------------

print("\n" + "=" * 70)
print("LOCKED 500 → EVALUATION INDEX MATCHING")
print("=" * 70)

locked = pd.read_csv(XAI_SUBSET)

locked_images = set(
    locked["image"].astype(str).tolist()
)

print("Locked XAI images:", len(locked_images))

for p, image_col in compatible_tables:

    try:
        df = pd.read_csv(p, usecols=[image_col])
    except Exception:
        continue

    available = set(df[image_col].astype(str).tolist())

    overlap = locked_images & available

    print("\nFILE:", p)
    print("  indexed images :", len(available))
    print("  locked overlap :", len(overlap))
    print(
        "  complete match :",
        len(overlap) == len(locked_images)
    )

# ----------------------------------------------------------------
# 10. Status
# ----------------------------------------------------------------

print("\n" + "=" * 70)
print("CELL 6 STATUS")
print("=" * 70)

print("Checkpoint roots located              : PASS")
print("Checkpoint hashes compared            : PASS")
print("Cache candidates discovered           : PASS")
print("clean_224 candidates inspected        : PASS")
print("hair_224 candidates inspected         : PASS")
print("Evaluation index candidates inspected: PASS")
print("No model loaded                       : YES")
print("No IG computation                     : YES")
print("No GPU computation                    : YES")

print("\nIMPORTANT:")
print("Do NOT choose checkpoint root A or B yet.")
print("Do NOT compute IG yet.")
print("We will choose the canonical source only")
print("after reviewing this audit.")

print("\nREADY FOR CELL 6 OUTPUT REVIEW")

CELL 6 — CHECKPOINT + CACHE IDENTITY AUDIT

Checkpoint root A:
/kaggle/input/notebooks/tejasseshadriiiii/ml-open-lab-30m-trained/training_runs

Checkpoint root B:
/kaggle/input/notebooks/tejasseshadriiiii/isic-kaggle-analysis/trained_models_readonly/training_runs

CHECKPOINT BYTE-IDENTITY COMPARISON
rho_0.0_rep_1      | A=True | B=True | IDENTICAL=True
rho_0.0_rep_2      | A=True | B=True | IDENTICAL=True
rho_0.0_rep_3      | A=True | B=True | IDENTICAL=True
rho_0.0_rep_4      | A=True | B=True | IDENTICAL=True
rho_0.0_rep_5      | A=True | B=True | IDENTICAL=True
rho_0.2_rep_1      | A=True | B=True | IDENTICAL=True
rho_0.2_rep_2      | A=True | B=True | IDENTICAL=True
rho_0.2_rep_3      | A=True | B=True | IDENTICAL=True
rho_0.2_rep_4      | A=True | B=True | IDENTICAL=True
rho_0.2_rep_5      | A=True | B=True | IDENTICAL=True
rho_0.4_rep_1      | A=True | B=True | IDENTICAL=True
rho_0.4_rep_2      | A=True | B=True | IDENTICAL=True
rho_0.4_rep_3      | A=True | B=True | IDENTICAL=Tr

In [7]:
# ================================================================
# CELL 7 — FIND ACTUAL PRODUCTION IMAGE CACHE
# ================================================================

from pathlib import Path
import json
import hashlib
import numpy as np

print("=" * 70)
print("CELL 7 — ACTUAL PRODUCTION IMAGE CACHE DISCOVERY")
print("=" * 70)

INPUT_ROOT = Path("/kaggle/input")

# ----------------------------------------------------------------
# 1. Find every clean_224.npy / hair_224.npy under ALL Kaggle input
# ----------------------------------------------------------------

print("\n" + "=" * 70)
print("GLOBAL CACHE SEARCH")
print("=" * 70)

clean_all = sorted(INPUT_ROOT.rglob("clean_224.npy"))
hair_all = sorted(INPUT_ROOT.rglob("hair_224.npy"))

print("clean_224.npy found:", len(clean_all))
for p in clean_all:
    print("  ", p)

print("\nhair_224.npy found:", len(hair_all))
for p in hair_all:
    print("  ", p)

# ----------------------------------------------------------------
# 2. Find all cache-record / hash files
# ----------------------------------------------------------------

print("\n" + "=" * 70)
print("CACHE RECORD SEARCH")
print("=" * 70)

cache_record_files = []

for p in INPUT_ROOT.rglob("*"):
    if not p.is_file():
        continue

    name = p.name.lower()

    if (
        "cache" in name
        and (
            name.endswith(".json")
            or name.endswith(".txt")
            or name.endswith(".md")
            or name.endswith(".csv")
        )
    ):
        cache_record_files.append(p)

print("Cache-related metadata files:", len(cache_record_files))

for p in cache_record_files[:200]:
    print("  ", p)

# ----------------------------------------------------------------
# 3. Inspect JSON cache records
# ----------------------------------------------------------------

print("\n" + "=" * 70)
print("CACHE RECORD CONTENTS")
print("=" * 70)

for p in cache_record_files:

    if p.suffix.lower() != ".json":
        continue

    try:
        obj = json.loads(p.read_text(encoding="utf-8"))
    except Exception:
        continue

    text = json.dumps(obj, indent=2)

    # Only print records that actually mention clean/hair/cache
    low = text.lower()

    if (
        "clean_224" in low
        or "hair_224" in low
        or "cache" in low
    ):
        print("\n" + "-" * 70)
        print("FILE:", p)
        print(text[:12000])

# ----------------------------------------------------------------
# 4. Inspect candidate arrays if found
# ----------------------------------------------------------------

print("\n" + "=" * 70)
print("CACHE ARRAY METADATA")
print("=" * 70)

for label, paths in [
    ("clean_224", clean_all),
    ("hair_224", hair_all),
]:

    for p in paths:

        print("\n", label)
        print("PATH:", p)

        arr = np.load(p, mmap_mode="r")

        print("shape :", arr.shape)
        print("dtype :", arr.dtype)
        print("ndim  :", arr.ndim)

        if arr.ndim == 4:
            print("image shape:", arr.shape[1:])
            print("min:", float(arr.min()))
            print("max:", float(arr.max()))

            # Inspect first image only
            first = np.asarray(arr[0])

            print(
                "first image:",
                "shape=", first.shape,
                "dtype=", first.dtype,
                "min=", float(first.min()),
                "max=", float(first.max()),
            )

# ----------------------------------------------------------------
# 5. Inspect directory containing discovered cache
# ----------------------------------------------------------------

print("\n" + "=" * 70)
print("CACHE DIRECTORY CONTENTS")
print("=" * 70)

cache_dirs = set()

for p in clean_all + hair_all:
    cache_dirs.add(p.parent)

for d in sorted(cache_dirs):

    print("\nDIRECTORY:", d)

    try:
        children = sorted(d.iterdir())
    except Exception as e:
        print("Could not list:", e)
        continue

    for child in children:
        if child.is_file():
            print(
                f"  {child.name:35s} "
                f"{child.stat().st_size / (1024**2):.2f} MB"
            )

# ----------------------------------------------------------------
# 6. Compare clean/hair array counts and shapes
# ----------------------------------------------------------------

print("\n" + "=" * 70)
print("CLEAN ↔ HAIR CACHE CONSISTENCY")
print("=" * 70)

if len(clean_all) == 1 and len(hair_all) == 1:

    clean = np.load(clean_all[0], mmap_mode="r")
    hair = np.load(hair_all[0], mmap_mode="r")

    print("Same number of images:", clean.shape[0] == hair.shape[0])
    print("Same image shape    :", clean.shape == hair.shape)
    print("Clean shape         :", clean.shape)
    print("Hair shape          :", hair.shape)

    assert clean.shape == hair.shape

else:
    print("Cache pair not uniquely resolved yet.")

# ----------------------------------------------------------------
# 7. Look for image-index files in the SAME directory as cache
# ----------------------------------------------------------------

print("\n" + "=" * 70)
print("LOCAL CACHE INDEX FILES")
print("=" * 70)

for d in sorted(cache_dirs):

    print("\nDIRECTORY:", d)

    for p in sorted(d.iterdir()):

        if not p.is_file():
            continue

        if p.suffix.lower() not in {
            ".csv", ".json", ".txt", ".md", ".parquet"
        }:
            continue

        print("  ", p.name)

# ----------------------------------------------------------------
# 8. Final status
# ----------------------------------------------------------------

print("\n" + "=" * 70)
print("CELL 7 STATUS")
print("=" * 70)

print("Global cache search completed       : PASS")
print("Cache metadata search completed     : PASS")
print("No model loaded                     : YES")
print("No IG computation                   : YES")
print("No GPU computation                  : YES")

print("\nIMPORTANT:")
print("If exactly one clean_224.npy + hair_224.npy pair is found,")
print("we will inspect its associated index/record next.")
print("Do NOT run IG yet.")

print("\nREADY FOR CELL 7 OUTPUT REVIEW")

CELL 7 — ACTUAL PRODUCTION IMAGE CACHE DISCOVERY

GLOBAL CACHE SEARCH
clean_224.npy found: 0

hair_224.npy found: 0

CACHE RECORD SEARCH
Cache-related metadata files: 14
   /kaggle/input/datasets/tejasseshadriiiii/isic-final-project/step2T_image_cache_record/CACHE_RECORD.json
   /kaggle/input/datasets/tejasseshadriiiii/isic-final-project/step2T_image_cache_record/CACHE_FINAL_HASH.json
   /kaggle/input/notebooks/tejasseshadriiiii/ml-open-lab-30m-trained/kaggle_records/cache_record.json
   /kaggle/input/notebooks/tejasseshadriiiii/ml-open-lab-30m-trained/kaggle_records/cache_final_hash.json
   /kaggle/input/notebooks/tejasseshadriiiii/ml-open-lab-30m-trained/isic_final_project/step2T_image_cache_record/CACHE_RECORD.json
   /kaggle/input/notebooks/tejasseshadriiiii/ml-open-lab-30m-trained/isic_final_project/step2T_image_cache_record/CACHE_FINAL_HASH.json
   /kaggle/input/notebooks/tejasseshadriiiii/isic-kaggle-analysis/analysis_records/cache_record.json
   /kaggle/input/notebooks/tejasses

In [8]:
# ================================================================
# CELL 8 — ACTUAL FILESYSTEM CACHE LOCATION AUDIT
# ================================================================

from pathlib import Path
import os
import hashlib

print("=" * 70)
print("CELL 8 — ACTUAL FILESYSTEM CACHE LOCATION AUDIT")
print("=" * 70)

# Canonical hashes from the locked cache record
EXPECTED_HASHES = {
    "cache_index.csv":
        "d86bd82bb5294e11a542a344f7429e200dc0169baad9df9a9b8d96359987842c",

    "clean_224.npy":
        "455ddd37115fbac20af4448a792c7e96376196e798cb5bc5759107d839c5e912",

    "hair_224.npy":
        "5051802ed71028c38ca945992e33e979850919cea6f820c6c68156baa498cb49",

    "mask_area.npy":
        "bc6b6d5b65cc4d5714797f8170fe153db61e36736a6e7fda112f8edf9fad7e9c",
}

SEARCH_ROOTS = [
    Path("/kaggle/input"),
    Path("/kaggle/working"),
    Path("/kaggle/temp"),
    Path("/content"),
]

# ----------------------------------------------------------------
# SHA256 helper
# ----------------------------------------------------------------

def sha256_file(path, chunk_size=1024 * 1024):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)

            if not chunk:
                break

            h.update(chunk)

    return h.hexdigest()


# ----------------------------------------------------------------
# 1. Search exact filenames
# ----------------------------------------------------------------

print("\n" + "=" * 70)
print("EXACT CACHE FILENAME SEARCH")
print("=" * 70)

found = {
    name: []
    for name in EXPECTED_HASHES
}

for root in SEARCH_ROOTS:

    if not root.exists():
        print("ROOT NOT PRESENT:", root)
        continue

    print("\nSearching:", root)

    for filename in EXPECTED_HASHES:

        matches = []

        try:
            matches = list(root.rglob(filename))
        except Exception as e:
            print("Search error:", root, filename, e)

        for p in matches:
            if p.is_file():
                found[filename].append(p)

                print("FOUND:", p)


# ----------------------------------------------------------------
# 2. Summarize exact matches
# ----------------------------------------------------------------

print("\n" + "=" * 70)
print("EXACT FILENAME SUMMARY")
print("=" * 70)

for filename, paths in found.items():

    print(f"\n{filename}")
    print("count:", len(paths))

    for p in paths:
        try:
            size_mb = p.stat().st_size / (1024 ** 2)
        except Exception:
            size_mb = float("nan")

        print(
            " ",
            p,
            "| size =",
            f"{size_mb:.2f} MB"
        )


# ----------------------------------------------------------------
# 3. Hash any discovered files
# ----------------------------------------------------------------

print("\n" + "=" * 70)
print("HASH VERIFICATION")
print("=" * 70)

verified = {}

for filename, paths in found.items():

    verified[filename] = []

    for p in paths:

        print("\nFILE:", p)

        try:
            actual = sha256_file(p)
        except Exception as e:
            print("HASH ERROR:", e)
            continue

        expected = EXPECTED_HASHES[filename]

        identical = (actual == expected)

        print("Expected:", expected)
        print("Actual  :", actual)
        print("IDENTICAL:", identical)

        verified[filename].append({
            "path": str(p),
            "sha256": actual,
            "expected": expected,
            "identical": identical,
        })


# ----------------------------------------------------------------
# 4. Search likely cache directories by name
# ----------------------------------------------------------------

print("\n" + "=" * 70)
print("CACHE-DIRECTORY SEARCH")
print("=" * 70)

cache_dirs = []

for root in SEARCH_ROOTS:

    if not root.exists():
        continue

    try:
        for p in root.rglob("*"):
            if not p.is_dir():
                continue

            name = p.name.lower()

            if (
                "isic_cache" in name
                or "image_cache" in name
                or "cache" == name
            ):
                cache_dirs.append(p)

    except Exception as e:
        print("Directory search error:", root, e)

cache_dirs = sorted(set(cache_dirs))

print("Candidate cache directories:", len(cache_dirs))

for d in cache_dirs[:200]:
    print(" ", d)


# ----------------------------------------------------------------
# 5. Inspect likely /content/isic_cache explicitly
# ----------------------------------------------------------------

print("\n" + "=" * 70)
print("EXPLICIT /content/isic_cache CHECK")
print("=" * 70)

content_cache = Path("/content/isic_cache")

print("Exists:", content_cache.exists())
print("Is directory:", content_cache.is_dir())

if content_cache.exists() and content_cache.is_dir():

    for p in sorted(content_cache.iterdir()):

        if p.is_file():

            size_mb = p.stat().st_size / (1024 ** 2)

            print(
                f"{p.name:25s} "
                f"{size_mb:12.2f} MB"
            )


# ----------------------------------------------------------------
# 6. Final interpretation
# ----------------------------------------------------------------

print("\n" + "=" * 70)
print("CELL 8 STATUS")
print("=" * 70)

for filename in EXPECTED_HASHES:

    paths = found[filename]

    exact = [
        x for x in verified[filename]
        if x["identical"]
    ]

    print(
        f"{filename:20s} "
        f"found={len(paths):2d} "
        f"hash-matched={len(exact):2d}"
    )

print("\nNo model loaded : YES")
print("No IG computed  : YES")
print("No GPU used     : YES")

print("\nDO NOT RUN IG YET.")
print("We first need the canonical cache files physically available.")

CELL 8 — ACTUAL FILESYSTEM CACHE LOCATION AUDIT

EXACT CACHE FILENAME SEARCH

Searching: /kaggle/input

Searching: /kaggle/working
ROOT NOT PRESENT: /kaggle/temp

Searching: /content

EXACT FILENAME SUMMARY

cache_index.csv
count: 0

clean_224.npy
count: 0

hair_224.npy
count: 0

mask_area.npy
count: 0

HASH VERIFICATION

CACHE-DIRECTORY SEARCH
Candidate cache directories: 4
  /kaggle/input/datasets/tejasseshadriiiii/isic-final-project/step2T_image_cache_record
  /kaggle/input/notebooks/tejasseshadriiiii/isic-kaggle-analysis/isic_final_project/step2T_image_cache_record
  /kaggle/input/notebooks/tejasseshadriiiii/isic-kaggle-analysis/trained_models_readonly/isic_final_project/step2T_image_cache_record
  /kaggle/input/notebooks/tejasseshadriiiii/ml-open-lab-30m-trained/isic_final_project/step2T_image_cache_record

EXPLICIT /content/isic_cache CHECK
Exists: False
Is directory: False

CELL 8 STATUS
cache_index.csv      found= 0 hash-matched= 0
clean_224.npy        found= 0 hash-matched= 0


In [9]:
# ================================================================
# CELL 9 — SOURCE IMAGE + RECONSTRUCTION INPUT AUDIT
# ================================================================

from pathlib import Path
import pandas as pd
import numpy as np
import hashlib
from PIL import Image

print("=" * 70)
print("CELL 9 — SOURCE IMAGE + RECONSTRUCTION INPUT AUDIT")
print("=" * 70)

# ----------------------------------------------------------------
# Locked project/data paths
# ----------------------------------------------------------------

PROJECT_DATA_ROOT = Path(
    "/kaggle/input/datasets/tejasseshadriiiii/isic-final-project"
)

LOCKED_POP_PATH = (
    PROJECT_DATA_ROOT
    / "step0I_locked_clean_population"
    / "LOCKED_CLEAN_development_population.csv"
)

XAI_SUBSET_PATH = (
    PROJECT_DATA_ROOT
    / "step2U_xai_subset_lock"
    / "LOCKED_XAI_SUBSET.csv"
)

GEN_PATH = (
    PROJECT_DATA_ROOT
    / "step1J_generator_recovery_lock"
    / "recovered_original_hair_generator.py"
)

print("\nLocked population:")
print(LOCKED_POP_PATH)

print("\nLocked XAI subset:")
print(XAI_SUBSET_PATH)

print("\nFrozen generator:")
print(GEN_PATH)


# ----------------------------------------------------------------
# 1. Verify locked files
# ----------------------------------------------------------------

print("\n" + "=" * 70)
print("LOCKED INPUT EXISTENCE")
print("=" * 70)

for p in [
    LOCKED_POP_PATH,
    XAI_SUBSET_PATH,
    GEN_PATH,
]:

    print(
        f"{str(p):95s}",
        "EXISTS" if p.exists() else "MISSING"
    )

assert LOCKED_POP_PATH.exists()
assert XAI_SUBSET_PATH.exists()
assert GEN_PATH.exists()


# ----------------------------------------------------------------
# 2. Load locked population
# ----------------------------------------------------------------

pop = pd.read_csv(LOCKED_POP_PATH)

print("\n" + "=" * 70)
print("LOCKED POPULATION")
print("=" * 70)

print("Shape:", pop.shape)
print("Columns:", pop.columns.tolist())

assert list(pop.columns) == [
    "image",
    "lesion_id",
    "class_name",
    "malignant",
    "split",
]

print("Unique images:", pop["image"].nunique())
print("Duplicate image rows:", pop["image"].duplicated().sum())

assert pop["image"].nunique() == len(pop)


# ----------------------------------------------------------------
# 3. Load locked XAI subset
# ----------------------------------------------------------------

xai = pd.read_csv(XAI_SUBSET_PATH)

print("\n" + "=" * 70)
print("LOCKED XAI SUBSET")
print("=" * 70)

print("Shape:", xai.shape)
print("Columns:", xai.columns.tolist())

assert len(xai) == 500
assert xai["image"].nunique() == 500

print("500 unique XAI images confirmed.")


# ----------------------------------------------------------------
# 4. Check that every XAI image belongs to locked population
# ----------------------------------------------------------------

pop_images = set(pop["image"].astype(str))
xai_images = set(xai["image"].astype(str))

missing_from_population = sorted(
    xai_images - pop_images
)

print("\n" + "=" * 70)
print("XAI → LOCKED POPULATION COMPATIBILITY")
print("=" * 70)

print(
    "XAI images missing from locked population:",
    len(missing_from_population)
)

assert len(missing_from_population) == 0

print("✓ All 500 XAI images belong to locked population.")


# ----------------------------------------------------------------
# 5. Search for the ORIGINAL ISIC source image directory
# ----------------------------------------------------------------

print("\n" + "=" * 70)
print("SOURCE IMAGE DIRECTORY SEARCH")
print("=" * 70)

EXPECTED_SOURCE = Path(
    "/kaggle/input/datasets/alifshahariar/"
    "isic-2019-dataset-full/"
    "isic-2019-dataset-full/"
    "train"
)

candidate_source_dirs = []

# First test the path explicitly recorded by the original
# cache-builder record.
if EXPECTED_SOURCE.exists() and EXPECTED_SOURCE.is_dir():
    candidate_source_dirs.append(EXPECTED_SOURCE)

print("Recorded source path:")
print(EXPECTED_SOURCE)
print(
    "Exists:",
    EXPECTED_SOURCE.exists(),
    "| Directory:",
    EXPECTED_SOURCE.is_dir()
)


# ----------------------------------------------------------------
# 6. If recorded path exists, inspect it
# ----------------------------------------------------------------

if EXPECTED_SOURCE.exists() and EXPECTED_SOURCE.is_dir():

    source_files = [
        p for p in EXPECTED_SOURCE.iterdir()
        if p.is_file()
        and p.suffix.lower() in {".jpg", ".jpeg"}
    ]

    print("\nSource JPEG files:", len(source_files))

    # Build filename stem lookup without loading images
    source_ids = {
        p.stem
        for p in source_files
    }

    missing_xai_source = sorted(
        xai_images - source_ids
    )

    missing_population_source = sorted(
        pop_images - source_ids
    )

    print(
        "XAI images missing from source:",
        len(missing_xai_source)
    )

    print(
        "Locked population images missing from source:",
        len(missing_population_source)
    )

    if missing_xai_source:
        print("\nFirst missing XAI source IDs:")
        for x in missing_xai_source[:20]:
            print(" ", x)

    assert len(missing_xai_source) == 0

    print(
        "\n✓ All 500 XAI source images are physically available."
    )

else:

    print(
        "\nRecorded source directory is NOT mounted."
    )

    print(
        "Do not reconstruct anything yet."
    )


# ----------------------------------------------------------------
# 7. Inspect a small number of source images
# ----------------------------------------------------------------

print("\n" + "=" * 70)
print("SOURCE IMAGE FORMAT SPOT CHECK")
print("=" * 70)

if EXPECTED_SOURCE.exists():

    sample_ids = sorted(xai_images)[:5]

    for image_id in sample_ids:

        candidates = [
            EXPECTED_SOURCE / f"{image_id}.jpg",
            EXPECTED_SOURCE / f"{image_id}.jpeg",
            EXPECTED_SOURCE / f"{image_id}.JPG",
            EXPECTED_SOURCE / f"{image_id}.JPEG",
        ]

        src = next(
            (p for p in candidates if p.exists()),
            None
        )

        assert src is not None

        with Image.open(src) as im:

            print(
                image_id,
                "| format=", im.format,
                "| mode=", im.mode,
                "| size=", im.size
            )


# ----------------------------------------------------------------
# 8. Verify PIL / NumPy versions against locked cache record
# ----------------------------------------------------------------

print("\n" + "=" * 70)
print("REPRODUCTION ENVIRONMENT")
print("=" * 70)

import PIL
import numpy

print("Current PIL   :", PIL.__version__)
print("Current NumPy :", numpy.__version__)

print("Locked cache PIL   : 11.3.0")
print("Locked cache NumPy : 2.1.3")

print(
    "PIL version match   :",
    PIL.__version__ == "11.3.0"
)

print(
    "NumPy version match :",
    numpy.__version__ == "2.1.3"
)


# ----------------------------------------------------------------
# 9. Final status
# ----------------------------------------------------------------

print("\n" + "=" * 70)
print("CELL 9 STATUS")
print("=" * 70)

print("Locked population loaded       : PASS")
print("500 XAI images verified        : PASS")
print("XAI → population mapping       : PASS")

if EXPECTED_SOURCE.exists():
    print("Recorded source directory      : FOUND")
    print("500 source images              : VERIFIED")
else:
    print("Recorded source directory      : NOT FOUND")

print("\nNo model loaded : YES")
print("No IG computed  : YES")
print("No GPU used     : YES")

print("\nIMPORTANT:")
print("Do NOT regenerate the cache yet.")
print("Do NOT run IG yet.")
print("We will decide the reconstruction route only")
print("after reviewing this audit.")

print("\nREADY FOR CELL 9 OUTPUT REVIEW")

CELL 9 — SOURCE IMAGE + RECONSTRUCTION INPUT AUDIT

Locked population:
/kaggle/input/datasets/tejasseshadriiiii/isic-final-project/step0I_locked_clean_population/LOCKED_CLEAN_development_population.csv

Locked XAI subset:
/kaggle/input/datasets/tejasseshadriiiii/isic-final-project/step2U_xai_subset_lock/LOCKED_XAI_SUBSET.csv

Frozen generator:
/kaggle/input/datasets/tejasseshadriiiii/isic-final-project/step1J_generator_recovery_lock/recovered_original_hair_generator.py

LOCKED INPUT EXISTENCE
/kaggle/input/datasets/tejasseshadriiiii/isic-final-project/step0I_locked_clean_population/LOCKED_CLEAN_development_population.csv EXISTS
/kaggle/input/datasets/tejasseshadriiiii/isic-final-project/step2U_xai_subset_lock/LOCKED_XAI_SUBSET.csv EXISTS
/kaggle/input/datasets/tejasseshadriiiii/isic-final-project/step1J_generator_recovery_lock/recovered_original_hair_generator.py EXISTS

LOCKED POPULATION
Shape: (23227, 5)
Columns: ['image', 'lesion_id', 'class_name', 'malignant', 'split']
Unique image

In [10]:
# ================================================================
# CELL 10 — LOCKED ROW + GENERATOR MASK RECONSTRUCTION AUDIT
# ================================================================

from pathlib import Path
import pandas as pd
import numpy as np
import hashlib
import types
import ast

print("=" * 70)
print("CELL 10 — LOCKED ROW + GENERATOR MASK RECONSTRUCTION AUDIT")
print("=" * 70)


# ----------------------------------------------------------------
# Paths
# ----------------------------------------------------------------

PROJECT_DATA_ROOT = Path(
    "/kaggle/input/datasets/tejasseshadriiiii/isic-final-project"
)

LOCKED_POP_PATH = (
    PROJECT_DATA_ROOT
    / "step0I_locked_clean_population"
    / "LOCKED_CLEAN_development_population.csv"
)

XAI_SUBSET_PATH = (
    PROJECT_DATA_ROOT
    / "step2U_xai_subset_lock"
    / "LOCKED_XAI_SUBSET.csv"
)

GEN_PATH = (
    PROJECT_DATA_ROOT
    / "step1J_generator_recovery_lock"
    / "recovered_original_hair_generator.py"
)


# ----------------------------------------------------------------
# 1. Load locked population and XAI subset
# ----------------------------------------------------------------

pop = pd.read_csv(LOCKED_POP_PATH)
xai = pd.read_csv(XAI_SUBSET_PATH)

print("\nPopulation:", pop.shape)
print("XAI subset:", xai.shape)


# ----------------------------------------------------------------
# 2. Reconstruct the ORIGINAL cache ordering
#
# The historical cache builder explicitly sorted:
#     split
#     image
# and then assigned row = 0...N-1.
# ----------------------------------------------------------------

pop_sorted = (
    pop
    .sort_values(["split", "image"])
    .reset_index(drop=True)
)

pop_sorted["expected_cache_row"] = np.arange(
    len(pop_sorted),
    dtype=np.int64
)


# ----------------------------------------------------------------
# 3. Verify XAI subset row values against original cache ordering
# ----------------------------------------------------------------

print("\n" + "=" * 70)
print("XAI SUBSET → HISTORICAL CACHE ROW AUDIT")
print("=" * 70)

row_check = xai.merge(
    pop_sorted[
        [
            "image",
            "lesion_id",
            "expected_cache_row",
        ]
    ],
    on=["image", "lesion_id"],
    how="left",
    validate="one_to_one",
)

print(
    "Rows with missing population match:",
    row_check["expected_cache_row"].isna().sum()
)

assert row_check["expected_cache_row"].notna().all()

row_check["row_match"] = (
    row_check["row"].astype(np.int64)
    ==
    row_check["expected_cache_row"].astype(np.int64)
)

print(
    "Exact cache-row matches:",
    int(row_check["row_match"].sum()),
    "/",
    len(row_check)
)

print(
    "Row mismatches:",
    int((~row_check["row_match"]).sum())
)

if (~row_check["row_match"]).any():

    print("\nFIRST ROW MISMATCHES:")

    print(
        row_check.loc[
            ~row_check["row_match"],
            [
                "image",
                "row",
                "expected_cache_row",
            ]
        ].head(20).to_string(index=False)
    )

assert row_check["row_match"].all()

print("✓ All 500 locked cache-row values are exact.")


# ----------------------------------------------------------------
# 4. Load frozen generator definitions ONLY
#
# This deliberately avoids executing any module-level test/build
# code from the recovered generator.
# ----------------------------------------------------------------

print("\n" + "=" * 70)
print("FROZEN GENERATOR LOAD")
print("=" * 70)

source = GEN_PATH.read_text(encoding="utf-8")
tree = ast.parse(source)

safe_nodes = []

for node in tree.body:

    # Literal imports
    if isinstance(node, (ast.Import, ast.ImportFrom)):
        safe_nodes.append(node)

    # Literal constant assignments
    elif isinstance(node, ast.Assign):

        try:
            ast.literal_eval(node.value)
            safe_nodes.append(node)
        except Exception:
            pass

    elif isinstance(node, ast.AnnAssign):

        if node.value is not None:

            try:
                ast.literal_eval(node.value)
                safe_nodes.append(node)
            except Exception:
                pass

    # Function definitions
    elif isinstance(
        node,
        (ast.FunctionDef, ast.AsyncFunctionDef)
    ):
        safe_nodes.append(node)


module_ast = ast.Module(
    body=safe_nodes,
    type_ignores=[]
)

ast.fix_missing_locations(module_ast)

generator = types.ModuleType(
    "frozen_hair_generator"
)

generator.__file__ = str(GEN_PATH)

exec(
    compile(
        module_ast,
        str(GEN_PATH),
        "exec"
    ),
    generator.__dict__
)

assert hasattr(generator, "generate_hair_mask")
assert hasattr(generator, "artifact_seed")
assert hasattr(generator, "cubic_bezier")
assert hasattr(generator, "apply_hair")

print("✓ generate_hair_mask() found")
print("✓ apply_hair() found")
print("✓ artifact_seed() found")
print("✓ cubic_bezier() found")


# ----------------------------------------------------------------
# 5. Verify locked generator constants
# ----------------------------------------------------------------

print("\n" + "=" * 70)
print("GENERATOR CONSTANTS")
print("=" * 70)

print(
    "GLOBAL_SEED:",
    generator.GLOBAL_SEED
)

print(
    "PRIMARY_SEVERITY:",
    generator.PRIMARY_SEVERITY
)

assert generator.GLOBAL_SEED == 20260919
assert generator.PRIMARY_SEVERITY == "medium"

print("✓ GLOBAL_SEED matches locked value.")
print("✓ PRIMARY_SEVERITY matches locked value.")


# ----------------------------------------------------------------
# 6. Verify generator SHA256
# ----------------------------------------------------------------

def sha256_file(path, chunk_size=1024 * 1024):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        while True:

            chunk = f.read(chunk_size)

            if not chunk:
                break

            h.update(chunk)

    return h.hexdigest()


GEN_EXPECTED_SHA = (
    "32be8092c85458bed31a547f2ba6c57c3e1c2cf257d7cb0f76bd28eebc7b26c2"
)

gen_sha = sha256_file(GEN_PATH)

print("\nGenerator SHA256:")
print(gen_sha)

print(
    "Expected:",
    GEN_EXPECTED_SHA
)

assert gen_sha == GEN_EXPECTED_SHA

print("✓ Frozen generator byte hash verified.")


# ----------------------------------------------------------------
# 7. Reconstruct the exact binary mask for all 500 XAI images
#
# No images are modified or cached here.
# ----------------------------------------------------------------

print("\n" + "=" * 70)
print("500-IMAGE MASK RECONSTRUCTION")
print("=" * 70)

mask_results = []

for i, row in xai.sort_values("image").reset_index(drop=True).iterrows():

    image_id = str(row["image"])

    expected_area = int(row["mask_area"])

    mask = generator.generate_hair_mask(
        image_id=image_id,
        severity=generator.PRIMARY_SEVERITY,
        global_seed=generator.GLOBAL_SEED,
        size=224,
    )

    mask = np.asarray(mask).astype(bool)

    assert mask.shape == (224, 224)

    generated_area = int(mask.sum())

    mask_results.append(
        {
            "image": image_id,
            "expected_mask_area": expected_area,
            "generated_mask_area": generated_area,
            "area_match": (
                generated_area == expected_area
            ),
        }
    )


mask_check = pd.DataFrame(mask_results)


# ----------------------------------------------------------------
# 8. Report mask reconstruction results
# ----------------------------------------------------------------

print(
    "\nExact mask-area matches:",
    int(mask_check["area_match"].sum()),
    "/",
    len(mask_check)
)

print(
    "Mask-area mismatches:",
    int((~mask_check["area_match"]).sum())
)

if (~mask_check["area_match"]).any():

    print("\nFIRST MASK-AREA MISMATCHES:")

    print(
        mask_check.loc[
            ~mask_check["area_match"]
        ].head(20).to_string(index=False)
    )


assert mask_check["area_match"].all()

print(
    "\n✓ All 500 reconstructed masks have exactly"
    " the locked mask area."
)


# ----------------------------------------------------------------
# 9. Compare area distribution
# ----------------------------------------------------------------

print("\n" + "=" * 70)
print("MASK AREA DISTRIBUTION")
print("=" * 70)

print(
    "Locked min:",
    int(xai["mask_area"].min())
)

print(
    "Generated min:",
    int(mask_check["generated_mask_area"].min())
)

print(
    "Locked mean:",
    float(xai["mask_area"].mean())
)

print(
    "Generated mean:",
    float(mask_check["generated_mask_area"].mean())
)

print(
    "Locked max:",
    int(xai["mask_area"].max())
)

print(
    "Generated max:",
    int(mask_check["generated_mask_area"].max())
)


# ----------------------------------------------------------------
# 10. Final status
# ----------------------------------------------------------------

print("\n" + "=" * 70)
print("CELL 10 STATUS")
print("=" * 70)

print("Historical cache-row mapping : VERIFIED")
print("Frozen generator hash        : VERIFIED")
print("Generator constants          : VERIFIED")
print("500 mask reconstructions     : COMPLETED")
print("500/500 mask areas identical  : VERIFIED")

print("\nNumPy version:")
print(np.__version__)

print("\nNo model loaded : YES")
print("No IG computed  : YES")
print("No GPU used     : YES")

print("\nIMPORTANT:")
print("This cell reconstructed MASKS only.")
print("No clean/hair image cache has been generated.")
print("Do NOT run IG yet.")

print("\nREADY FOR CELL 10 OUTPUT REVIEW")

CELL 10 — LOCKED ROW + GENERATOR MASK RECONSTRUCTION AUDIT

Population: (23227, 5)
XAI subset: (500, 8)

XAI SUBSET → HISTORICAL CACHE ROW AUDIT
Rows with missing population match: 0
Exact cache-row matches: 500 / 500
Row mismatches: 0
✓ All 500 locked cache-row values are exact.

FROZEN GENERATOR LOAD
✓ generate_hair_mask() found
✓ apply_hair() found
✓ artifact_seed() found
✓ cubic_bezier() found

GENERATOR CONSTANTS
GLOBAL_SEED: 20260919
PRIMARY_SEVERITY: medium
✓ GLOBAL_SEED matches locked value.
✓ PRIMARY_SEVERITY matches locked value.

Generator SHA256:
32be8092c85458bed31a547f2ba6c57c3e1c2cf257d7cb0f76bd28eebc7b26c2
Expected: 32be8092c85458bed31a547f2ba6c57c3e1c2cf257d7cb0f76bd28eebc7b26c2
✓ Frozen generator byte hash verified.

500-IMAGE MASK RECONSTRUCTION

Exact mask-area matches: 500 / 500
Mask-area mismatches: 0

✓ All 500 reconstructed masks have exactly the locked mask area.

MASK AREA DISTRIBUTION
Locked min: 23
Generated min: 23
Locked mean: 525.728
Generated mean: 525.7

In [11]:
# ================================================================
# CELL 11 — RECONSTRUCT 500-IMAGE WORKING CACHE
# ================================================================

from pathlib import Path
import pandas as pd
import numpy as np
from PIL import Image
import hashlib
import types
import ast
import json

print("=" * 70)
print("CELL 11 — RECONSTRUCT 500-IMAGE WORKING CACHE")
print("=" * 70)


# ----------------------------------------------------------------
# Paths
# ----------------------------------------------------------------

PROJECT_DATA_ROOT = Path(
    "/kaggle/input/datasets/tejasseshadriiiii/isic-final-project"
)

LOCKED_POP_PATH = (
    PROJECT_DATA_ROOT
    / "step0I_locked_clean_population"
    / "LOCKED_CLEAN_development_population.csv"
)

XAI_SUBSET_PATH = (
    PROJECT_DATA_ROOT
    / "step2U_xai_subset_lock"
    / "LOCKED_XAI_SUBSET.csv"
)

GEN_PATH = (
    PROJECT_DATA_ROOT
    / "step1J_generator_recovery_lock"
    / "recovered_original_hair_generator.py"
)

SOURCE_DIR = Path(
    "/kaggle/input/datasets/alifshahariar/"
    "isic-2019-dataset-full/"
    "isic-2019-dataset-full/"
    "train"
)

OUT_DIR = Path(
    "/kaggle/working/t1_4c_xai_cache_500"
)

OUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ----------------------------------------------------------------
# Output paths
# ----------------------------------------------------------------

CLEAN_OUT = OUT_DIR / "clean_500.npy"
HAIR_OUT = OUT_DIR / "hair_500.npy"
MASK_OUT = OUT_DIR / "mask_500.npy"
AREA_OUT = OUT_DIR / "mask_area_500.npy"
INDEX_OUT = OUT_DIR / "xai_cache_index_500.csv"
RECORD_OUT = OUT_DIR / "reconstruction_record.json"


# ----------------------------------------------------------------
# 1. Load locked XAI subset
# ----------------------------------------------------------------

xai = pd.read_csv(XAI_SUBSET_PATH)

assert len(xai) == 500
assert xai["image"].nunique() == 500

print("\nLocked XAI images:", len(xai))


# ----------------------------------------------------------------
# 2. Load frozen generator definitions only
# ----------------------------------------------------------------

source = GEN_PATH.read_text(
    encoding="utf-8"
)

tree = ast.parse(source)

safe_nodes = []

for node in tree.body:

    if isinstance(
        node,
        (ast.Import, ast.ImportFrom)
    ):
        safe_nodes.append(node)

    elif isinstance(
        node,
        ast.Assign
    ):

        try:
            ast.literal_eval(node.value)
            safe_nodes.append(node)
        except Exception:
            pass

    elif isinstance(
        node,
        ast.AnnAssign
    ):

        if node.value is not None:

            try:
                ast.literal_eval(node.value)
                safe_nodes.append(node)
            except Exception:
                pass

    elif isinstance(
        node,
        (ast.FunctionDef, ast.AsyncFunctionDef)
    ):
        safe_nodes.append(node)


module_ast = ast.Module(
    body=safe_nodes,
    type_ignores=[]
)

ast.fix_missing_locations(module_ast)

generator = types.ModuleType(
    "frozen_hair_generator"
)

generator.__file__ = str(GEN_PATH)

exec(
    compile(
        module_ast,
        str(GEN_PATH),
        "exec"
    ),
    generator.__dict__
)

assert generator.GLOBAL_SEED == 20260919
assert generator.PRIMARY_SEVERITY == "medium"

print("✓ Frozen generator loaded")
print("✓ GLOBAL_SEED =", generator.GLOBAL_SEED)
print("✓ PRIMARY_SEVERITY =", generator.PRIMARY_SEVERITY)


# ----------------------------------------------------------------
# 3. Remove only OUR previous reconstruction outputs
# ----------------------------------------------------------------

for p in [
    CLEAN_OUT,
    HAIR_OUT,
    MASK_OUT,
    AREA_OUT,
    INDEX_OUT,
    RECORD_OUT,
]:

    if p.exists():
        p.unlink()

print("\n✓ Previous T1.4c reconstruction outputs cleared.")


# ----------------------------------------------------------------
# 4. Allocate exact 500-image arrays
# ----------------------------------------------------------------

N = len(xai)

clean_mm = np.lib.format.open_memmap(
    CLEAN_OUT,
    mode="w+",
    dtype=np.uint8,
    shape=(N, 224, 224, 3),
)

hair_mm = np.lib.format.open_memmap(
    HAIR_OUT,
    mode="w+",
    dtype=np.uint8,
    shape=(N, 224, 224, 3),
)

mask_mm = np.lib.format.open_memmap(
    MASK_OUT,
    mode="w+",
    dtype=np.uint8,
    shape=(N, 224, 224),
)

area_mm = np.lib.format.open_memmap(
    AREA_OUT,
    mode="w+",
    dtype=np.int32,
    shape=(N,),
)


# ----------------------------------------------------------------
# 5. Reconstruct each image exactly
# ----------------------------------------------------------------

records = []

print("\n" + "=" * 70)
print("RECONSTRUCTING 500 IMAGES")
print("=" * 70)

for r, row in xai.reset_index(drop=True).iterrows():

    image_id = str(row["image"])

    expected_area = int(
        row["mask_area"]
    )

    # ------------------------------------------------------------
    # Locate original JPEG
    # ------------------------------------------------------------

    candidates = [
        SOURCE_DIR / f"{image_id}.jpg",
        SOURCE_DIR / f"{image_id}.jpeg",
        SOURCE_DIR / f"{image_id}.JPG",
        SOURCE_DIR / f"{image_id}.JPEG",
    ]

    src = next(
        (
            p
            for p in candidates
            if p.exists()
        ),
        None
    )

    assert src is not None, (
        f"Source image not found: {image_id}"
    )

    # ------------------------------------------------------------
    # Exact historical clean preprocessing
    # ------------------------------------------------------------

    clean = np.asarray(
        Image.open(src)
        .convert("RGB")
        .resize(
            (224, 224),
            Image.Resampling.LANCZOS
        ),
        dtype=np.uint8
    )

    assert clean.shape == (224, 224, 3)
    assert clean.dtype == np.uint8

    # ------------------------------------------------------------
    # Exact frozen mask
    # ------------------------------------------------------------

    mask = generator.generate_hair_mask(
        image_id=image_id,
        severity=generator.PRIMARY_SEVERITY,
        global_seed=generator.GLOBAL_SEED,
        size=224,
    )

    mask = np.asarray(mask).astype(np.uint8)

    assert mask.shape == (224, 224)

    generated_area = int(
        (mask > 0).sum()
    )

    # ------------------------------------------------------------
    # Locked mask-area assertion
    # ------------------------------------------------------------

    assert generated_area == expected_area, (
        f"Mask area mismatch for {image_id}: "
        f"generated={generated_area}, "
        f"locked={expected_area}"
    )

    # ------------------------------------------------------------
    # Exact frozen hair rendering
    # ------------------------------------------------------------

    hair = generator.apply_hair(
        image=clean,
        mask=mask,
        severity=generator.PRIMARY_SEVERITY,
        global_seed=generator.GLOBAL_SEED,
        image_id=image_id,
    )

    hair = np.asarray(
        hair,
        dtype=np.uint8
    )

    assert hair.shape == (224, 224, 3)
    assert hair.dtype == np.uint8

    # ------------------------------------------------------------
    # Exact causal-integrity check
    # ------------------------------------------------------------

    changed = np.any(
        hair != clean,
        axis=2
    )

    mask_bool = mask > 0

    outside_changed = int(
        np.sum(
            changed & ~mask_bool
        )
    )

    assert outside_changed == 0, (
        f"Causal-integrity failure for {image_id}: "
        f"{outside_changed} pixels changed outside mask"
    )

    # ------------------------------------------------------------
    # Determinism check
    # ------------------------------------------------------------

    hair_repeat = generator.apply_hair(
        image=clean,
        mask=mask,
        severity=generator.PRIMARY_SEVERITY,
        global_seed=generator.GLOBAL_SEED,
        image_id=image_id,
    )

    hair_repeat = np.asarray(
        hair_repeat,
        dtype=np.uint8
    )

    assert np.array_equal(
        hair,
        hair_repeat
    ), (
        f"Non-deterministic hair rendering: {image_id}"
    )

    # ------------------------------------------------------------
    # Store
    # ------------------------------------------------------------

    clean_mm[r] = clean
    hair_mm[r] = hair
    mask_mm[r] = mask
    area_mm[r] = generated_area

    records.append(
        {
            "row": r,
            "image": image_id,
            "source_path": str(src),
            "mask_area_locked": expected_area,
            "mask_area_generated": generated_area,
            "outside_mask_changed_pixels": outside_changed,
        }
    )

    if (
        (r + 1) % 50 == 0
        or r == N - 1
    ):
        print(
            f"reconstructed {r + 1:3d}/{N}"
        )


# ----------------------------------------------------------------
# 6. Flush memory maps
# ----------------------------------------------------------------

clean_mm.flush()
hair_mm.flush()
mask_mm.flush()
area_mm.flush()

del clean_mm
del hair_mm
del mask_mm
del area_mm


# ----------------------------------------------------------------
# 7. Write index
# ----------------------------------------------------------------

index_df = pd.DataFrame(records)

index_df.to_csv(
    INDEX_OUT,
    index=False
)


# ----------------------------------------------------------------
# 8. Reload outputs and perform final structural audit
# ----------------------------------------------------------------

print("\n" + "=" * 70)
print("FINAL RECONSTRUCTION AUDIT")
print("=" * 70)

clean = np.load(
    CLEAN_OUT,
    mmap_mode="r"
)

hair = np.load(
    HAIR_OUT,
    mmap_mode="r"
)

mask = np.load(
    MASK_OUT,
    mmap_mode="r"
)

area = np.load(
    AREA_OUT,
    mmap_mode="r"
)

index_df = pd.read_csv(
    INDEX_OUT
)

print("clean:", clean.shape, clean.dtype)
print("hair :", hair.shape, hair.dtype)
print("mask :", mask.shape, mask.dtype)
print("area :", area.shape, area.dtype)
print("index:", index_df.shape)


# ----------------------------------------------------------------
# 9. Structural assertions
# ----------------------------------------------------------------

assert clean.shape == (
    500,
    224,
    224,
    3
)

assert hair.shape == (
    500,
    224,
    224,
    3
)

assert mask.shape == (
    500,
    224,
    224
)

assert area.shape == (500,)

assert clean.dtype == np.uint8
assert hair.dtype == np.uint8
assert mask.dtype == np.uint8
assert area.dtype == np.int32


# ----------------------------------------------------------------
# 10. Locked mask-area comparison
# ----------------------------------------------------------------

locked_area = xai["mask_area"].to_numpy(
    dtype=np.int32
)

assert np.array_equal(
    area,
    locked_area
)

print(
    "✓ 500/500 mask areas match locked XAI subset."
)


# ----------------------------------------------------------------
# 11. Full causal-integrity audit
#
# Compare all 500 reconstructed images.
# ----------------------------------------------------------------

outside_total = 0
changed_total = 0

for r in range(500):

    changed = np.any(
        hair[r] != clean[r],
        axis=2
    )

    mask_bool = mask[r] > 0

    changed_count = int(
        changed.sum()
    )

    outside_count = int(
        np.sum(
            changed & ~mask_bool
        )
    )

    changed_total += changed_count
    outside_total += outside_count

    assert outside_count == 0


print(
    "Total changed pixels inside/outside audit:"
)

print(
    "  changed pixels:",
    changed_total
)

print(
    "  changed outside mask:",
    outside_total
)

assert outside_total == 0

print(
    "✓ 500/500 causal-integrity checks passed."
)


# ----------------------------------------------------------------
# 12. Reconstruction record
# ----------------------------------------------------------------

record = {
    "protocol": "T1.4c_black_baseline_sensitivity_v1",
    "purpose": "500-image working cache reconstruction",
    "n_images": 500,
    "source_dir": str(SOURCE_DIR),
    "generator": str(GEN_PATH),
    "generator_sha256": (
        "32be8092c85458bed31a547f2ba6c57c3e1c2cf257d7cb0f76bd28eebc7b26c2"
    ),
    "global_seed": 20260919,
    "severity": "medium",
    "preprocessing": (
        "RGB -> resize (224,224) "
        "LANCZOS -> generate_hair_mask -> apply_hair"
    ),
    "numpy_version": np.__version__,
    "pil_version": __import__("PIL").__version__,
    "outputs": {
        "clean": str(CLEAN_OUT),
        "hair": str(HAIR_OUT),
        "mask": str(MASK_OUT),
        "mask_area": str(AREA_OUT),
        "index": str(INDEX_OUT),
    },
    "validation": {
        "row_mapping": "verified in Cell 10",
        "mask_area": "500/500 exact",
        "causal_integrity": "500/500 exact",
        "deterministic_hair_rendering": "500/500 exact",
    },
}

with open(
    RECORD_OUT,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        record,
        f,
        indent=2
    )


# ----------------------------------------------------------------
# 13. Final status
# ----------------------------------------------------------------

print("\n" + "=" * 70)
print("CELL 11 STATUS")
print("=" * 70)

print("500 clean images reconstructed       : PASS")
print("500 hair images reconstructed        : PASS")
print("500 masks reconstructed              : PASS")
print("500 mask areas exact                 : PASS")
print("Causal integrity                     : PASS")
print("Deterministic hair rendering         : PASS")
print("Working cache saved                  : PASS")

print("\nOutput directory:")
print(OUT_DIR)

print("\nNumPy:", np.__version__)
print("PIL  :", __import__("PIL").__version__)

print("\nNo model loaded : YES")
print("No IG computed  : YES")
print("No GPU used     : YES")

print("\nIMPORTANT:")
print("This is a 500-image T1.4c working cache.")
print("It is NOT being claimed byte-identical to the unavailable")
print("historical 23,227-image cache.")
print("The historical cache hashes remain locked as provenance.")
print("Do NOT run IG yet.")

print("\nREADY FOR CELL 11 OUTPUT REVIEW")

CELL 11 — RECONSTRUCT 500-IMAGE WORKING CACHE

Locked XAI images: 500
✓ Frozen generator loaded
✓ GLOBAL_SEED = 20260919
✓ PRIMARY_SEVERITY = medium

✓ Previous T1.4c reconstruction outputs cleared.

RECONSTRUCTING 500 IMAGES
reconstructed  50/500
reconstructed 100/500
reconstructed 150/500
reconstructed 200/500
reconstructed 250/500
reconstructed 300/500
reconstructed 350/500
reconstructed 400/500
reconstructed 450/500
reconstructed 500/500

FINAL RECONSTRUCTION AUDIT
clean: (500, 224, 224, 3) uint8
hair : (500, 224, 224, 3) uint8
mask : (500, 224, 224) uint8
area : (500,) int32
index: (500, 6)
✓ 500/500 mask areas match locked XAI subset.
Total changed pixels inside/outside audit:
  changed pixels: 262863
  changed outside mask: 0
✓ 500/500 causal-integrity checks passed.

CELL 11 STATUS
500 clean images reconstructed       : PASS
500 hair images reconstructed        : PASS
500 masks reconstructed              : PASS
500 mask areas exact                 : PASS
Causal integrity       

In [12]:
# ================================================================
# CELL 12 — FINAL INPUT-INTEGRITY GATE BEFORE IG
# ================================================================

from pathlib import Path
import pandas as pd
import numpy as np
import types
import ast
import hashlib

print("=" * 70)
print("CELL 12 — FINAL INPUT-INTEGRITY GATE BEFORE IG")
print("=" * 70)


# ----------------------------------------------------------------
# Paths
# ----------------------------------------------------------------

PROJECT_DATA_ROOT = Path(
    "/kaggle/input/datasets/tejasseshadriiiii/isic-final-project"
)

XAI_SUBSET_PATH = (
    PROJECT_DATA_ROOT
    / "step2U_xai_subset_lock"
    / "LOCKED_XAI_SUBSET.csv"
)

GEN_PATH = (
    PROJECT_DATA_ROOT
    / "step1J_generator_recovery_lock"
    / "recovered_original_hair_generator.py"
)

CACHE_DIR = Path(
    "/kaggle/working/t1_4c_xai_cache_500"
)

CLEAN_PATH = CACHE_DIR / "clean_500.npy"
HAIR_PATH = CACHE_DIR / "hair_500.npy"
MASK_PATH = CACHE_DIR / "mask_500.npy"
AREA_PATH = CACHE_DIR / "mask_area_500.npy"
INDEX_PATH = CACHE_DIR / "xai_cache_index_500.csv"


# ----------------------------------------------------------------
# 1. Load locked subset and reconstructed cache
# ----------------------------------------------------------------

xai = pd.read_csv(
    XAI_SUBSET_PATH
).reset_index(drop=True)

cache_index = pd.read_csv(
    INDEX_PATH
).reset_index(drop=True)

clean = np.load(
    CLEAN_PATH,
    mmap_mode="r"
)

hair = np.load(
    HAIR_PATH,
    mmap_mode="r"
)

mask = np.load(
    MASK_PATH,
    mmap_mode="r"
)

area = np.load(
    AREA_PATH,
    mmap_mode="r"
)

assert len(xai) == 500
assert len(cache_index) == 500

print("\nLoaded:")
print("  XAI subset :", xai.shape)
print("  cache index:", cache_index.shape)
print("  clean      :", clean.shape)
print("  hair       :", hair.shape)
print("  mask       :", mask.shape)
print("  area       :", area.shape)


# ----------------------------------------------------------------
# 2. Exact image identity/order audit
# ----------------------------------------------------------------

xai_images = (
    xai["image"]
    .astype(str)
    .tolist()
)

cache_images = (
    cache_index["image"]
    .astype(str)
    .tolist()
)

assert xai_images == cache_images, (
    "XAI subset image ordering differs from reconstructed cache."
)

print(
    "✓ Image identity/order: 500/500 exact."
)


# ----------------------------------------------------------------
# 3. Exact row-number audit
# ----------------------------------------------------------------

expected_rows = list(
    range(500)
)

actual_rows = (
    cache_index["row"]
    .astype(int)
    .tolist()
)

assert actual_rows == expected_rows

print(
    "✓ Working-cache row numbering: 0..499 exact."
)


# ----------------------------------------------------------------
# 4. Exact locked mask-area audit
# ----------------------------------------------------------------

locked_area = xai["mask_area"].to_numpy(
    dtype=np.int32
)

assert np.array_equal(
    area,
    locked_area
)

print(
    "✓ Mask area: 500/500 exact."
)


# ----------------------------------------------------------------
# 5. Frozen generator reload
# ----------------------------------------------------------------

source = GEN_PATH.read_text(
    encoding="utf-8"
)

tree = ast.parse(source)

safe_nodes = []

for node in tree.body:

    if isinstance(
        node,
        (ast.Import, ast.ImportFrom)
    ):
        safe_nodes.append(node)

    elif isinstance(
        node,
        ast.Assign
    ):

        try:
            ast.literal_eval(node.value)
            safe_nodes.append(node)
        except Exception:
            pass

    elif isinstance(
        node,
        ast.AnnAssign
    ):

        if node.value is not None:

            try:
                ast.literal_eval(node.value)
                safe_nodes.append(node)
            except Exception:
                pass

    elif isinstance(
        node,
        (ast.FunctionDef, ast.AsyncFunctionDef)
    ):
        safe_nodes.append(node)


module_ast = ast.Module(
    body=safe_nodes,
    type_ignores=[]
)

ast.fix_missing_locations(
    module_ast
)

generator = types.ModuleType(
    "frozen_generator_cell12"
)

generator.__file__ = str(
    GEN_PATH
)

exec(
    compile(
        module_ast,
        str(GEN_PATH),
        "exec"
    ),
    generator.__dict__
)

assert generator.GLOBAL_SEED == 20260919
assert generator.PRIMARY_SEVERITY == "medium"

print(
    "✓ Frozen generator reloaded."
)


# ----------------------------------------------------------------
# 6. Full 500-image mask equality audit
# ----------------------------------------------------------------

print(
    "\nRechecking all 500 reconstructed masks..."
)

mask_mismatches = 0
mask_area_mismatches = 0

for r, image_id in enumerate(
    xai_images
):

    regenerated = generator.generate_hair_mask(
        image_id=image_id,
        severity=generator.PRIMARY_SEVERITY,
        global_seed=generator.GLOBAL_SEED,
        size=224,
    )

    regenerated = np.asarray(
        regenerated,
        dtype=np.uint8
    )

    if not np.array_equal(
        regenerated,
        mask[r]
    ):
        mask_mismatches += 1

    regenerated_area = int(
        (regenerated > 0).sum()
    )

    if regenerated_area != int(
        area[r]
    ):
        mask_area_mismatches += 1


print(
    "Mask mismatches:",
    mask_mismatches
)

print(
    "Mask-area mismatches:",
    mask_area_mismatches
)

assert mask_mismatches == 0
assert mask_area_mismatches == 0

print(
    "✓ 500/500 masks exactly reproduce the frozen generator."
)


# ----------------------------------------------------------------
# 7. Array validity audit
# ----------------------------------------------------------------

assert clean.dtype == np.uint8
assert hair.dtype == np.uint8
assert mask.dtype == np.uint8
assert area.dtype == np.int32

assert np.isfinite(
    clean.astype(np.float32)
).all()

assert np.isfinite(
    hair.astype(np.float32)
).all()

assert np.isfinite(
    mask.astype(np.float32)
).all()

assert np.isfinite(
    area.astype(np.float32)
).all()

assert int(clean.min()) >= 0
assert int(clean.max()) <= 255

assert int(hair.min()) >= 0
assert int(hair.max()) <= 255

assert set(
    np.unique(mask)
).issubset({0, 1, 255})

print(
    "✓ Array dtype/range/finiteness audit passed."
)


# ----------------------------------------------------------------
# 8. Full causal-integrity audit
# ----------------------------------------------------------------

outside_changed_total = 0
inside_changed_total = 0

for r in range(500):

    changed = np.any(
        hair[r] != clean[r],
        axis=2
    )

    mask_bool = mask[r] > 0

    inside = changed & mask_bool
    outside = changed & ~mask_bool

    inside_changed_total += int(
        inside.sum()
    )

    outside_changed_total += int(
        outside.sum()
    )

assert outside_changed_total == 0

print(
    "✓ Full causal-integrity audit passed."
)

print(
    "  changed inside mask :",
    inside_changed_total
)

print(
    "  changed outside mask:",
    outside_changed_total
)


# ----------------------------------------------------------------
# 9. Saved-file reopen test
# ----------------------------------------------------------------

clean2 = np.load(
    CLEAN_PATH,
    mmap_mode="r"
)

hair2 = np.load(
    HAIR_PATH,
    mmap_mode="r"
)

mask2 = np.load(
    MASK_PATH,
    mmap_mode="r"
)

area2 = np.load(
    AREA_PATH,
    mmap_mode="r"
)

assert np.array_equal(
    clean2,
    clean
)

assert np.array_equal(
    hair2,
    hair
)

assert np.array_equal(
    mask2,
    mask
)

assert np.array_equal(
    area2,
    area
)

print(
    "✓ Saved-cache reopen test passed."
)


# ----------------------------------------------------------------
# 10. Final status
# ----------------------------------------------------------------

print("\n" + "=" * 70)
print("CELL 12 STATUS")
print("=" * 70)

print("Image identity/order              : VERIFIED")
print("Row numbering                     : VERIFIED")
print("Locked mask areas                 : VERIFIED")
print("Frozen generator masks            : 500/500 EXACT")
print("Array dtype/range/finiteness      : VERIFIED")
print("Causal integrity                  : VERIFIED")
print("Saved-cache reopen                : VERIFIED")

print("\nNo model loaded : YES")
print("No IG computed  : YES")
print("No GPU used     : YES")

print("\nREADY FOR IG STAGE.")
print("Do NOT compute IG in this cell.")
print("Wait for Cell 12 output review.")

CELL 12 — FINAL INPUT-INTEGRITY GATE BEFORE IG

Loaded:
  XAI subset : (500, 8)
  cache index: (500, 6)
  clean      : (500, 224, 224, 3)
  hair       : (500, 224, 224, 3)
  mask       : (500, 224, 224)
  area       : (500,)
✓ Image identity/order: 500/500 exact.
✓ Working-cache row numbering: 0..499 exact.
✓ Mask area: 500/500 exact.
✓ Frozen generator reloaded.

Rechecking all 500 reconstructed masks...
Mask mismatches: 0
Mask-area mismatches: 0
✓ 500/500 masks exactly reproduce the frozen generator.
✓ Array dtype/range/finiteness audit passed.
✓ Full causal-integrity audit passed.
  changed inside mask : 262863
  changed outside mask: 0
✓ Saved-cache reopen test passed.

CELL 12 STATUS
Image identity/order              : VERIFIED
Row numbering                     : VERIFIED
Locked mask areas                 : VERIFIED
Frozen generator masks            : 500/500 EXACT
Array dtype/range/finiteness      : VERIFIED
Causal integrity                  : VERIFIED
Saved-cache reopen         

In [13]:
# ================================================================
# CELL 13 — PRODUCTION CHECKPOINT IDENTITY AUDIT
# ================================================================

from pathlib import Path
import hashlib
import json
import pandas as pd
import torch


print("=" * 70)
print("CELL 13 — PRODUCTION CHECKPOINT IDENTITY AUDIT")
print("=" * 70)


# ----------------------------------------------------------------
# 1. Expected production model set
# ----------------------------------------------------------------

RHO_LEVELS = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
N_REPLICATES = 5

expected_models = [
    f"rho{rho:.1f}_rep{rep}"
    for rho in RHO_LEVELS
    for rep in range(1, N_REPLICATES + 1)
]

print("\nExpected production models:", len(expected_models))

for m in expected_models:
    print(" ", m)


# ----------------------------------------------------------------
# 2. Known frozen checkpoint roots
# ----------------------------------------------------------------

ROOT_A = Path(
    "/kaggle/input/notebooks/tejasseshadriiiii/"
    "ml-open-lab-30m-trained/training_runs"
)

ROOT_B = Path(
    "/kaggle/input/notebooks/tejasseshadriiiii/"
    "isic-kaggle-analysis/trained_models_readonly/training_runs"
)

print("\nCheckpoint roots:")
print("A:", ROOT_A)
print("B:", ROOT_B)

assert ROOT_A.exists(), f"Missing checkpoint root A: {ROOT_A}"
assert ROOT_B.exists(), f"Missing checkpoint root B: {ROOT_B}"

print("✓ Both checkpoint roots exist.")


# ----------------------------------------------------------------
# 3. Locate ONLY final_epoch15.pt
#
# IMPORTANT:
# best_eval_ba.pt is deliberately excluded.
# ----------------------------------------------------------------

def find_final_checkpoint(root, model_name):

    matches = sorted(
        root.glob(
            f"**/{model_name}/final_epoch15.pt"
        )
    )

    return matches


records = []

print("\n" + "=" * 70)
print("CHECKPOINT DISCOVERY")
print("=" * 70)

for model_name in expected_models:

    a = find_final_checkpoint(
        ROOT_A,
        model_name
    )

    b = find_final_checkpoint(
        ROOT_B,
        model_name
    )

    print(
        f"{model_name:16s} "
        f"A={len(a)} "
        f"B={len(b)}"
    )

    assert len(a) == 1, (
        f"Expected exactly one final_epoch15.pt "
        f"in root A for {model_name}, found {len(a)}"
    )

    assert len(b) == 1, (
        f"Expected exactly one final_epoch15.pt "
        f"in root B for {model_name}, found {len(b)}"
    )

    records.append(
        {
            "model": model_name,
            "root_A": str(a[0]),
            "root_B": str(b[0]),
        }
    )


assert len(records) == 30

print(
    "\n✓ Exactly 30 production checkpoints found "
    "in each root."
)


# ----------------------------------------------------------------
# 4. SHA256 byte identity across the two roots
# ----------------------------------------------------------------

def sha256_file(path, chunk_size=1024 * 1024):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        while True:

            chunk = f.read(chunk_size)

            if not chunk:
                break

            h.update(chunk)

    return h.hexdigest()


print("\n" + "=" * 70)
print("CROSS-ROOT BYTE IDENTITY")
print("=" * 70)

for rec in records:

    sha_a = sha256_file(
        rec["root_A"]
    )

    sha_b = sha256_file(
        rec["root_B"]
    )

    rec["sha256_A"] = sha_a
    rec["sha256_B"] = sha_b

    assert sha_a == sha_b, (
        f"Checkpoint byte mismatch: "
        f"{rec['model']}\n"
        f"A={sha_a}\n"
        f"B={sha_b}"
    )

    print(
        f"{rec['model']:16s} "
        f"{sha_a}"
    )

print(
    "\n✓ All 30 A/B checkpoint pairs are byte-identical."
)


# ----------------------------------------------------------------
# 5. Verify filenames are exactly final_epoch15.pt
# ----------------------------------------------------------------

for rec in records:

    assert Path(
        rec["root_A"]
    ).name == "final_epoch15.pt"

    assert Path(
        rec["root_B"]
    ).name == "final_epoch15.pt"

print(
    "✓ All production checkpoints are final_epoch15.pt."
)


# ----------------------------------------------------------------
# 6. Confirm no accidental best_eval_ba selection
# ----------------------------------------------------------------

best_files_A = list(
    ROOT_A.glob(
        "**/best_eval_ba.pt"
    )
)

best_files_B = list(
    ROOT_B.glob(
        "**/best_eval_ba.pt"
    )
)

print("\nbest_eval_ba.pt files discovered:")
print("  Root A:", len(best_files_A))
print("  Root B:", len(best_files_B))

print(
    "✓ best_eval_ba.pt is NOT used by this audit."
)


# ----------------------------------------------------------------
# 7. Check checkpoint file sizes
# ----------------------------------------------------------------

sizes = []

for rec in records:

    size_a = Path(
        rec["root_A"]
    ).stat().st_size

    size_b = Path(
        rec["root_B"]
    ).stat().st_size

    assert size_a == size_b

    sizes.append(size_a)

    rec["size_bytes"] = size_a


print("\nCheckpoint size:")
print(
    "  min:",
    min(sizes),
    "bytes"
)

print(
    "  max:",
    max(sizes),
    "bytes"
)

assert min(sizes) == max(sizes)

print(
    "✓ All 30 production checkpoints have identical file size."
)


# ----------------------------------------------------------------
# 8. Load ONE checkpoint on CPU only
#
# This is a structural dry-run, NOT model inference.
# ----------------------------------------------------------------

probe_path = Path(
    records[0]["root_A"]
)

print("\n" + "=" * 70)
print("CPU CHECKPOINT STRUCTURE PROBE")
print("=" * 70)

print(
    "Probe model:",
    records[0]["model"]
)

print(
    "Probe path:",
    probe_path
)

checkpoint = torch.load(
    probe_path,
    map_location="cpu",
    weights_only=False
)

print(
    "\nCheckpoint Python type:",
    type(checkpoint)
)


# ----------------------------------------------------------------
# 9. Inspect top-level checkpoint structure
# ----------------------------------------------------------------

if isinstance(
    checkpoint,
    dict
):

    print(
        "\nTop-level keys:"
    )

    for key in checkpoint.keys():
        print(
            " ",
            key,
            "->",
            type(checkpoint[key]).__name__
        )

else:

    print(
        "\nCheckpoint is not a dictionary."
    )


# ----------------------------------------------------------------
# 10. Identify state_dict
# ----------------------------------------------------------------

state_dict = None
state_dict_key = None

if isinstance(
    checkpoint,
    dict
):

    candidate_keys = [
        "state_dict",
        "model_state_dict",
        "model",
        "net",
    ]

    for key in candidate_keys:

        value = checkpoint.get(
            key
        )

        if isinstance(
            value,
            dict
        ):

            tensor_values = [
                v
                for v in value.values()
                if torch.is_tensor(v)
            ]

            if len(tensor_values) > 0:

                state_dict = value
                state_dict_key = key
                break


if state_dict is None:

    if isinstance(
        checkpoint,
        dict
    ):

        tensor_values = [
            v
            for v in checkpoint.values()
            if torch.is_tensor(v)
        ]

        if len(tensor_values) > 0:

            state_dict = checkpoint
            state_dict_key = "<top-level>"

assert state_dict is not None, (
    "Could not identify a tensor state_dict "
    "inside the production checkpoint."
)

print(
    "\n✓ State dict identified:"
)

print(
    "  source:",
    state_dict_key
)

print(
    "  tensors:",
    len(state_dict)
)


# ----------------------------------------------------------------
# 11. Inspect ResNet/head keys
# ----------------------------------------------------------------

keys = list(
    state_dict.keys()
)

print(
    "\nFirst 20 state-dict keys:"
)

for key in keys[:20]:
    print(
        " ",
        key
    )


# ----------------------------------------------------------------
# 12. Identify final classification layer
# ----------------------------------------------------------------

fc_candidates = [
    key
    for key in keys
    if key.endswith(
        "fc.weight"
    )
]

print(
    "\nFC weight candidates:",
    fc_candidates
)

assert len(fc_candidates) == 1, (
    f"Expected exactly one fc.weight, "
    f"found {fc_candidates}"
)

fc_weight_key = fc_candidates[0]

fc_weight = state_dict[
    fc_weight_key
]

print(
    "FC weight shape:",
    tuple(fc_weight.shape)
)

assert fc_weight.ndim == 2
assert fc_weight.shape[0] == 2

print(
    "✓ Classification head has exactly 2 outputs."
)


# ----------------------------------------------------------------
# 13. FC bias
# ----------------------------------------------------------------

fc_bias_key = fc_weight_key.replace(
    "weight",
    "bias"
)

assert fc_bias_key in state_dict

fc_bias = state_dict[
    fc_bias_key
]

print(
    "FC bias shape:",
    tuple(fc_bias.shape)
)

assert tuple(
    fc_bias.shape
) == (2,)

print(
    "✓ Two-class output head verified."
)


# ----------------------------------------------------------------
# 14. Save checkpoint manifest
# ----------------------------------------------------------------

manifest_df = pd.DataFrame(
    records
)

MANIFEST_PATH = Path(
    "/kaggle/working/"
    "T1_4c_checkpoint_identity_manifest.csv"
)

manifest_df.to_csv(
    MANIFEST_PATH,
    index=False
)

JSON_PATH = Path(
    "/kaggle/working/"
    "T1_4c_checkpoint_identity_manifest.json"
)

with open(
    JSON_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        {
            "protocol":
                "T1.4c_black_baseline_sensitivity_v1",
            "checkpoint":
                "final_epoch15.pt",
            "n_models":
                30,
            "roots": [
                str(ROOT_A),
                str(ROOT_B),
            ],
            "all_cross_root_byte_identical":
                True,
            "probe_model":
                records[0]["model"],
            "probe_state_dict_source":
                state_dict_key,
            "probe_fc_weight_shape":
                list(fc_weight.shape),
            "probe_fc_bias_shape":
                list(fc_bias.shape),
        },
        f,
        indent=2
    )


# ----------------------------------------------------------------
# FINAL STATUS
# ----------------------------------------------------------------

print("\n" + "=" * 70)
print("CELL 13 STATUS")
print("=" * 70)

print(
    "Expected production models       : 30"
)

print(
    "Root A checkpoints               : 30"
)

print(
    "Root B checkpoints               : 30"
)

print(
    "A/B byte-identical pairs         : 30/30"
)

print(
    "Production checkpoint            : final_epoch15.pt"
)

print(
    "2-output classification head     : VERIFIED"
)

print(
    "CPU checkpoint structural probe  : PASS"
)

print(
    "\nNo GPU inference : YES"
)

print(
    "No IG computed   : YES"
)

print(
    "\nManifest:"
)

print(
    MANIFEST_PATH
)

print(
    JSON_PATH
)

print(
    "\nREADY FOR CELL 13 OUTPUT REVIEW."
)

CELL 13 — PRODUCTION CHECKPOINT IDENTITY AUDIT

Expected production models: 30
  rho0.0_rep1
  rho0.0_rep2
  rho0.0_rep3
  rho0.0_rep4
  rho0.0_rep5
  rho0.2_rep1
  rho0.2_rep2
  rho0.2_rep3
  rho0.2_rep4
  rho0.2_rep5
  rho0.4_rep1
  rho0.4_rep2
  rho0.4_rep3
  rho0.4_rep4
  rho0.4_rep5
  rho0.6_rep1
  rho0.6_rep2
  rho0.6_rep3
  rho0.6_rep4
  rho0.6_rep5
  rho0.8_rep1
  rho0.8_rep2
  rho0.8_rep3
  rho0.8_rep4
  rho0.8_rep5
  rho1.0_rep1
  rho1.0_rep2
  rho1.0_rep3
  rho1.0_rep4
  rho1.0_rep5

Checkpoint roots:
A: /kaggle/input/notebooks/tejasseshadriiiii/ml-open-lab-30m-trained/training_runs
B: /kaggle/input/notebooks/tejasseshadriiiii/isic-kaggle-analysis/trained_models_readonly/training_runs
✓ Both checkpoint roots exist.

CHECKPOINT DISCOVERY
rho0.0_rep1      A=0 B=0


AssertionError: Expected exactly one final_epoch15.pt in root A for rho0.0_rep1, found 0

In [14]:
# ================================================================
# CELL 13R — ACTUAL PRODUCTION CHECKPOINT DISCOVERY
# ================================================================

from pathlib import Path
import hashlib
import pandas as pd

print("=" * 70)
print("CELL 13R — ACTUAL PRODUCTION CHECKPOINT DISCOVERY")
print("=" * 70)


ROOT_A = Path(
    "/kaggle/input/notebooks/tejasseshadriiiii/"
    "ml-open-lab-30m-trained/training_runs"
)

ROOT_B = Path(
    "/kaggle/input/notebooks/tejasshadriiiii/"
    "isic-kaggle-analysis/trained_models_readonly/training_runs"
)

# Correct the second path if the directory exists under the known
# spelling from the earlier audit.
if not ROOT_B.exists():

    ROOT_B = Path(
        "/kaggle/input/notebooks/tejasseshadriiiii/"
        "isic-kaggle-analysis/trained_models_readonly/training_runs"
    )


assert ROOT_A.exists(), f"Root A missing: {ROOT_A}"
assert ROOT_B.exists(), f"Root B missing: {ROOT_B}"

print("\nRoot A:", ROOT_A)
print("Root B:", ROOT_B)


# ----------------------------------------------------------------
# Discover every final_epoch15.pt recursively
# ----------------------------------------------------------------

files_A = sorted(
    ROOT_A.glob(
        "**/final_epoch15.pt"
    )
)

files_B = sorted(
    ROOT_B.glob(
        "**/final_epoch15.pt"
    )
)

print("\n" + "=" * 70)
print("DISCOVERY")
print("=" * 70)

print(
    "Root A final_epoch15.pt:",
    len(files_A)
)

print(
    "Root B final_epoch15.pt:",
    len(files_B)
)

assert len(files_A) == 30, (
    f"Expected 30 production checkpoints in A, "
    f"found {len(files_A)}"
)

assert len(files_B) == 30, (
    f"Expected 30 production checkpoints in B, "
    f"found {len(files_B)}"
)

print(
    "✓ 30 final_epoch15.pt files found in each root."
)


# ----------------------------------------------------------------
# Print actual directory/model names
# ----------------------------------------------------------------

print("\n" + "=" * 70)
print("ACTUAL ROOT A CHECKPOINT PATHS")
print("=" * 70)

for i, p in enumerate(files_A, 1):

    print(
        f"{i:02d}. {p}"
    )


print("\n" + "=" * 70)
print("ACTUAL ROOT B CHECKPOINT PATHS")
print("=" * 70)

for i, p in enumerate(files_B, 1):

    print(
        f"{i:02d}. {p}"
    )


# ----------------------------------------------------------------
# Extract useful path components
# ----------------------------------------------------------------

print("\n" + "=" * 70)
print("PATH STRUCTURE")
print("=" * 70)

for p in files_A:

    rel = p.relative_to(ROOT_A)

    print(
        "relative:",
        rel
    )

    print(
        "  parent:",
        p.parent.name
    )

    print(
        "  grandparent:",
        p.parent.parent.name
    )

    print()


# ----------------------------------------------------------------
# Check whether A/B contain the same relative checkpoint paths
# ----------------------------------------------------------------

rel_A = {
    str(p.relative_to(ROOT_A))
    for p in files_A
}

rel_B = {
    str(p.relative_to(ROOT_B))
    for p in files_B
}

print("=" * 70)
print("RELATIVE PATH COMPARISON")
print("=" * 70)

print(
    "A relative paths:",
    len(rel_A)
)

print(
    "B relative paths:",
    len(rel_B)
)

print(
    "Intersection:",
    len(rel_A & rel_B)
)

print(
    "A-only:",
    len(rel_A - rel_B)
)

print(
    "B-only:",
    len(rel_B - rel_A)
)


if rel_A != rel_B:

    print("\n⚠ Relative checkpoint paths differ.")

    if rel_A - rel_B:

        print("\nA-only examples:")
        for x in sorted(rel_A - rel_B)[:10]:
            print(" ", x)

    if rel_B - rel_A:

        print("\nB-only examples:")
        for x in sorted(rel_B - rel_A)[:10]:
            print(" ", x)

else:

    print(
        "✓ Root A and Root B have identical "
        "relative checkpoint paths."
    )


# ----------------------------------------------------------------
# Check for best_eval_ba.pt separately
# ----------------------------------------------------------------

best_A = sorted(
    ROOT_A.glob(
        "**/best_eval_ba.pt"
    )
)

best_B = sorted(
    ROOT_B.glob(
        "**/best_eval_ba.pt"
    )
)

print("\n" + "=" * 70)
print("BEST-CHECKPOINT DISCOVERY")
print("=" * 70)

print(
    "best_eval_ba.pt in A:",
    len(best_A)
)

print(
    "best_eval_ba.pt in B:",
    len(best_B)
)


# ----------------------------------------------------------------
# Final status
# ----------------------------------------------------------------

print("\n" + "=" * 70)
print("CELL 13R STATUS")
print("=" * 70)

print(
    "Production final_epoch15 files A :",
    len(files_A)
)

print(
    "Production final_epoch15 files B :",
    len(files_B)
)

print(
    "No model loaded                   : YES"
)

print(
    "No GPU used                       : YES"
)

print(
    "No IG computed                    : YES"
)

print(
    "\nIMPORTANT:"
)

print(
    "This cell only discovers the actual "
    "checkpoint naming/path convention."
)

print(
    "Do NOT rename or move any checkpoints."
)

print(
    "\nREADY FOR CELL 13R OUTPUT REVIEW."
)

CELL 13R — ACTUAL PRODUCTION CHECKPOINT DISCOVERY

Root A: /kaggle/input/notebooks/tejasseshadriiiii/ml-open-lab-30m-trained/training_runs
Root B: /kaggle/input/notebooks/tejasseshadriiiii/isic-kaggle-analysis/trained_models_readonly/training_runs

DISCOVERY
Root A final_epoch15.pt: 30
Root B final_epoch15.pt: 30
✓ 30 final_epoch15.pt files found in each root.

ACTUAL ROOT A CHECKPOINT PATHS
01. /kaggle/input/notebooks/tejasseshadriiiii/ml-open-lab-30m-trained/training_runs/rho_0.0_rep_1/final_epoch15.pt
02. /kaggle/input/notebooks/tejasseshadriiiii/ml-open-lab-30m-trained/training_runs/rho_0.0_rep_2/final_epoch15.pt
03. /kaggle/input/notebooks/tejasseshadriiiii/ml-open-lab-30m-trained/training_runs/rho_0.0_rep_3/final_epoch15.pt
04. /kaggle/input/notebooks/tejasseshadriiiii/ml-open-lab-30m-trained/training_runs/rho_0.0_rep_4/final_epoch15.pt
05. /kaggle/input/notebooks/tejasseshadriiiii/ml-open-lab-30m-trained/training_runs/rho_0.0_rep_5/final_epoch15.pt
06. /kaggle/input/notebooks/te

In [15]:
# ================================================================
# CELL 13B — CHECKPOINT BYTE IDENTITY + STRUCTURAL PROBE
# ================================================================

from pathlib import Path
import hashlib
import json
import pandas as pd
import torch


print("=" * 70)
print("CELL 13B — CHECKPOINT BYTE IDENTITY + STRUCTURAL PROBE")
print("=" * 70)


# ----------------------------------------------------------------
# Frozen roots from Cell 13R
# ----------------------------------------------------------------

ROOT_A = Path(
    "/kaggle/input/notebooks/tejasseshadriiiii/"
    "ml-open-lab-30m-trained/training_runs"
)

ROOT_B = Path(
    "/kaggle/input/notebooks/tejasshadriiiii/"
    "isic-kaggle-analysis/trained_models_readonly/training_runs"
)

# Correct spelling used by Cell 13R
ROOT_B = Path(
    "/kaggle/input/notebooks/tejassh.../"
    "isic-kaggle-analysis/trained_models_readonly/training_runs"
)

# Resolve the actual existing root instead of relying on a typo.
if not ROOT_B.exists():

    candidates = list(
        Path("/kaggle/input/notebooks").glob(
            "*/isic-kaggle-analysis/"
            "trained_models_readonly/training_runs"
        )
    )

    assert len(candidates) == 1, (
        "Could not uniquely resolve Root B."
    )

    ROOT_B = candidates[0]


assert ROOT_A.exists()
assert ROOT_B.exists()

print("\nRoot A:", ROOT_A)
print("Root B:", ROOT_B)


# ----------------------------------------------------------------
# Discover exact relative checkpoint paths
# ----------------------------------------------------------------

files_A = sorted(
    ROOT_A.glob(
        "**/final_epoch15.pt"
    )
)

files_B = sorted(
    ROOT_B.glob(
        "**/final_epoch15.pt"
    )
)

assert len(files_A) == 30
assert len(files_B) == 30

rel_A = {
    str(p.relative_to(ROOT_A)): p
    for p in files_A
}

rel_B = {
    str(p.relative_to(ROOT_B)): p
    for p in files_B
}

assert set(rel_A) == set(rel_B)

print(
    "\n✓ 30 matching relative checkpoint paths."
)


# ----------------------------------------------------------------
# SHA256 helper
# ----------------------------------------------------------------

def sha256_file(
    path,
    chunk_size=1024 * 1024
):

    h = hashlib.sha256()

    with open(
        path,
        "rb"
    ) as f:

        while True:

            chunk = f.read(
                chunk_size
            )

            if not chunk:
                break

            h.update(chunk)

    return h.hexdigest()


# ----------------------------------------------------------------
# Full byte identity audit
# ----------------------------------------------------------------

print("\n" + "=" * 70)
print("A/B BYTE IDENTITY AUDIT")
print("=" * 70)

records = []

for rel in sorted(rel_A):

    pa = rel_A[rel]
    pb = rel_B[rel]

    sha_a = sha256_file(pa)
    sha_b = sha256_file(pb)

    size_a = pa.stat().st_size
    size_b = pb.stat().st_size

    assert size_a == size_b
    assert sha_a == sha_b

    model_name = Path(rel).parent.name

    records.append(
        {
            "model": model_name,
            "relative_path": rel,
            "root_A": str(pa),
            "root_B": str(pb),
            "size_bytes": size_a,
            "sha256": sha_a,
        }
    )

    print(
        f"{model_name:16s} "
        f"{size_a:10d} bytes "
        f"{sha_a}"
    )

assert len(records) == 30

print(
    "\n✓ 30/30 checkpoint pairs are byte-identical."
)


# ----------------------------------------------------------------
# Checkpoint sizes
# ----------------------------------------------------------------

sizes = [
    r["size_bytes"]
    for r in records
]

print(
    "\nCheckpoint size range:",
    min(sizes),
    "to",
    max(sizes),
    "bytes"
)


# ----------------------------------------------------------------
# CPU-only structural probe
# ----------------------------------------------------------------

probe = rel_A[
    "rho_0.0_rep_1/final_epoch15.pt"
]

print("\n" + "=" * 70)
print("CPU-ONLY CHECKPOINT PROBE")
print("=" * 70)

print(
    "Probe:",
    probe
)

checkpoint = torch.load(
    probe,
    map_location="cpu",
    weights_only=False
)

print(
    "Checkpoint type:",
    type(checkpoint).__name__
)


# ----------------------------------------------------------------
# Identify state dictionary
# ----------------------------------------------------------------

state_dict = None
state_source = None

if isinstance(
    checkpoint,
    dict
):

    print(
        "\nTop-level keys:"
    )

    for k in checkpoint.keys():

        print(
            f"  {k}: "
            f"{type(checkpoint[k]).__name__}"
        )

    for candidate in [
        "state_dict",
        "model_state_dict",
        "model",
        "net",
    ]:

        value = checkpoint.get(
            candidate
        )

        if isinstance(
            value,
            dict
        ):

            tensors = [
                v
                for v in value.values()
                if torch.is_tensor(v)
            ]

            if tensors:

                state_dict = value
                state_source = candidate
                break

    # Some checkpoints store the state dict directly.
    if state_dict is None:

        tensor_values = [
            v
            for v in checkpoint.values()
            if torch.is_tensor(v)
        ]

        if tensor_values:

            state_dict = checkpoint
            state_source = "<top-level>"


assert state_dict is not None, (
    "No tensor state_dict could be identified."
)

print(
    "\n✓ State dict source:",
    state_source
)

print(
    "✓ State-dict tensor count:",
    len(state_dict)
)


# ----------------------------------------------------------------
# Inspect key structure
# ----------------------------------------------------------------

keys = list(
    state_dict.keys()
)

print(
    "\nFirst 20 state-dict keys:"
)

for k in keys[:20]:

    print(
        " ",
        k
    )


# ----------------------------------------------------------------
# Find classification head
# ----------------------------------------------------------------

fc_weights = [
    k
    for k in keys
    if k.endswith(
        "fc.weight"
    )
]

fc_biases = [
    k
    for k in keys
    if k.endswith(
        "fc.bias"
    )
]

print(
    "\nFC weight keys:",
    fc_weights
)

print(
    "FC bias keys:",
    fc_biases
)

assert len(fc_weights) == 1
assert len(fc_biases) == 1

fc_weight_key = fc_weights[0]
fc_bias_key = fc_biases[0]

fc_weight = state_dict[
    fc_weight_key
]

fc_bias = state_dict[
    fc_bias_key
]

print(
    "FC weight shape:",
    tuple(fc_weight.shape)
)

print(
    "FC bias shape:",
    tuple(fc_bias.shape)
)

assert tuple(
    fc_weight.shape
) == (2, 512)

assert tuple(
    fc_bias.shape
) == (2,)

print(
    "✓ ResNet-18-style 512 → 2 classification head verified."
)


# ----------------------------------------------------------------
# Check representative backbone keys
# ----------------------------------------------------------------

required_prefixes = [
    "conv1.",
    "bn1.",
    "layer1.",
    "layer2.",
    "layer3.",
    "layer4.",
]

for prefix in required_prefixes:

    assert any(
        k.startswith(prefix)
        for k in keys
    ), (
        f"Missing expected backbone prefix: {prefix}"
    )

print(
    "✓ ResNet backbone key structure verified."
)


# ----------------------------------------------------------------
# Save manifest
# ----------------------------------------------------------------

manifest_df = pd.DataFrame(
    records
)

CSV_OUT = Path(
    "/kaggle/working/"
    "T1_4c_checkpoint_identity_manifest.csv"
)

JSON_OUT = Path(
    "/kaggle/working/"
    "T1_4c_checkpoint_identity_manifest.json"
)

manifest_df.to_csv(
    CSV_OUT,
    index=False
)

with open(
    JSON_OUT,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        {
            "protocol":
                "T1.4c_black_baseline_sensitivity_v1",
            "checkpoint":
                "final_epoch15.pt",
            "n_models":
                30,
            "byte_identical_A_B":
                True,
            "probe_model":
                "rho_0.0_rep_1",
            "state_dict_source":
                state_source,
            "fc_weight_shape":
                list(fc_weight.shape),
            "fc_bias_shape":
                list(fc_bias.shape),
        },
        f,
        indent=2
    )


# ----------------------------------------------------------------
# FINAL STATUS
# ----------------------------------------------------------------

print("\n" + "=" * 70)
print("CELL 13B STATUS")
print("=" * 70)

print(
    "30 production checkpoints found       : PASS"
)

print(
    "A/B relative paths identical           : PASS"
)

print(
    "A/B byte identity                      : 30/30 PASS"
)

print(
    "final_epoch15 selection                : VERIFIED"
)

print(
    "CPU checkpoint load                    : PASS"
)

print(
    "2-output classification head           : VERIFIED"
)

print(
    "ResNet backbone structure              : VERIFIED"
)

print(
    "\nGPU inference : NO"
)

print(
    "IG computation: NO"
)

print(
    "\nManifest:"
)

print(
    CSV_OUT
)

print(
    JSON_OUT
)

print(
    "\nREADY FOR CELL 13B OUTPUT REVIEW."
)

CELL 13B — CHECKPOINT BYTE IDENTITY + STRUCTURAL PROBE

Root A: /kaggle/input/notebooks/tejasseshadriiiii/ml-open-lab-30m-trained/training_runs
Root B: /kaggle/input/notebooks/tejasseshadriiiii/isic-kaggle-analysis/trained_models_readonly/training_runs

✓ 30 matching relative checkpoint paths.

A/B BYTE IDENTITY AUDIT
rho_0.0_rep_1      44790155 bytes 7542507471917b74bdfbd44310fdc3ccdd09fe5cb89f1a88de40078452d633c6
rho_0.0_rep_2      44790155 bytes ab828644cccf1f64b8ee13bb424551935d8942e3d64bbe570ad53c2d8a37abe4
rho_0.0_rep_3      44790155 bytes 79d85c503a08141cc49d44eb7e058ad0c170dbb84816e8f7d751c6690c52e574
rho_0.0_rep_4      44790155 bytes 191f800f46d7fb838fc2088605c620f1704c138ff098d29246b38c67f3125d56
rho_0.0_rep_5      44790155 bytes 73d67398d280b68897c6873889be735d232fb80adcf7601bf50274b73fc8fcb3
rho_0.2_rep_1      44790155 bytes acf9df52c142441d16bcde0654951f9851f1fb6c3f8df7b67ae2fdf3fdec6c7c
rho_0.2_rep_2      44790155 bytes 5c9103d1e168f20f6f46104c059102271e92102c611f650f3ea7

In [16]:
# ================================================================
# CELL 14 — SINGLE-MODEL / SINGLE-IMAGE IG DRY RUN
# ================================================================

from pathlib import Path
import ast
import types
import hashlib
import math

import numpy as np
import pandas as pd
import torch
import torchvision
import torchvision.transforms.functional as TF


print("=" * 70)
print("CELL 14 — SINGLE-MODEL / SINGLE-IMAGE IG DRY RUN")
print("=" * 70)


# ----------------------------------------------------------------
# 1. Paths
# ----------------------------------------------------------------

PROJECT_ROOT = Path(
    "/kaggle/input/datasets/tejassh.../"
    "isic-final-project"
)

# Resolve actual project root safely.
if not PROJECT_ROOT.exists():

    candidates = list(
        Path("/kaggle/input/datasets").glob(
            "*/isic-final-project"
        )
    )

    assert len(candidates) >= 1, (
        "Could not locate isic-final-project."
    )

    # Prefer the known Tejasseshadri project.
    preferred = [
        p for p in candidates
        if "tejassh" in str(p).lower()
    ]

    PROJECT_ROOT = (
        preferred[0]
        if preferred
        else candidates[0]
    )

print(
    "\nProject root:",
    PROJECT_ROOT
)


GEN_PATH = (
    PROJECT_ROOT
    / "step1J_generator_recovery_lock"
    / "recovered_original_hair_generator.py"
)

ANALYSIS_PATH = (
    PROJECT_ROOT
    / "step2W_analysis_code_freeze"
    / "analysis_pipeline.py"
)

XAI_SUBSET_PATH = (
    PROJECT_ROOT
    / "step2U_xai_subset_lock"
    / "LOCKED_XAI_SUBSET.csv"
)

CACHE_DIR = Path(
    "/kaggle/working/t1_4c_xai_cache_500"
)

CLEAN_PATH = CACHE_DIR / "clean_500.npy"
HAIR_PATH = CACHE_DIR / "hair_500.npy"
MASK_PATH = CACHE_DIR / "mask_500.npy"
AREA_PATH = CACHE_DIR / "mask_area_500.npy"


CHECKPOINT = Path(
    "/kaggle/input/notebooks/"
    "tejasseshadriiiii/"
    "ml-open-lab-30m-trained/"
    "training_runs/"
    "rho_0.0_rep_1/"
    "final_epoch15.pt"
)

assert GEN_PATH.exists()
assert ANALYSIS_PATH.exists()
assert XAI_SUBSET_PATH.exists()
assert CLEAN_PATH.exists()
assert HAIR_PATH.exists()
assert MASK_PATH.exists()
assert AREA_PATH.exists()
assert CHECKPOINT.exists()

print("✓ All required files exist.")


# ----------------------------------------------------------------
# 2. Load production analysis pipeline
#
# We do NOT execute the full file.
# We extract the frozen IG implementation and constants.
# ----------------------------------------------------------------

analysis_source = ANALYSIS_PATH.read_text(
    encoding="utf-8"
)

analysis_tree = ast.parse(
    analysis_source
)

analysis_nodes = []

for node in analysis_tree.body:

    # Imports
    if isinstance(
        node,
        (ast.Import, ast.ImportFrom)
    ):
        analysis_nodes.append(node)

    # Literal constant assignments
    elif isinstance(
        node,
        ast.Assign
    ):

        try:
            ast.literal_eval(
                node.value
            )
            analysis_nodes.append(node)
        except Exception:
            pass

    elif isinstance(
        node,
        ast.AnnAssign
    ):

        if node.value is not None:

            try:
                ast.literal_eval(
                    node.value
                )
                analysis_nodes.append(node)
            except Exception:
                pass

    # Function definitions
    elif isinstance(
        node,
        (
            ast.FunctionDef,
            ast.AsyncFunctionDef
        )
    ):

        if node.name == "ig_enrichment":
            analysis_nodes.append(node)


analysis_module_ast = ast.Module(
    body=analysis_nodes,
    type_ignores=[]
)

ast.fix_missing_locations(
    analysis_module_ast
)

analysis_ns = {}

exec(
    compile(
        analysis_module_ast,
        str(ANALYSIS_PATH),
        "exec"
    ),
    analysis_ns
)

assert "ig_enrichment" in analysis_ns

ig_enrichment = analysis_ns[
    "ig_enrichment"
]

print(
    "✓ Frozen production ig_enrichment loaded."
)


# ----------------------------------------------------------------
# 3. Verify frozen IG constants
# ----------------------------------------------------------------

for name in [
    "IG_STEPS",
    "IG_RETRY_STEPS",
    "IG_SIGMA",
    "IG_TOL_REL",
    "IG_TAU_ABS",
    "XAI_EPS",
]:

    assert name in analysis_ns

    print(
        f"{name}:",
        analysis_ns[name]
    )


# ----------------------------------------------------------------
# 4. Load frozen ResNet-18 architecture
# ----------------------------------------------------------------

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print(
    "\nDevice:",
    device
)

assert device.type == "cuda", (
    "CUDA is not available. "
    "Do not proceed with the IG dry run."
)


model = torchvision.models.resnet18(
    weights=None
)

model.fc = torch.nn.Linear(
    model.fc.in_features,
    2
)

state_dict = torch.load(
    CHECKPOINT,
    map_location="cpu",
    weights_only=True
)

assert isinstance(
    state_dict,
    dict
)

missing, unexpected = model.load_state_dict(
    state_dict,
    strict=True
)

assert not missing
assert not unexpected

model = model.to(
    device
)

model.eval()

print(
    "✓ Production checkpoint loaded "
    "strictly into ResNet-18."
)


# ----------------------------------------------------------------
# 5. Load 500-image reconstructed cache
# ----------------------------------------------------------------

xai = pd.read_csv(
    XAI_SUBSET_PATH
).reset_index(drop=True)

clean = np.load(
    CLEAN_PATH,
    mmap_mode="r"
)

hair = np.load(
    HAIR_PATH,
    mmap_mode="r"
)

mask = np.load(
    MASK_PATH,
    mmap_mode="r"
)

area = np.load(
    AREA_PATH,
    mmap_mode="r"
)

assert len(xai) == 500

# Deterministic first image.
r = 0

image_id = str(
    xai.loc[r, "image"]
)

hair_u8 = np.asarray(
    hair[r]
)

clean_u8 = np.asarray(
    clean[r]
)

mask_bool = (
    np.asarray(mask[r]) > 0
)

assert hair_u8.shape == (
    224,
    224,
    3
)

assert clean_u8.shape == (
    224,
    224,
    3
)

assert mask_bool.shape == (
    224,
    224
)

print(
    "\nDry-run image:",
    image_id
)

print(
    "Locked mask area:",
    int(
        xai.loc[r, "mask_area"]
    )
)

print(
    "Reconstructed mask area:",
    int(
        mask_bool.sum()
    )
)

assert int(
    mask_bool.sum()
) == int(
    xai.loc[r, "mask_area"]
)

print(
    "✓ Dry-run image/mask identity verified."
)


# ----------------------------------------------------------------
# 6. Construct exact raw RGB black baseline
#
# IMPORTANT:
# This is [0,0,0] in raw [0,1] pixel space BEFORE ImageNet
# normalization, matching frozen make_black_baseline().
# ----------------------------------------------------------------

x = (
    torch.from_numpy(
        np.ascontiguousarray(
            hair_u8
        )
    )
    .to(device)
    .permute(
        2,
        0,
        1
    )
    .float()
    .unsqueeze(0)
    / 255.0
)

black = torch.zeros_like(
    x
)

print(
    "\nInput range:",
    float(x.min()),
    "to",
    float(x.max())
)

print(
    "Black baseline:",
    float(black.min()),
    "to",
    float(black.max())
)

assert torch.all(
    black == 0
)


# ----------------------------------------------------------------
# 7. Exact Gaussian baseline
#
# This reproduces production ig_enrichment:
# sigma = IG_SIGMA
# kernel = 2*ceil(3*sigma)+1
# ----------------------------------------------------------------

sigma = float(
    analysis_ns["IG_SIGMA"]
)

k = (
    2
    * int(
        math.ceil(
            3 * sigma
        )
    )
    + 1
)

gaussian = TF.gaussian_blur(
    x,
    kernel_size=[k, k],
    sigma=[sigma, sigma]
)

print(
    "\nGaussian sigma:",
    sigma
)

print(
    "Gaussian kernel:",
    k
)


# ----------------------------------------------------------------
# 8. Verify target logit definition independently
# ----------------------------------------------------------------

mean = torch.tensor(
    analysis_ns["_MEAN"],
    device=device
).view(
    1,
    3,
    1,
    1
)

std = torch.tensor(
    analysis_ns["_STD"],
    device=device
).view(
    1,
    3,
    1,
    1
)

xn = (
    x - mean
) / std

gn = (
    gaussian - mean
) / std

bn = (
    black - mean
) / std


with torch.no_grad():

    out_x = model(
        xn
    )

    out_g = model(
        gn
    )

    out_b = model(
        bn
    )

    delta_gaussian = float(
        (
            (out_x[:, 1] - out_x[:, 0])
            -
            (out_g[:, 1] - out_g[:, 0])
        ).item()
    )

    delta_black = float(
        (
            (out_x[:, 1] - out_x[:, 0])
            -
            (out_b[:, 1] - out_b[:, 0])
        ).item()
    )


print(
    "\nTarget logit check:"
)

print(
    "hair target z:",
    float(
        (out_x[:, 1] - out_x[:, 0]).item()
    )
)

print(
    "Gaussian target z:",
    float(
        (out_g[:, 1] - out_g[:, 0]).item()
    )
)

print(
    "Black target z:",
    float(
        (out_b[:, 1] - out_b[:, 0]).item()
    )
)

print(
    "Δz hair vs Gaussian:",
    delta_gaussian
)

print(
    "Δz hair vs Black:",
    delta_black
)


# ----------------------------------------------------------------
# 9. Production IG — Gaussian baseline
# ----------------------------------------------------------------

print(
    "\n" + "=" * 70
)

print(
    "RUNNING PRODUCTION IG — GAUSSIAN BASELINE"
)

print(
    "=" * 70
)

result_gaussian = ig_enrichment(
    model=model,
    hair_u8=hair_u8,
    mask_bool=mask_bool,
    device=device,
)


print(
    "\nGaussian result:"
)

for k_, v_ in result_gaussian.items():

    print(
        f"  {k_:30s}: {v_}"
    )


# ----------------------------------------------------------------
# 10. Black-baseline IG
#
# We reproduce the SAME production implementation but with the
# baseline changed from Gaussian to raw-black.
# ----------------------------------------------------------------

def ig_enrichment_black(
    model,
    hair_u8,
    mask_bool,
    device,
    steps,
    retry_steps,
    sigma,
    chunk=25,
):

    mean = torch.tensor(
        analysis_ns["_MEAN"],
        device=device
    ).view(
        1,
        3,
        1,
        1
    )

    std = torch.tensor(
        analysis_ns["_STD"],
        device=device
    ).view(
        1,
        3,
        1,
        1
    )

    model.eval()

    x = (
        torch.from_numpy(
            np.ascontiguousarray(
                hair_u8
            )
        )
        .to(device)
        .permute(
            2,
            0,
            1
        )
        .float()
        .unsqueeze(0)
        / 255.0
    )

    # EXACT BLACK BASELINE:
    # raw RGB [0,0,0] before normalization.
    base = torch.zeros_like(
        x
    )

    xn = (
        x - mean
    ) / std

    bn = (
        base - mean
    ) / std

    def zf(t):

        o = model(t)

        return (
            o[:, 1]
            -
            o[:, 0]
        )

    def run(n):

        acc = torch.zeros_like(
            xn
        )

        alphas = (
            torch.arange(
                1,
                n + 1,
                device=device,
                dtype=torch.float32
            )
            / n
        )

        for s in range(
            0,
            n,
            chunk
        ):

            a = alphas[
                s:s + chunk
            ].view(
                -1,
                1,
                1,
                1
            )

            pts = (
                bn
                +
                a * (
                    xn - bn
                )
            ).requires_grad_(
                True
            )

            g = torch.autograd.grad(
                zf(pts).sum(),
                pts
            )[0]

            acc += g.sum(
                0,
                keepdim=True
            )

        ig = (
            xn - bn
        ) * acc / n

        with torch.no_grad():

            delta = float(
                (
                    zf(xn)
                    -
                    zf(bn)
                ).item()
            )

        return (
            ig,
            delta,
            abs(
                float(
                    ig.sum().item()
                )
                -
                delta
            )
        )

    ig, delta, err = run(
        steps
    )

    used = steps

    tol = max(
        analysis_ns["IG_TOL_REL"]
        * abs(delta),
        analysis_ns["IG_TAU_ABS"]
    )

    if err > tol:

        ig, delta, err = run(
            retry_steps
        )

        used = retry_steps

    a = (
        ig.abs()
        .sum(1)[0]
    )

    total = float(
        a.sum().item()
    )

    area_frac = float(
        torch.from_numpy(
            np.ascontiguousarray(
                mask_bool
            )
        )
        .to(device)
        .float()
        .mean()
        .item()
    )

    undefined = (
        total
        <= analysis_ns["XAI_EPS"]
    ) or (
        area_frac == 0.0
    )

    enrichment = (
        float("nan")
        if undefined
        else float(
            (
                a[
                    torch.from_numpy(
                        np.ascontiguousarray(
                            mask_bool
                        )
                    )
                    .to(device)
                    .bool()
                ].sum().item()
                /
                total
            )
            /
            area_frac
        )
    )

    signed_in = float(
        ig.sum(1)[0][
            torch.from_numpy(
                np.ascontiguousarray(
                    mask_bool
                )
            )
            .to(device)
            .bool()
        ].sum().item()
    )

    return {
        "enrichment": enrichment,
        "undefined": bool(undefined),
        "completeness_error": err,
        "delta": delta,
        "tolerance": tol,
        "steps_used": used,
        "completeness_failed": bool(
            err > tol
        ),
        "signed_attribution_inside_mask":
            signed_in,
        "mask_area_fraction":
            area_frac,
    }


print(
    "\n" + "=" * 70
)

print(
    "RUNNING BLACK-BASELINE IG"
)

print(
    "=" * 70
)

result_black = ig_enrichment_black(
    model=model,
    hair_u8=hair_u8,
    mask_bool=mask_bool,
    device=device,
    steps=int(
        analysis_ns["IG_STEPS"]
    ),
    retry_steps=int(
        analysis_ns["IG_RETRY_STEPS"]
    ),
    sigma=float(
        analysis_ns["IG_SIGMA"]
    ),
)


print(
    "\nBlack result:"
)

for k_, v_ in result_black.items():

    print(
        f"  {k_:30s}: {v_}"
    )


# ----------------------------------------------------------------
# 11. Dry-run comparison
# ----------------------------------------------------------------

print(
    "\n" + "=" * 70
)

print(
    "DRY-RUN COMPARISON"
)

print(
    "=" * 70
)

g_enr = result_gaussian[
    "enrichment"
]

b_enr = result_black[
    "enrichment"
]

print(
    "Gaussian enrichment:",
    g_enr
)

print(
    "Black enrichment   :",
    b_enr
)

print(
    "Absolute difference:",
    abs(
        b_enr - g_enr
    )
)

if np.isfinite(g_enr) and g_enr != 0:

    print(
        "Relative difference (%):",
        100.0
        * abs(
            b_enr - g_enr
        )
        / abs(g_enr)
    )


# ----------------------------------------------------------------
# 12. Sanity assertions
# ----------------------------------------------------------------

assert np.isfinite(
    g_enr
)

assert np.isfinite(
    b_enr
)

assert not result_gaussian[
    "undefined"
]

assert not result_black[
    "undefined"
]

assert not result_gaussian[
    "completeness_failed"
]

assert not result_black[
    "completeness_failed"
]

assert result_gaussian[
    "steps_used"
] in [
    int(analysis_ns["IG_STEPS"]),
    int(analysis_ns["IG_RETRY_STEPS"]),
]

assert result_black[
    "steps_used"
] in [
    int(analysis_ns["IG_STEPS"]),
    int(analysis_ns["IG_RETRY_STEPS"]),
]


# ----------------------------------------------------------------
# FINAL STATUS
# ----------------------------------------------------------------

print(
    "\n" + "=" * 70
)

print(
    "CELL 14 STATUS"
)

print(
    "=" * 70
)

print(
    "Checkpoint strict load              : PASS"
)

print(
    "Production IG implementation        : LOADED"
)

print(
    "Gaussian baseline IG                : PASS"
)

print(
    "Black baseline IG                   : PASS"
)

print(
    "Gaussian completeness               : PASS"
)

print(
    "Black completeness                  : PASS"
)

print(
    "Gaussian enrichment finite          : PASS"
)

print(
    "Black enrichment finite             : PASS"
)

print(
    "GPU used                            : YES"
)

print(
    "Images evaluated                    : 1"
)

print(
    "Models evaluated                    : 1"
)

print(
    "\nIMPORTANT:"
)

print(
    "This is a DRY RUN only."
)

print(
    "Do NOT launch the full 30-model × "
    "15-image evaluation until this output "
    "has been reviewed."
)

print(
    "\nREADY FOR CELL 14 OUTPUT REVIEW."
)

CELL 14 — SINGLE-MODEL / SINGLE-IMAGE IG DRY RUN

Project root: /kaggle/input/datasets/tejasseshadriiiii/isic-final-project
✓ All required files exist.
✓ Frozen production ig_enrichment loaded.
IG_STEPS: 50
IG_RETRY_STEPS: 100
IG_SIGMA: 10.0
IG_TOL_REL: 0.05
IG_TAU_ABS: 0.05
XAI_EPS: 1e-08

Device: cuda
✓ Production checkpoint loaded strictly into ResNet-18.

Dry-run image: ISIC_0032121
Locked mask area: 815
Reconstructed mask area: 815
✓ Dry-run image/mask identity verified.

Input range: 0.019607843831181526 to 0.9215686917304993
Black baseline: 0.0 to 0.0


/tmp/ipykernel_58/1293092163.py:398: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  torch.from_numpy(



Gaussian sigma: 10.0
Gaussian kernel: 61

Target logit check:
hair target z: -13.112485885620117
Gaussian target z: 5.580613136291504
Black target z: 2.4608917236328125
Δz hair vs Gaussian: -18.693099975585938
Δz hair vs Black: -15.57337760925293

RUNNING PRODUCTION IG — GAUSSIAN BASELINE

Gaussian result:
  enrichment                    : 6.727071441398107
  undefined                     : False
  completeness_error            : 0.17713356018066406
  delta                         : -18.693099975585938
  tolerance                     : 0.9346549987792969
  steps_used                    : 50
  completeness_failed           : False
  signed_attribution_inside_mask: -12.16073226928711
  mask_area_fraction            : 0.016242824494838715

RUNNING BLACK-BASELINE IG

Black result:
  enrichment                    : 0.06610739089122357
  undefined                     : False
  completeness_error            : 0.06316566467285156
  delta                         : -15.57337760925293
  toleranc

In [17]:
# ================================================================
# CELL 15 — T1.4c FULL EVALUATION MANIFEST / SAMPLING LOCK
# ================================================================

from pathlib import Path
import hashlib
import json
import random

import numpy as np
import pandas as pd


print("=" * 70)
print("CELL 15 — T1.4c FULL EVALUATION MANIFEST / SAMPLING LOCK")
print("=" * 70)


# ----------------------------------------------------------------
# 1. Frozen protocol constants
# ----------------------------------------------------------------

T14C_PROTOCOL = (
    "T1.4c_black_baseline_sensitivity_v1"
)

MASTER_SEED = 20260925
SUBSAMPLE_SEED = 20260924

N_MODELS = 30
N_IMAGES_PER_MODEL = 15

RHO_LEVELS = [
    0.0,
    0.2,
    0.4,
    0.6,
    0.8,
    1.0,
]

N_REPLICATES = 5

assert (
    N_MODELS
    ==
    len(RHO_LEVELS)
    * N_REPLICATES
)

print("\nProtocol:", T14C_PROTOCOL)
print("Master seed:", MASTER_SEED)
print("Subsample seed:", SUBSAMPLE_SEED)
print("Models:", N_MODELS)
print("Images/model:", N_IMAGES_PER_MODEL)
print("rho levels:", RHO_LEVELS)
print("Replicates/rho:", N_REPLICATES)


# ----------------------------------------------------------------
# 2. Paths
# ----------------------------------------------------------------

PROJECT_CANDIDATES = list(
    Path("/kaggle/input/datasets").glob(
        "*/isic-final-project"
    )
)

assert len(PROJECT_CANDIDATES) >= 1

preferred = [
    p
    for p in PROJECT_CANDIDATES
    if "tejassh" in str(p).lower()
]

PROJECT_ROOT = (
    preferred[0]
    if preferred
    else PROJECT_CANDIDATES[0]
)

XAI_SUBSET_PATH = (
    PROJECT_ROOT
    / "step2U_xai_subset_lock"
    / "LOCKED_XAI_SUBSET.csv"
)

CACHE_DIR = Path(
    "/kaggle/working/t1_4c_xai_cache_500"
)

CLEAN_PATH = CACHE_DIR / "clean_500.npy"
HAIR_PATH = CACHE_DIR / "hair_500.npy"
MASK_PATH = CACHE_DIR / "mask_500.npy"
AREA_PATH = CACHE_DIR / "mask_area_500.npy"
INDEX_PATH = CACHE_DIR / "xai_cache_index_500.csv"

ROOT_A = Path(
    "/kaggle/input/notebooks/"
    "tejasseshadriiiii/"
    "ml-open-lab-30m-trained/"
    "training_runs"
)

ROOT_B = Path(
    "/kaggle/input/notebooks/"
    "tejasseshadriiiii/"
    "isic-kaggle-analysis/"
    "trained_models_readonly/"
    "training_runs"
)

assert XAI_SUBSET_PATH.exists()
assert CLEAN_PATH.exists()
assert HAIR_PATH.exists()
assert MASK_PATH.exists()
assert AREA_PATH.exists()
assert INDEX_PATH.exists()
assert ROOT_A.exists()
assert ROOT_B.exists()

print("\n✓ Required T1.4c inputs exist.")


# ----------------------------------------------------------------
# 3. Load locked 500-image XAI subset
# ----------------------------------------------------------------

xai = pd.read_csv(
    XAI_SUBSET_PATH
).reset_index(drop=True)

cache_index = pd.read_csv(
    INDEX_PATH
).reset_index(drop=True)

assert len(xai) == 500
assert len(cache_index) == 500

assert (
    xai["image"].astype(str).tolist()
    ==
    cache_index["image"].astype(str).tolist()
)

print(
    "✓ Locked XAI subset and working-cache "
    "index are aligned: 500/500."
)


# ----------------------------------------------------------------
# 4. Load cache metadata only
# ----------------------------------------------------------------

clean = np.load(
    CLEAN_PATH,
    mmap_mode="r"
)

hair = np.load(
    HAIR_PATH,
    mmap_mode="r"
)

mask = np.load(
    MASK_PATH,
    mmap_mode="r"
)

area = np.load(
    AREA_PATH,
    mmap_mode="r"
)

assert clean.shape == (
    500,
    224,
    224,
    3
)

assert hair.shape == (
    500,
    224,
    224,
    3
)

assert mask.shape == (
    500,
    224,
    224
)

assert area.shape == (
    500,
)

print(
    "✓ Working cache shapes verified."
)


# ----------------------------------------------------------------
# 5. Deterministic 15-image subsample
#
# IMPORTANT:
# One common image set is used across ALL 30 models.
# This creates paired model comparisons and avoids allowing
# different image composition to explain baseline differences.
# ----------------------------------------------------------------

all_rows = np.arange(
    len(xai),
    dtype=np.int64
)

rng = random.Random(
    SUBSAMPLE_SEED
)

selected_rows = sorted(
    rng.sample(
        list(all_rows),
        N_IMAGES_PER_MODEL
    )
)

assert len(selected_rows) == 15
assert len(set(selected_rows)) == 15

selected_images = [
    str(
        xai.loc[r, "image"]
    )
    for r in selected_rows
]

assert len(
    set(selected_images)
) == 15

print(
    "\nSelected image rows:",
    selected_rows
)

print(
    "\nSelected image IDs:"
)

for i, image_id in enumerate(
    selected_images,
    1
):

    print(
        f"{i:02d}. {image_id}"
    )


# ----------------------------------------------------------------
# 6. Verify deterministic selection independently
# ----------------------------------------------------------------

rng_check = random.Random(
    SUBSAMPLE_SEED
)

selected_rows_check = sorted(
    rng_check.sample(
        list(all_rows),
        N_IMAGES_PER_MODEL
    )
)

assert (
    selected_rows_check
    ==
    selected_rows
)

print(
    "\n✓ Deterministic image subsample verified."
)


# ----------------------------------------------------------------
# 7. Build exact 30-model production manifest
# ----------------------------------------------------------------

model_records = []

for rho in RHO_LEVELS:

    for rep in range(
        1,
        N_REPLICATES + 1
    ):

        model_name = (
            f"rho_{rho:.1f}_rep_{rep}"
        )

        checkpoint = (
            ROOT_A
            / model_name
            / "final_epoch15.pt"
        )

        checkpoint_B = (
            ROOT_B
            / model_name
            / "final_epoch15.pt"
        )

        assert checkpoint.exists(), (
            f"Missing checkpoint: {checkpoint}"
        )

        assert checkpoint_B.exists(), (
            f"Missing Root B checkpoint: "
            f"{checkpoint_B}"
        )

        model_records.append(
            {
                "model": model_name,
                "rho": rho,
                "replicate": rep,
                "checkpoint": str(
                    checkpoint
                ),
                "checkpoint_B": str(
                    checkpoint_B
                ),
            }
        )


models_df = pd.DataFrame(
    model_records
)

assert len(models_df) == 30

assert models_df[
    "model"
].nunique() == 30

assert models_df[
    "rho"
].nunique() == 6

assert (
    models_df
    .groupby("rho")
    .size()
    .eq(5)
    .all()
)

print(
    "\n✓ 30-model production manifest verified."
)


# ----------------------------------------------------------------
# 8. Build model × image evaluation manifest
# ----------------------------------------------------------------

eval_records = []

for _, model_row in models_df.iterrows():

    for local_image_index, cache_row in enumerate(
        selected_rows
    ):

        image_id = str(
            xai.loc[
                cache_row,
                "image"
            ]
        )

        lesion_id = str(
            xai.loc[
                cache_row,
                "lesion_id"
            ]
        )

        mask_area_locked = int(
            xai.loc[
                cache_row,
                "mask_area"
            ]
        )

        eval_records.append(
            {
                "model": model_row["model"],
                "rho": float(
                    model_row["rho"]
                ),
                "replicate": int(
                    model_row["replicate"]
                ),
                "checkpoint": model_row[
                    "checkpoint"
                ],
                "image_local_index":
                    local_image_index,
                "cache_row":
                    int(cache_row),
                "image":
                    image_id,
                "lesion_id":
                    lesion_id,
                "mask_area":
                    mask_area_locked,
            }
        )


eval_df = pd.DataFrame(
    eval_records
)

assert len(eval_df) == (
    N_MODELS
    * N_IMAGES_PER_MODEL
)

assert len(eval_df) == 450

assert eval_df[
    ["model", "cache_row"]
].drop_duplicates().shape[0] == 450

assert eval_df[
    ["model", "image"]
].drop_duplicates().shape[0] == 450

print(
    "✓ Model × image manifest:",
    len(eval_df),
    "pairs."
)


# ----------------------------------------------------------------
# 9. Every model receives exactly the SAME 15 images
# ----------------------------------------------------------------

image_sets = {}

for model_name, g in eval_df.groupby(
    "model"
):

    image_sets[
        model_name
    ] = tuple(
        g.sort_values(
            "image_local_index"
        )["image"].tolist()
    )

reference_images = next(
    iter(
        image_sets.values()
    )
)

for model_name, imgs in image_sets.items():

    assert imgs == reference_images, (
        f"Image-set mismatch for {model_name}"
    )

print(
    "✓ All 30 models use the same 15 images."
)


# ----------------------------------------------------------------
# 10. Each rho has exactly 5 replicates
# ----------------------------------------------------------------

rho_model_counts = (
    models_df
    .groupby("rho")["model"]
    .nunique()
)

print(
    "\nModels per rho:"
)

print(
    rho_model_counts.to_string()
)

assert rho_model_counts.eq(5).all()

print(
    "✓ Exactly 5 replicates at every rho."
)


# ----------------------------------------------------------------
# 11. Planned IG evaluations
# ----------------------------------------------------------------

n_model_image_pairs = len(
    eval_df
)

n_baselines = 2

planned_ig = (
    n_model_image_pairs
    * n_baselines
)

assert n_model_image_pairs == 450
assert planned_ig == 900

print(
    "\nPlanned model-image pairs:",
    n_model_image_pairs
)

print(
    "Baselines:",
    n_baselines
)

print(
    "Planned IG evaluations:",
    planned_ig
)


# ----------------------------------------------------------------
# 12. Output schema
# ----------------------------------------------------------------

required_output_columns = [
    "model",
    "rho",
    "replicate",
    "checkpoint",
    "cache_row",
    "image",
    "lesion_id",
    "mask_area",
    "gaussian_enrichment",
    "black_enrichment",
    "gaussian_completeness_error",
    "black_completeness_error",
    "gaussian_delta",
    "black_delta",
    "gaussian_steps_used",
    "black_steps_used",
    "gaussian_completeness_failed",
    "black_completeness_failed",
    "gaussian_undefined",
    "black_undefined",
    "gaussian_signed_inside",
    "black_signed_inside",
    "mask_area_fraction",
]


empty_output = pd.DataFrame(
    columns=required_output_columns
)

assert list(
    empty_output.columns
) == required_output_columns

print(
    "\n✓ Output schema locked:"
)

print(
    "  columns:",
    len(required_output_columns)
)


# ----------------------------------------------------------------
# 13. Explicit writable-transfer rule
#
# This prevents the Cell 14 memmap warning.
# ----------------------------------------------------------------

def writable_contiguous_uint8(
    arr
):
    return np.array(
        arr,
        dtype=np.uint8,
        order="C",
        copy=True
    )


test_hair = writable_contiguous_uint8(
    hair[selected_rows[0]]
)

test_clean = writable_contiguous_uint8(
    clean[selected_rows[0]]
)

assert test_hair.flags.writeable
assert test_clean.flags.writeable
assert test_hair.flags.c_contiguous
assert test_clean.flags.c_contiguous

print(
    "\n✓ Future image→Torch transfers will use "
    "writable contiguous copies."
)


# ----------------------------------------------------------------
# 14. Manifest hash
# ----------------------------------------------------------------

manifest_csv_text = (
    eval_df
    .sort_values(
        [
            "model",
            "image_local_index"
        ]
    )
    .to_csv(
        index=False
    )
)

manifest_hash = hashlib.sha256(
    manifest_csv_text.encode(
        "utf-8"
    )
).hexdigest()

print(
    "\nEvaluation manifest SHA256:"
)

print(
    manifest_hash
)


# ----------------------------------------------------------------
# 15. Save locked manifest
# ----------------------------------------------------------------

OUT_DIR = Path(
    "/kaggle/working/"
    "t1_4c_full_evaluation_lock"
)

OUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

MANIFEST_CSV = (
    OUT_DIR
    / "T1_4c_full_evaluation_manifest.csv"
)

MANIFEST_JSON = (
    OUT_DIR
    / "T1_4c_full_evaluation_manifest.json"
)

SELECTED_ROWS = (
    OUT_DIR
    / "T1_4c_selected_rows.csv"
)

eval_df.to_csv(
    MANIFEST_CSV,
    index=False
)

pd.DataFrame(
    {
        "local_image_index":
            range(
                N_IMAGES_PER_MODEL
            ),
        "cache_row":
            selected_rows,
        "image":
            selected_images,
    }
).to_csv(
    SELECTED_ROWS,
    index=False
)

with open(
    MANIFEST_JSON,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        {
            "protocol":
                T14C_PROTOCOL,
            "master_seed":
                MASTER_SEED,
            "subsample_seed":
                SUBSAMPLE_SEED,
            "n_models":
                N_MODELS,
            "n_images_total":
                500,
            "n_images_per_model":
                N_IMAGES_PER_MODEL,
            "n_model_image_pairs":
                450,
            "n_baselines":
                2,
            "planned_ig_evaluations":
                900,
            "rho_levels":
                RHO_LEVELS,
            "replicates_per_rho":
                N_REPLICATES,
            "selected_cache_rows":
                selected_rows,
            "selected_images":
                selected_images,
            "manifest_sha256":
                manifest_hash,
            "checkpoint":
                "final_epoch15.pt",
            "baseline_1":
                "gaussian_blurred_same_hair_image",
            "baseline_2":
                "raw_rgb_black_before_normalization",
            "future_torch_transfer":
                "writable_contiguous_uint8_copy",
        },
        f,
        indent=2
    )


# ----------------------------------------------------------------
# 16. Final status
# ----------------------------------------------------------------

print("\n" + "=" * 70)
print("CELL 15 STATUS")
print("=" * 70)

print(
    "30 production models                  : VERIFIED"
)

print(
    "5 replicates per rho                  : VERIFIED"
)

print(
    "15 common images/model                : VERIFIED"
)

print(
    "450 model-image pairs                 : VERIFIED"
)

print(
    "900 planned IG evaluations            : VERIFIED"
)

print(
    "Duplicate model-image pairs           : NONE"
)

print(
    "Common paired image set               : VERIFIED"
)

print(
    "Output schema                          : LOCKED"
)

print(
    "Writable image transfer rule          : LOCKED"
)

print(
    "\nNo IG computed : YES"
)

print(
    "No GPU computation : YES"
)

print(
    "\nManifest:"
)

print(
    MANIFEST_CSV
)

print(
    MANIFEST_JSON
)

print(
    SELECTED_ROWS
)

print(
    "\nEvaluation manifest SHA256:"
)

print(
    manifest_hash
)

print(
    "\nIMPORTANT:"
)

print(
    "This cell only locks the full evaluation plan."
)

print(
    "Do NOT launch the 900 IG evaluations yet."
)

print(
    "\nREADY FOR CELL 15 OUTPUT REVIEW."
)

CELL 15 — T1.4c FULL EVALUATION MANIFEST / SAMPLING LOCK

Protocol: T1.4c_black_baseline_sensitivity_v1
Master seed: 20260925
Subsample seed: 20260924
Models: 30
Images/model: 15
rho levels: [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
Replicates/rho: 5

✓ Required T1.4c inputs exist.
✓ Locked XAI subset and working-cache index are aligned: 500/500.
✓ Working cache shapes verified.

Selected image rows: [np.int64(7), np.int64(29), np.int64(74), np.int64(279), np.int64(286), np.int64(291), np.int64(323), np.int64(338), np.int64(359), np.int64(372), np.int64(402), np.int64(417), np.int64(419), np.int64(467), np.int64(480)]

Selected image IDs:
01. ISIC_0014089_downsampled
02. ISIC_0071629
03. ISIC_0030763
04. ISIC_0064735
05. ISIC_0055108
06. ISIC_0070236
07. ISIC_0069001
08. ISIC_0032318
09. ISIC_0027517
10. ISIC_0029781
11. ISIC_0032396
12. ISIC_0071722
13. ISIC_0058556
14. ISIC_0068243
15. ISIC_0027057

✓ Deterministic image subsample verified.

✓ 30-model production manifest verified.
✓ Model × im

TypeError: Object of type int64 is not JSON serializable

In [18]:
# ================================================================
# CELL 15 FIX — JSON SERIALIZATION
# ================================================================

import json
from pathlib import Path

# Convert NumPy scalar types to native Python types
selected_rows_json = [
    int(x) for x in selected_rows
]

selected_images_json = [
    str(x) for x in selected_images
]

with open(
    MANIFEST_JSON,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        {
            "protocol": str(T14C_PROTOCOL),
            "master_seed": int(MASTER_SEED),
            "subsample_seed": int(SUBSAMPLE_SEED),

            "n_models": int(N_MODELS),
            "n_images_total": 500,
            "n_images_per_model":
                int(N_IMAGES_PER_MODEL),

            "n_model_image_pairs": 450,
            "n_baselines": 2,
            "planned_ig_evaluations": 900,

            "rho_levels": [
                float(x)
                for x in RHO_LEVELS
            ],

            "replicates_per_rho":
                int(N_REPLICATES),

            "selected_cache_rows":
                selected_rows_json,

            "selected_images":
                selected_images_json,

            "manifest_sha256":
                str(manifest_hash),

            "checkpoint":
                "final_epoch15.pt",

            "baseline_1":
                "gaussian_blurred_same_hair_image",

            "baseline_2":
                "raw_rgb_black_before_normalization",

            "future_torch_transfer":
                "writable_contiguous_uint8_copy",
        },
        f,
        indent=2
    )

print("✓ JSON manifest saved successfully:")
print(MANIFEST_JSON)

# Re-open it to prove it is valid JSON
with open(
    MANIFEST_JSON,
    "r",
    encoding="utf-8"
) as f:
    manifest_check = json.load(f)

print("✓ JSON re-open validation passed.")

print("\nManifest SHA256:")
print(manifest_check["manifest_sha256"])

print("\nSelected rows:")
print(manifest_check["selected_cache_rows"])

print("\nSelected images:")
for i, image_id in enumerate(
    manifest_check["selected_images"],
    1
):
    print(f"{i:02d}. {image_id}")

print("\n" + "=" * 70)
print("CELL 15 STATUS: PASS")
print("=" * 70)
print("30 models                  : VERIFIED")
print("15 common images/model    : VERIFIED")
print("450 model-image pairs     : VERIFIED")
print("900 planned IG evaluations: VERIFIED")
print("JSON manifest             : SAVED + VALIDATED")
print("IG computation            : NOT RUN")

✓ JSON manifest saved successfully:
/kaggle/working/t1_4c_full_evaluation_lock/T1_4c_full_evaluation_manifest.json
✓ JSON re-open validation passed.

Manifest SHA256:
32b02fe45b83f2df0e935185f804dbdc90de1f4088bd6fe406d9f479c341bfe6

Selected rows:
[7, 29, 74, 279, 286, 291, 323, 338, 359, 372, 402, 417, 419, 467, 480]

Selected images:
01. ISIC_0014089_downsampled
02. ISIC_0071629
03. ISIC_0030763
04. ISIC_0064735
05. ISIC_0055108
06. ISIC_0070236
07. ISIC_0069001
08. ISIC_0032318
09. ISIC_0027517
10. ISIC_0029781
11. ISIC_0032396
12. ISIC_0071722
13. ISIC_0058556
14. ISIC_0068243
15. ISIC_0027057

CELL 15 STATUS: PASS
30 models                  : VERIFIED
15 common images/model    : VERIFIED
450 model-image pairs     : VERIFIED
900 planned IG evaluations: VERIFIED
JSON manifest             : SAVED + VALIDATED
IG computation            : NOT RUN


In [20]:
# ================================================================
# CELL 16A — T1.4c ONE-MODEL PRODUCTION VALIDATION
# PATH-RESOLUTION FIX
# ================================================================

from pathlib import Path
import sys
import json
import hashlib
import importlib.util

import numpy as np
import pandas as pd
import torch


print("=" * 70)
print("CELL 16A — T1.4c ONE-MODEL PRODUCTION VALIDATION")
print("PATH-RESOLUTION FIX")
print("=" * 70)


# ----------------------------------------------------------------
# 1. Configuration
# ----------------------------------------------------------------

MODEL_NAME = "rho_0.0_rep_1"

MANIFEST_PATH = Path(
    "/kaggle/working/"
    "t1_4c_full_evaluation_lock/"
    "T1_4c_full_evaluation_manifest.csv"
)

CACHE_DIR = Path(
    "/kaggle/working/t1_4c_xai_cache_500"
)

HAIR_PATH = CACHE_DIR / "hair_500.npy"
MASK_PATH = CACHE_DIR / "mask_500.npy"
AREA_PATH = CACHE_DIR / "mask_area_500.npy"

ROOT_A = Path(
    "/kaggle/input/notebooks/"
    "tejasseshadriiiii/"
    "ml-open-lab-30m-trained/"
    "training_runs"
)


# ----------------------------------------------------------------
# 2. Discover project root
# ----------------------------------------------------------------

project_candidates = list(
    Path("/kaggle/input/datasets").glob(
        "*/isic-final-project"
    )
)

assert project_candidates, (
    "Could not find isic-final-project under "
    "/kaggle/input/datasets"
)

preferred = [
    p for p in project_candidates
    if "tejassh" in str(p).lower()
]

PROJECT_ROOT = (
    preferred[0]
    if preferred
    else project_candidates[0]
)

print("\nProject root:")
print(PROJECT_ROOT)


# ----------------------------------------------------------------
# 3. Discover exact frozen source files
#
# We do NOT assume nested directory structure.
# ----------------------------------------------------------------

analysis_matches = list(
    PROJECT_ROOT.rglob(
        "analysis_pipeline.py"
    )
)

generator_matches = list(
    PROJECT_ROOT.rglob(
        "recovered_original_hair_generator.py"
    )
)

training_matches = list(
    PROJECT_ROOT.rglob(
        "train_one_model.py"
    )
)


print("\nanalysis_pipeline.py candidates:")
for p in analysis_matches:
    print("  ", p)

print("\nrecovered_original_hair_generator.py candidates:")
for p in generator_matches:
    print("  ", p)

print("\ntrain_one_model.py candidates:")
for p in training_matches:
    print("  ", p)


# ----------------------------------------------------------------
# 4. Select exact expected frozen locations
# ----------------------------------------------------------------

def choose_unique(
    matches,
    filename,
    preferred_fragments=()
):

    if len(matches) == 1:
        return matches[0]

    preferred_matches = [
        p
        for p in matches
        if all(
            fragment in str(p)
            for fragment in preferred_fragments
        )
    ]

    if len(preferred_matches) == 1:
        return preferred_matches[0]

    if len(matches) == 0:
        raise AssertionError(
            f"Could not find {filename}"
        )

    raise AssertionError(
        f"Ambiguous {filename}: "
        f"{len(matches)} candidates found.\n"
        + "\n".join(
            str(p)
            for p in matches
        )
    )


ANALYSIS_PATH = choose_unique(
    analysis_matches,
    "analysis_pipeline.py",
    ("step2W_analysis_code_freeze",)
)

GENERATOR_PATH = choose_unique(
    generator_matches,
    "recovered_original_hair_generator.py",
    ("step1J_generator_recovery_lock",)
)

TRAINING_PATH = choose_unique(
    training_matches,
    "train_one_model.py",
    ("step2X_training_code_freeze",)
)


print("\n✓ Exact frozen source files resolved:")

print(
    "Analysis :",
    ANALYSIS_PATH
)

print(
    "Generator:",
    GENERATOR_PATH
)

print(
    "Training  :",
    TRAINING_PATH
)


# ----------------------------------------------------------------
# 5. Verify remaining required files
# ----------------------------------------------------------------

assert MANIFEST_PATH.exists(), MANIFEST_PATH
assert HAIR_PATH.exists(), HAIR_PATH
assert MASK_PATH.exists(), MASK_PATH
assert AREA_PATH.exists(), AREA_PATH

CHECKPOINT_PATH = (
    ROOT_A
    / MODEL_NAME
    / "final_epoch15.pt"
)

assert CHECKPOINT_PATH.exists(), (
    f"Checkpoint not found:\n{CHECKPOINT_PATH}"
)

print("\n✓ Manifest/cache/checkpoint files exist.")


# ----------------------------------------------------------------
# 6. SHA256 helper
# ----------------------------------------------------------------

def sha256_file(path):

    h = hashlib.sha256()

    with open(
        path,
        "rb"
    ) as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):

            h.update(chunk)

    return h.hexdigest()


analysis_hash = sha256_file(
    ANALYSIS_PATH
)

generator_hash = sha256_file(
    GENERATOR_PATH
)

training_hash = sha256_file(
    TRAINING_PATH
)

checkpoint_hash = sha256_file(
    CHECKPOINT_PATH
)


print("\nFrozen-source hashes:")

print(
    "analysis  :",
    analysis_hash
)

print(
    "generator :",
    generator_hash
)

print(
    "training  :",
    training_hash
)

print(
    "checkpoint:",
    checkpoint_hash
)


# ----------------------------------------------------------------
# 7. Load frozen analysis pipeline
# ----------------------------------------------------------------

spec = importlib.util.spec_from_file_location(
    "analysis_pipeline_t14c",
    str(ANALYSIS_PATH)
)

ap = importlib.util.module_from_spec(
    spec
)

spec.loader.exec_module(ap)

assert hasattr(
    ap,
    "ig_enrichment"
)

ig_enrichment = ap.ig_enrichment

print(
    "\n✓ Production ig_enrichment loaded."
)


# ----------------------------------------------------------------
# 8. Verify exact frozen IG constants
# ----------------------------------------------------------------

IG_STEPS = int(
    ap.IG_STEPS
)

IG_RETRY_STEPS = int(
    ap.IG_RETRY_STEPS
)

IG_SIGMA = float(
    ap.IG_SIGMA
)

IG_TOL_REL = float(
    ap.IG_TOL_REL
)

IG_TAU_ABS = float(
    ap.IG_TAU_ABS
)

XAI_EPS = float(
    ap.XAI_EPS
)

assert IG_STEPS == 50
assert IG_RETRY_STEPS == 100
assert IG_SIGMA == 10.0

print("\nFrozen IG configuration:")
print("IG steps       :", IG_STEPS)
print("Retry steps    :", IG_RETRY_STEPS)
print("Sigma          :", IG_SIGMA)
print("Relative tol   :", IG_TOL_REL)
print("Absolute tol   :", IG_TAU_ABS)
print("XAI epsilon    :", XAI_EPS)


# ----------------------------------------------------------------
# 9. Load locked 15-image manifest
# ----------------------------------------------------------------

manifest = pd.read_csv(
    MANIFEST_PATH
)

model_manifest = manifest[
    manifest["model"] == MODEL_NAME
].copy()

assert len(
    model_manifest
) == 15

model_manifest = (
    model_manifest
    .sort_values(
        "image_local_index"
    )
    .reset_index(drop=True)
)

assert (
    model_manifest["image"]
    .nunique()
    ==
    15
)

print(
    "\n✓ Locked 15-image manifest loaded."
)


# ----------------------------------------------------------------
# 10. Load reconstructed working cache
# ----------------------------------------------------------------

hair_cache = np.load(
    HAIR_PATH,
    mmap_mode="r"
)

mask_cache = np.load(
    MASK_PATH,
    mmap_mode="r"
)

area_cache = np.load(
    AREA_PATH,
    mmap_mode="r"
)

assert hair_cache.shape == (
    500,
    224,
    224,
    3
)

assert mask_cache.shape == (
    500,
    224,
    224
)

assert area_cache.shape == (
    500,
)

print(
    "✓ Working cache loaded."
)


# ----------------------------------------------------------------
# 11. Load exact production checkpoint
# ----------------------------------------------------------------

import torchvision
import torch.nn as nn

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

assert device.type == "cuda", (
    "CUDA is required for production validation."
)

model = torchvision.models.resnet18(
    weights=None
)

model.fc = nn.Linear(
    model.fc.in_features,
    2
)

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu"
)

assert isinstance(
    checkpoint,
    dict
)

missing, unexpected = (
    model.load_state_dict(
        checkpoint,
        strict=True
    )
)

assert not missing
assert not unexpected

model = model.to(device)
model.eval()

print(
    "\n✓ Production ResNet-18 checkpoint loaded."
)

print(
    "Device:",
    device
)


# ----------------------------------------------------------------
# 12. Black baseline implementation
# ----------------------------------------------------------------

def black_baseline_ig(
    model,
    hair_u8,
    mask_bool,
    device,
    steps=IG_STEPS,
    retry_steps=IG_RETRY_STEPS,
    sigma=IG_SIGMA,
    chunk=25,
):

    import torchvision.transforms.functional as TF

    mean = torch.tensor(
        ap._MEAN,
        device=device
    ).view(
        1, 3, 1, 1
    )

    std = torch.tensor(
        ap._STD,
        device=device
    ).view(
        1, 3, 1, 1
    )

    model.eval()

    hair_u8 = np.array(
        hair_u8,
        dtype=np.uint8,
        order="C",
        copy=True
    )

    x = (
        torch.from_numpy(
            hair_u8
        )
        .to(device)
        .permute(2, 0, 1)
        .float()
        .unsqueeze(0)
        / 255.0
    )

    # BLACK = raw RGB [0,0,0]
    # BEFORE ImageNet normalization.
    base = torch.zeros_like(x)

    xn = (
        x - mean
    ) / std

    bn = (
        base - mean
    ) / std

    mask_bool = np.array(
        mask_bool,
        dtype=bool,
        order="C",
        copy=True
    )

    m = torch.from_numpy(
        mask_bool
    ).to(device).bool()

    def zf(t):

        o = model(t)

        return o[:, 1] - o[:, 0]

    def run(n):

        acc = torch.zeros_like(
            xn
        )

        alphas = (
            torch.arange(
                1,
                n + 1,
                device=device,
                dtype=torch.float32
            )
            / n
        )

        for s in range(
            0,
            n,
            chunk
        ):

            a = (
                alphas[s:s + chunk]
                .view(
                    -1,
                    1,
                    1,
                    1
                )
            )

            pts = (
                bn
                +
                a * (xn - bn)
            ).requires_grad_(True)

            g = torch.autograd.grad(
                zf(pts).sum(),
                pts
            )[0]

            acc += g.sum(
                0,
                keepdim=True
            )

        ig = (
            (xn - bn)
            * acc
            / n
        )

        with torch.no_grad():

            delta = float(
                (
                    zf(xn)
                    -
                    zf(bn)
                ).item()
            )

        err = abs(
            float(
                ig.sum().item()
            )
            -
            delta
        )

        return (
            ig,
            delta,
            err
        )

    ig, delta, err = run(
        steps
    )

    used = steps

    tol = max(
        IG_TOL_REL * abs(delta),
        IG_TAU_ABS
    )

    if err > tol:

        ig, delta, err = run(
            retry_steps
        )

        used = retry_steps

    with torch.no_grad():

        a = (
            ig.abs()
            .sum(1)[0]
        )

        total = float(
            a.sum().item()
        )

        area_frac = float(
            m.float()
            .mean()
            .item()
        )

        undefined = (
            total <= XAI_EPS
            or
            area_frac == 0.0
        )

        enrichment = (
            float("nan")
            if undefined
            else
            float(
                (
                    a[m].sum().item()
                    /
                    total
                )
                /
                area_frac
            )
        )

        signed_in = float(
            ig.sum(1)[0][m]
            .sum()
            .item()
        )

    return {
        "enrichment":
            enrichment,

        "undefined":
            bool(undefined),

        "completeness_error":
            float(err),

        "delta":
            float(delta),

        "tolerance":
            float(tol),

        "steps_used":
            int(used),

        "completeness_failed":
            bool(err > tol),

        "signed_attribution_inside_mask":
            signed_in,

        "mask_area_fraction":
            area_frac,
    }


print(
    "\n✓ Black-baseline implementation loaded."
)


# ----------------------------------------------------------------
# 13. Run 15-image validation
# ----------------------------------------------------------------

results = []

for i, row in model_manifest.iterrows():

    cache_row = int(
        row["cache_row"]
    )

    image_id = str(
        row["image"]
    )

    lesion_id = str(
        row["lesion_id"]
    )

    locked_area = int(
        row["mask_area"]
    )

    hair_u8 = np.array(
        hair_cache[cache_row],
        dtype=np.uint8,
        order="C",
        copy=True
    )

    mask_u8 = np.array(
        mask_cache[cache_row],
        dtype=np.uint8,
        order="C",
        copy=True
    )

    mask_bool = (
        mask_u8 > 0
    )

    computed_area = int(
        mask_bool.sum()
    )

    assert (
        computed_area
        ==
        locked_area
    )

    # ------------------------------------------------------------
    # Gaussian = EXACT frozen production IG
    # ------------------------------------------------------------

    gaussian = ig_enrichment(
        model,
        hair_u8,
        mask_bool,
        device
    )

    # ------------------------------------------------------------
    # Black = identical IG except baseline
    # ------------------------------------------------------------

    black = black_baseline_ig(
        model,
        hair_u8,
        mask_bool,
        device
    )

    assert np.isfinite(
        gaussian["enrichment"]
    )

    assert np.isfinite(
        black["enrichment"]
    )

    results.append(
        {
            "model":
                MODEL_NAME,

            "rho":
                float(row["rho"]),

            "replicate":
                int(row["replicate"]),

            "cache_row":
                cache_row,

            "image":
                image_id,

            "lesion_id":
                lesion_id,

            "mask_area":
                computed_area,

            "gaussian_enrichment":
                gaussian["enrichment"],

            "black_enrichment":
                black["enrichment"],

            "gaussian_completeness_error":
                gaussian[
                    "completeness_error"
                ],

            "black_completeness_error":
                black[
                    "completeness_error"
                ],

            "gaussian_delta":
                gaussian["delta"],

            "black_delta":
                black["delta"],

            "gaussian_steps_used":
                gaussian["steps_used"],

            "black_steps_used":
                black["steps_used"],

            "gaussian_completeness_failed":
                gaussian[
                    "completeness_failed"
                ],

            "black_completeness_failed":
                black[
                    "completeness_failed"
                ],

            "gaussian_undefined":
                gaussian["undefined"],

            "black_undefined":
                black["undefined"],

            "gaussian_signed_inside":
                gaussian[
                    "signed_attribution_inside_mask"
                ],

            "black_signed_inside":
                black[
                    "signed_attribution_inside_mask"
                ],

            "mask_area_fraction":
                gaussian[
                    "mask_area_fraction"
                ],
        }
    )

    print(
        f"[{i+1:02d}/15] "
        f"{image_id} | "
        f"G={gaussian['enrichment']:.6f} | "
        f"B={black['enrichment']:.6f} | "
        f"Gerr={gaussian['completeness_error']:.5f} | "
        f"Berr={black['completeness_error']:.5f} | "
        f"Gsteps={gaussian['steps_used']} | "
        f"Bsteps={black['steps_used']}"
    )


# ----------------------------------------------------------------
# 14. QA summary
# ----------------------------------------------------------------

results_df = pd.DataFrame(
    results
)

assert len(
    results_df
) == 15

assert (
    results_df["image"].nunique()
    == 15
)

results_df[
    "abs_enrichment_difference"
] = (
    results_df[
        "gaussian_enrichment"
    ]
    -
    results_df[
        "black_enrichment"
    ]
).abs()

results_df[
    "relative_enrichment_difference_pct"
] = (
    results_df[
        "abs_enrichment_difference"
    ]
    /
    results_df[
        "gaussian_enrichment"
    ].abs()
    * 100.0
)

OUT_DIR = Path(
    "/kaggle/working/"
    "t1_4c_one_model_validation"
)

OUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

RESULT_PATH = (
    OUT_DIR
    /
    "T1_4c_one_model_15_image_validation.csv"
)

results_df.to_csv(
    RESULT_PATH,
    index=False
)

gaussian_failures = int(
    results_df[
        "gaussian_completeness_failed"
    ].sum()
)

black_failures = int(
    results_df[
        "black_completeness_failed"
    ].sum()
)

gaussian_undefined = int(
    results_df[
        "gaussian_undefined"
    ].sum()
)

black_undefined = int(
    results_df[
        "black_undefined"
    ].sum()
)

print("\n" + "=" * 70)
print("CELL 16A — VALIDATION RESULT")
print("=" * 70)

print(
    "Model:",
    MODEL_NAME
)

print(
    "Images:",
    len(results_df)
)

print(
    "IG evaluations:",
    len(results_df) * 2
)

print(
    "\nGaussian completeness failures:",
    gaussian_failures,
    "/ 15"
)

print(
    "Black completeness failures:",
    black_failures,
    "/ 15"
)

print(
    "Gaussian undefined:",
    gaussian_undefined,
    "/ 15"
)

print(
    "Black undefined:",
    black_undefined,
    "/ 15"
)

print(
    "\nMean Gaussian enrichment:",
    f"{results_df['gaussian_enrichment'].mean():.6f}"
)

print(
    "Mean Black enrichment:",
    f"{results_df['black_enrichment'].mean():.6f}"
)

print(
    "Mean absolute difference:",
    f"{results_df['abs_enrichment_difference'].mean():.6f}"
)

print(
    "Mean relative difference:",
    f"{results_df['relative_enrichment_difference_pct'].mean():.3f}%"
)

print(
    "\nSaved:",
    RESULT_PATH
)

print(
    "\nCELL 16A COMPLETE."
)

print(
    "DO NOT START THE FULL 900-EVALUATION RUN UNTIL "
    "THIS OUTPUT IS REVIEWED."
)

CELL 16A — T1.4c ONE-MODEL PRODUCTION VALIDATION
PATH-RESOLUTION FIX

Project root:
/kaggle/input/datasets/tejasseshadriiiii/isic-final-project

analysis_pipeline.py candidates:
   /kaggle/input/datasets/tejasseshadriiiii/isic-final-project/step2W_analysis_code_freeze/analysis_pipeline.py

recovered_original_hair_generator.py candidates:
   /kaggle/input/datasets/tejasseshadriiiii/isic-final-project/step1J_generator_recovery_lock/recovered_original_hair_generator.py

train_one_model.py candidates:
   /kaggle/input/datasets/tejasseshadriiiii/isic-final-project/step2X_training_code_freeze/train_one_model.py

✓ Exact frozen source files resolved:
Analysis : /kaggle/input/datasets/tejasseshadriiiii/isic-final-project/step2W_analysis_code_freeze/analysis_pipeline.py
Generator: /kaggle/input/datasets/tejasseshadriiiii/isic-final-project/step1J_generator_recovery_lock/recovered_original_hair_generator.py
Training  : /kaggle/input/datasets/tejasseshadriiiii/isic-final-project/step2X_training_c

In [21]:
# ================================================================
# CELL 16B — T1.4c COMPLETENESS / RETRY DIAGNOSTIC
# CPU ONLY — NO NEW IG COMPUTATION
# ================================================================

from pathlib import Path
import numpy as np
import pandas as pd

print("=" * 70)
print("CELL 16B — COMPLETENESS / RETRY DIAGNOSTIC")
print("=" * 70)

RESULT_PATH = Path(
    "/kaggle/working/"
    "t1_4c_one_model_validation/"
    "T1_4c_one_model_15_image_validation.csv"
)

df = pd.read_csv(
    RESULT_PATH
)

assert len(df) == 15

# ---------------------------------------------------------------
# Reconstruct exact tolerances
# ---------------------------------------------------------------

df["gaussian_tolerance"] = np.maximum(
    0.05 * df["gaussian_delta"].abs(),
    0.05
)

df["black_tolerance"] = np.maximum(
    0.05 * df["black_delta"].abs(),
    0.05
)

# ---------------------------------------------------------------
# Error / tolerance ratio
# ---------------------------------------------------------------

df["gaussian_error_to_tolerance"] = (
    df["gaussian_completeness_error"]
    /
    df["gaussian_tolerance"]
)

df["black_error_to_tolerance"] = (
    df["black_completeness_error"]
    /
    df["black_tolerance"]
)

# ---------------------------------------------------------------
# Pass/fail independently reconstructed
# ---------------------------------------------------------------

df["gaussian_failure_reconstructed"] = (
    df["gaussian_error_to_tolerance"] > 1.0
)

df["black_failure_reconstructed"] = (
    df["black_error_to_tolerance"] > 1.0
)

assert (
    df["gaussian_failure_reconstructed"].values
    ==
    df["gaussian_completeness_failed"].values
).all()

assert (
    df["black_failure_reconstructed"].values
    ==
    df["black_completeness_failed"].values
).all()

# ---------------------------------------------------------------
# Print detailed table
# ---------------------------------------------------------------

cols = [
    "image",
    "gaussian_delta",
    "gaussian_completeness_error",
    "gaussian_tolerance",
    "gaussian_error_to_tolerance",
    "gaussian_steps_used",
    "gaussian_completeness_failed",

    "black_delta",
    "black_completeness_error",
    "black_tolerance",
    "black_error_to_tolerance",
    "black_steps_used",
    "black_completeness_failed",
]

display(
    df[cols].sort_values(
        "gaussian_error_to_tolerance",
        ascending=False
    ).reset_index(drop=True)
)

# ---------------------------------------------------------------
# Summary
# ---------------------------------------------------------------

print("\n" + "=" * 70)
print("SUMMARY")
print("=" * 70)

print(
    "Gaussian failures:",
    int(df["gaussian_completeness_failed"].sum()),
    "/",
    len(df)
)

print(
    "Black failures:",
    int(df["black_completeness_failed"].sum()),
    "/",
    len(df)
)

print(
    "\nGaussian max error/tolerance:",
    f"{df['gaussian_error_to_tolerance'].max():.4f}"
)

print(
    "Gaussian median error/tolerance:",
    f"{df['gaussian_error_to_tolerance'].median():.4f}"
)

print(
    "Black max error/tolerance:",
    f"{df['black_error_to_tolerance'].max():.4f}"
)

print(
    "Black median error/tolerance:",
    f"{df['black_error_to_tolerance'].median():.4f}"
)

print(
    "\nGaussian |delta| range:",
    f"{df['gaussian_delta'].abs().min():.6f}",
    "to",
    f"{df['gaussian_delta'].abs().max():.6f}"
)

print(
    "Black |delta| range:",
    f"{df['black_delta'].abs().min():.6f}",
    "to",
    f"{df['black_delta'].abs().max():.6f}"
)

# ---------------------------------------------------------------
# Which observations failed?
# ---------------------------------------------------------------

print("\nGaussian failures:")

for _, r in df[
    df["gaussian_completeness_failed"]
].iterrows():

    print(
        f"  {r['image']} | "
        f"|delta|={abs(r['gaussian_delta']):.6f} | "
        f"error={r['gaussian_completeness_error']:.6f} | "
        f"tol={r['gaussian_tolerance']:.6f} | "
        f"ratio={r['gaussian_error_to_tolerance']:.3f} | "
        f"steps={int(r['gaussian_steps_used'])}"
    )

print("\nBlack failures:")

for _, r in df[
    df["black_completeness_failed"]
].iterrows():

    print(
        f"  {r['image']} | "
        f"|delta|={abs(r['black_delta']):.6f} | "
        f"error={r['black_completeness_error']:.6f} | "
        f"tol={r['black_tolerance']:.6f} | "
        f"ratio={r['black_error_to_tolerance']:.3f} | "
        f"steps={int(r['black_steps_used'])}"
    )

print(
    "\nCELL 16B COMPLETE — NO NEW IG COMPUTED."
)

CELL 16B — COMPLETENESS / RETRY DIAGNOSTIC


,image,gaussian_delta,gaussian_completeness_error,gaussian_tolerance,gaussian_error_to_tolerance,gaussian_steps_used,gaussian_completeness_failed,black_delta,black_completeness_error,black_tolerance,black_error_to_tolerance,black_steps_used,black_completeness_failed
0,ISIC_0071722,-2.173762,0.742072,0.108688,6.827535,100,True,5.395520,0.103982,0.269776,0.385438,100,False
1,ISIC_0055108,0.317873,0.315253,0.050000,6.305060,100,True,-1.100229,0.051907,0.055011,0.943564,50,False
2,ISIC_0069001,-3.647220,0.884802,0.182361,4.851926,100,True,-3.542264,0.125839,0.177113,0.710502,100,False
3,ISIC_0032318,2.127849,0.195702,0.106392,1.839436,100,True,5.547820,0.023155,0.277391,0.083475,100,False
4,ISIC_0030763,-9.153769,0.627220,0.457688,1.370409,100,True,-10.935117,0.478927,0.546756,0.875942,50,False
5,ISIC_0058556,7.595016,0.504271,0.379751,1.327899,100,True,8.559875,0.014183,0.427994,0.033138,50,False
6,ISIC_0029781,10.286134,0.668884,0.514307,1.300555,100,True,11.182228,0.273399,0.559111,0.488989,50,False
7,ISIC_0064735,1.431848,0.060375,0.071592,0.843322,50,False,2.391564,0.135926,0.119578,1.136714,100,True
8,ISIC_0068243,9.511440,0.338003,0.475572,0.710730,50,False,13.588003,0.304280,0.679400,0.447866,100,False
9,ISIC_0071629,-6.032583,0.184824,0.301629,0.612752,100,False,-8.921570,0.009903,0.446078,0.022200,50,False



SUMMARY
Gaussian failures: 7 / 15
Black failures: 2 / 15

Gaussian max error/tolerance: 6.8275
Gaussian median error/tolerance: 0.8433
Black max error/tolerance: 3.1309
Black median error/tolerance: 0.4479

Gaussian |delta| range: 0.317873 to 11.720753
Black |delta| range: 1.100229 to 13.588003

Gaussian failures:
  ISIC_0030763 | |delta|=9.153769 | error=0.627220 | tol=0.457688 | ratio=1.370 | steps=100
  ISIC_0055108 | |delta|=0.317873 | error=0.315253 | tol=0.050000 | ratio=6.305 | steps=100
  ISIC_0069001 | |delta|=3.647220 | error=0.884802 | tol=0.182361 | ratio=4.852 | steps=100
  ISIC_0032318 | |delta|=2.127849 | error=0.195702 | tol=0.106392 | ratio=1.839 | steps=100
  ISIC_0029781 | |delta|=10.286134 | error=0.668884 | tol=0.514307 | ratio=1.301 | steps=100
  ISIC_0071722 | |delta|=2.173762 | error=0.742072 | tol=0.108688 | ratio=6.828 | steps=100
  ISIC_0058556 | |delta|=7.595016 | error=0.504271 | tol=0.379751 | ratio=1.328 | steps=100

Black failures:
  ISIC_0064735 | |del

In [22]:
# ================================================================
# CELL 16C — TARGETED 2000-STEP NUMERICAL SENSITIVITY AUDIT
#
# PURPOSE:
# Determine whether the frozen 50 -> 100 step IG failures
# materially change enrichment values.
#
# IMPORTANT:
# This is a diagnostic only.
# It does NOT replace the locked T1.4c production protocol.
# ================================================================

from pathlib import Path
import numpy as np
import pandas as pd
import torch


print("=" * 70)
print("CELL 16C — TARGETED 2000-STEP NUMERICAL SENSITIVITY AUDIT")
print("=" * 70)


# ----------------------------------------------------------------
# 1. Load Cell 16A results
# ----------------------------------------------------------------

RESULT_PATH = Path(
    "/kaggle/working/"
    "t1_4c_one_model_validation/"
    "T1_4c_one_model_15_image_validation.csv"
)

df16a = pd.read_csv(
    RESULT_PATH
)

assert len(df16a) == 15


# ----------------------------------------------------------------
# 2. Reconstruct frozen tolerances and identify failures
# ----------------------------------------------------------------

df16a["gaussian_tolerance"] = np.maximum(
    0.05 * df16a["gaussian_delta"].abs(),
    0.05
)

df16a["black_tolerance"] = np.maximum(
    0.05 * df16a["black_delta"].abs(),
    0.05
)

gaussian_failed_images = set(
    df16a.loc[
        df16a["gaussian_completeness_failed"],
        "image"
    ]
)

black_failed_images = set(
    df16a.loc[
        df16a["black_completeness_failed"],
        "image"
    ]
)

target_images = sorted(
    gaussian_failed_images
    |
    black_failed_images
)

print(
    "\nGaussian failed:",
    len(gaussian_failed_images)
)

print(
    "Black failed:",
    len(black_failed_images)
)

print(
    "Unique target images:",
    len(target_images)
)

print(
    "\nTarget images:"
)

for image_id in target_images:
    print(" ", image_id)


# ----------------------------------------------------------------
# 3. Recover model/cache
#
# Cell 16A variables may still exist, but we explicitly verify
# them rather than relying on hidden notebook state.
# ----------------------------------------------------------------

MODEL_NAME = "rho_0.0_rep_1"

CACHE_DIR = Path(
    "/kaggle/working/t1_4c_xai_cache_500"
)

HAIR_PATH = CACHE_DIR / "hair_500.npy"
MASK_PATH = CACHE_DIR / "mask_500.npy"

MANIFEST_PATH = Path(
    "/kaggle/working/"
    "t1_4c_full_evaluation_lock/"
    "T1_4c_full_evaluation_manifest.csv"
)

assert HAIR_PATH.exists()
assert MASK_PATH.exists()
assert MANIFEST_PATH.exists()


# ----------------------------------------------------------------
# 4. Use existing production model if available
# ----------------------------------------------------------------

assert "model" in globals(), (
    "Production model from Cell 16A is not available. "
    "Re-run Cell 16A before Cell 16C."
)

assert "device" in globals()
assert device.type == "cuda"

assert "ig_enrichment" in globals()


# ----------------------------------------------------------------
# 5. Load cache
# ----------------------------------------------------------------

hair_cache = np.load(
    HAIR_PATH,
    mmap_mode="r"
)

mask_cache = np.load(
    MASK_PATH,
    mmap_mode="r"
)

manifest = pd.read_csv(
    MANIFEST_PATH
)

model_manifest = (
    manifest[
        manifest["model"] == MODEL_NAME
    ]
    .sort_values(
        "image_local_index"
    )
    .reset_index(drop=True)
)

assert len(model_manifest) == 15


# ----------------------------------------------------------------
# 6. Generalized high-step Gaussian IG
#
# Uses EXACT production mathematical implementation,
# but explicitly requests 2000 integration steps.
# ----------------------------------------------------------------

def gaussian_highstep_ig(
    model,
    hair_u8,
    mask_bool,
    device,
    steps=2000,
    sigma=10.0,
    chunk=25,
):

    import torchvision.transforms.functional as TF

    mean = torch.tensor(
        ap._MEAN,
        device=device
    ).view(
        1, 3, 1, 1
    )

    std = torch.tensor(
        ap._STD,
        device=device
    ).view(
        1, 3, 1, 1
    )

    model.eval()

    hair_u8 = np.array(
        hair_u8,
        dtype=np.uint8,
        order="C",
        copy=True
    )

    x = (
        torch.from_numpy(
            hair_u8
        )
        .to(device)
        .permute(2, 0, 1)
        .float()
        .unsqueeze(0)
        / 255.0
    )

    k = (
        2
        *
        int(
            math.ceil(
                3 * sigma
            )
        )
        + 1
    )

    base = TF.gaussian_blur(
        x,
        kernel_size=[k, k],
        sigma=[sigma, sigma]
    )

    xn = (
        x - mean
    ) / std

    bn = (
        base - mean
    ) / std

    mask_bool = np.array(
        mask_bool,
        dtype=bool,
        order="C",
        copy=True
    )

    m = torch.from_numpy(
        mask_bool
    ).to(device).bool()

    def zf(t):

        o = model(t)

        return o[:, 1] - o[:, 0]

    acc = torch.zeros_like(
        xn
    )

    alphas = (
        torch.arange(
            1,
            steps + 1,
            device=device,
            dtype=torch.float32
        )
        /
        steps
    )

    for s in range(
        0,
        steps,
        chunk
    ):

        a = (
            alphas[
                s:s + chunk
            ]
            .view(
                -1,
                1,
                1,
                1
            )
        )

        pts = (
            bn
            +
            a * (xn - bn)
        ).requires_grad_(True)

        g = torch.autograd.grad(
            zf(pts).sum(),
            pts
        )[0]

        acc += g.sum(
            0,
            keepdim=True
        )

    ig = (
        (xn - bn)
        * acc
        /
        steps
    )

    with torch.no_grad():

        delta = float(
            (
                zf(xn)
                -
                zf(bn)
            ).item()
        )

        a = (
            ig.abs()
            .sum(1)[0]
        )

        total = float(
            a.sum().item()
        )

        area_frac = float(
            m.float()
            .mean()
            .item()
        )

        enrichment = float(
            (
                a[m].sum().item()
                /
                total
            )
            /
            area_frac
        )

        signed_inside = float(
            ig.sum(1)[0][m]
            .sum()
            .item()
        )

    error = abs(
        float(
            ig.sum().item()
        )
        -
        delta
    )

    tolerance = max(
        0.05 * abs(delta),
        0.05
    )

    return {
        "enrichment":
            enrichment,

        "delta":
            delta,

        "completeness_error":
            error,

        "tolerance":
            tolerance,

        "error_to_tolerance":
            error / tolerance,

        "signed_inside":
            signed_inside,

        "steps":
            steps,
    }


# ----------------------------------------------------------------
# 7. Generalized high-step BLACK IG
# ----------------------------------------------------------------

def black_highstep_ig(
    model,
    hair_u8,
    mask_bool,
    device,
    steps=2000,
    chunk=25,
):

    mean = torch.tensor(
        ap._MEAN,
        device=device
    ).view(
        1, 3, 1, 1
    )

    std = torch.tensor(
        ap._STD,
        device=device
    ).view(
        1, 3, 1, 1
    )

    model.eval()

    hair_u8 = np.array(
        hair_u8,
        dtype=np.uint8,
        order="C",
        copy=True
    )

    x = (
        torch.from_numpy(
            hair_u8
        )
        .to(device)
        .permute(2, 0, 1)
        .float()
        .unsqueeze(0)
        / 255.0
    )

    # Raw RGB black baseline
    base = torch.zeros_like(x)

    xn = (
        x - mean
    ) / std

    bn = (
        base - mean
    ) / std

    mask_bool = np.array(
        mask_bool,
        dtype=bool,
        order="C",
        copy=True
    )

    m = torch.from_numpy(
        mask_bool
    ).to(device).bool()

    def zf(t):

        o = model(t)

        return o[:, 1] - o[:, 0]

    acc = torch.zeros_like(
        xn
    )

    alphas = (
        torch.arange(
            1,
            steps + 1,
            device=device,
            dtype=torch.float32
        )
        /
        steps
    )

    for s in range(
        0,
        steps,
        chunk
    ):

        a = (
            alphas[
                s:s + chunk
            ]
            .view(
                -1,
                1,
                1,
                1
            )
        )

        pts = (
            bn
            +
            a * (xn - bn)
        ).requires_grad_(True)

        g = torch.autograd.grad(
            zf(pts).sum(),
            pts
        )[0]

        acc += g.sum(
            0,
            keepdim=True
        )

    ig = (
        (xn - bn)
        * acc
        /
        steps
    )

    with torch.no_grad():

        delta = float(
            (
                zf(xn)
                -
                zf(bn)
            ).item()
        )

        a = (
            ig.abs()
            .sum(1)[0]
        )

        total = float(
            a.sum().item()
        )

        area_frac = float(
            m.float()
            .mean()
            .item()
        )

        enrichment = float(
            (
                a[m].sum().item()
                /
                total
            )
            /
            area_frac
        )

        signed_inside = float(
            ig.sum(1)[0][m]
            .sum()
            .item()
        )

    error = abs(
        float(
            ig.sum().item()
        )
        -
        delta
    )

    tolerance = max(
        0.05 * abs(delta),
        0.05
    )

    return {
        "enrichment":
            enrichment,

        "delta":
            delta,

        "completeness_error":
            error,

        "tolerance":
            tolerance,

        "error_to_tolerance":
            error / tolerance,

        "signed_inside":
            signed_inside,

        "steps":
            steps,
    }


# ----------------------------------------------------------------
# 8. Run targeted audit
# ----------------------------------------------------------------

audit_rows = []

for image_id in target_images:

    row = model_manifest[
        model_manifest["image"]
        == image_id
    ]

    assert len(row) == 1

    row = row.iloc[0]

    cache_row = int(
        row["cache_row"]
    )

    hair_u8 = np.array(
        hair_cache[cache_row],
        dtype=np.uint8,
        order="C",
        copy=True
    )

    mask_bool = (
        np.array(
            mask_cache[cache_row],
            dtype=np.uint8,
            order="C",
            copy=True
        )
        > 0
    )

    gaussian_failed = (
        image_id
        in gaussian_failed_images
    )

    black_failed = (
        image_id
        in black_failed_images
    )

    print(
        "\n" + "-" * 70
    )

    print(
        image_id,
        "| Gaussian failed:",
        gaussian_failed,
        "| Black failed:",
        black_failed
    )

    # ------------------------------------------------------------
    # Only calculate a baseline's 2000-step value if it failed
    # under the production 50 -> 100-step criterion.
    # ------------------------------------------------------------

    gaussian_ref = None
    black_ref = None

    if gaussian_failed:

        gaussian_ref = gaussian_highstep_ig(
            model,
            hair_u8,
            mask_bool,
            device,
            steps=2000
        )

        print(
            "Gaussian 2000-step:",
            f"enrichment={gaussian_ref['enrichment']:.9f}",
            f"| error/tol={gaussian_ref['error_to_tolerance']:.4f}"
        )

    if black_failed:

        black_ref = black_highstep_ig(
            model,
            hair_u8,
            mask_bool,
            device,
            steps=2000
        )

        print(
            "Black 2000-step:",
            f"enrichment={black_ref['enrichment']:.9f}",
            f"| error/tol={black_ref['error_to_tolerance']:.4f}"
        )

    original = df16a[
        df16a["image"] == image_id
    ].iloc[0]

    record = {
        "image":
            image_id,

        "gaussian_failed":
            gaussian_failed,

        "black_failed":
            black_failed,
    }

    if gaussian_ref is not None:

        original_g = float(
            original[
                "gaussian_enrichment"
            ]
        )

        record.update(
            {
                "gaussian_50_100_enrichment":
                    original_g,

                "gaussian_2000_enrichment":
                    gaussian_ref[
                        "enrichment"
                    ],

                "gaussian_abs_enrichment_change":
                    abs(
                        gaussian_ref[
                            "enrichment"
                        ]
                        -
                        original_g
                    ),

                "gaussian_relative_enrichment_change_pct":
                    100.0
                    *
                    abs(
                        gaussian_ref[
                            "enrichment"
                        ]
                        -
                        original_g
                    )
                    /
                    abs(original_g),

                "gaussian_2000_error":
                    gaussian_ref[
                        "completeness_error"
                    ],

                "gaussian_2000_tolerance":
                    gaussian_ref[
                        "tolerance"
                    ],

                "gaussian_2000_error_to_tolerance":
                    gaussian_ref[
                        "error_to_tolerance"
                    ],
            }
        )

    if black_ref is not None:

        original_b = float(
            original[
                "black_enrichment"
            ]
        )

        record.update(
            {
                "black_50_100_enrichment":
                    original_b,

                "black_2000_enrichment":
                    black_ref[
                        "enrichment"
                    ],

                "black_abs_enrichment_change":
                    abs(
                        black_ref[
                            "enrichment"
                        ]
                        -
                        original_b
                    ),

                "black_relative_enrichment_change_pct":
                    100.0
                    *
                    abs(
                        black_ref[
                            "enrichment"
                        ]
                        -
                        original_b
                    )
                    /
                    abs(original_b),

                "black_2000_error":
                    black_ref[
                        "completeness_error"
                    ],

                "black_2000_tolerance":
                    black_ref[
                        "tolerance"
                    ],

                "black_2000_error_to_tolerance":
                    black_ref[
                        "error_to_tolerance"
                    ],
            }
        )

    audit_rows.append(
        record
    )


# ----------------------------------------------------------------
# 9. Save audit
# ----------------------------------------------------------------

audit_df = pd.DataFrame(
    audit_rows
)

OUT_DIR = Path(
    "/kaggle/working/"
    "t1_4c_one_model_validation"
)

AUDIT_PATH = (
    OUT_DIR
    /
    "T1_4c_targeted_2000step_audit.csv"
)

audit_df.to_csv(
    AUDIT_PATH,
    index=False
)


# ----------------------------------------------------------------
# 10. Final report
# ----------------------------------------------------------------

print("\n" + "=" * 70)
print("CELL 16C — 2000-STEP AUDIT RESULT")
print("=" * 70)

display(
    audit_df
)

if (
    "gaussian_relative_enrichment_change_pct"
    in audit_df.columns
):

    g_changes = audit_df[
        "gaussian_relative_enrichment_change_pct"
    ].dropna()

    print(
        "\nGaussian failed-case relative enrichment change:"
    )

    print(
        "Mean:",
        f"{g_changes.mean():.4f}%"
    )

    print(
        "Median:",
        f"{g_changes.median():.4f}%"
    )

    print(
        "Maximum:",
        f"{g_changes.max():.4f}%"
    )

if (
    "black_relative_enrichment_change_pct"
    in audit_df.columns
):

    b_changes = audit_df[
        "black_relative_enrichment_change_pct"
    ].dropna()

    print(
        "\nBlack failed-case relative enrichment change:"
    )

    print(
        "Mean:",
        f"{b_changes.mean():.4f}%"
    )

    print(
        "Median:",
        f"{b_changes.median():.4f}%"
    )

    print(
        "Maximum:",
        f"{b_changes.max():.4f}%"
    )

print(
    "\nSaved:",
    AUDIT_PATH
)

print(
    "\nCELL 16C COMPLETE."
)

print(
    "This was a numerical sensitivity audit only."
)

print(
    "The locked T1.4c 50 -> 100-step protocol has NOT been changed."
)

CELL 16C — TARGETED 2000-STEP NUMERICAL SENSITIVITY AUDIT

Gaussian failed: 7
Black failed: 2
Unique target images: 9

Target images:
  ISIC_0029781
  ISIC_0030763
  ISIC_0032318
  ISIC_0055108
  ISIC_0058556
  ISIC_0064735
  ISIC_0069001
  ISIC_0070236
  ISIC_0071722

----------------------------------------------------------------------
ISIC_0029781 | Gaussian failed: True | Black failed: False
Gaussian 2000-step: enrichment=8.058772996 | error/tol=0.0524

----------------------------------------------------------------------
ISIC_0030763 | Gaussian failed: True | Black failed: False
Gaussian 2000-step: enrichment=7.161335590 | error/tol=0.0126

----------------------------------------------------------------------
ISIC_0032318 | Gaussian failed: True | Black failed: False
Gaussian 2000-step: enrichment=4.551686710 | error/tol=0.0021

----------------------------------------------------------------------
ISIC_0055108 | Gaussian failed: True | Black failed: False
Gaussian 2000-step: e

,image,gaussian_failed,black_failed,gaussian_50_100_enrichment,gaussian_2000_enrichment,gaussian_abs_enrichment_change,gaussian_relative_enrichment_change_pct,gaussian_2000_error,gaussian_2000_tolerance,gaussian_2000_error_to_tolerance,black_50_100_enrichment,black_2000_enrichment,black_abs_enrichment_change,black_relative_enrichment_change_pct,black_2000_error,black_2000_tolerance,black_2000_error_to_tolerance
0,ISIC_0029781,True,False,8.053228,8.058773,0.005545,0.068849,0.026948,0.514307,0.052397,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,ISIC_0030763,True,False,7.130098,7.161336,0.031237,0.438102,0.005762,0.457688,0.012590,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,ISIC_0032318,True,False,4.555048,4.551687,0.003361,0.073790,0.000228,0.106392,0.002147,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,ISIC_0055108,True,False,3.910237,3.906791,0.003446,0.088116,0.006780,0.050000,0.135608,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,ISIC_0058556,True,False,8.231350,8.272905,0.041555,0.504844,0.003833,0.379751,0.010094,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,ISIC_0064735,False,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.088737,0.088537,0.000200,0.225295,0.003219,0.119578,0.026917
6,ISIC_0069001,True,False,2.941068,2.941383,0.000315,0.010713,0.026002,0.182361,0.142586,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,ISIC_0070236,False,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.196609,0.196644,0.000035,0.017586,0.014820,0.110074,0.134640
8,ISIC_0071722,True,False,3.224669,3.224591,0.000078,0.002431,0.024510,0.108688,0.225509,NaN,NaN,NaN,NaN,NaN,NaN,NaN



Gaussian failed-case relative enrichment change:
Mean: 0.1695%
Median: 0.0738%
Maximum: 0.5048%

Black failed-case relative enrichment change:
Mean: 0.1214%
Median: 0.1214%
Maximum: 0.2253%

Saved: /kaggle/working/t1_4c_one_model_validation/T1_4c_targeted_2000step_audit.csv

CELL 16C COMPLETE.
This was a numerical sensitivity audit only.
The locked T1.4c 50 -> 100-step protocol has NOT been changed.


In [23]:
# ================================================================
# CELL 16D — T1.4c FINAL PRE-FLIGHT GATE
# CPU ONLY — NO NEW IG COMPUTATION
# ================================================================

from pathlib import Path
import json
import pandas as pd
import numpy as np


print("=" * 70)
print("CELL 16D — T1.4c FINAL PRE-FLIGHT GATE")
print("=" * 70)


# ----------------------------------------------------------------
# 1. Load validation artifacts
# ----------------------------------------------------------------

BASE_DIR = Path(
    "/kaggle/working/t1_4c_one_model_validation"
)

VAL_PATH = (
    BASE_DIR
    /
    "T1_4c_one_model_15_image_validation.csv"
)

AUDIT_PATH = (
    BASE_DIR
    /
    "T1_4c_targeted_2000step_audit.csv"
)

MANIFEST_PATH = Path(
    "/kaggle/working/"
    "t1_4c_full_evaluation_lock/"
    "T1_4c_full_evaluation_manifest.csv"
)

assert VAL_PATH.exists()
assert AUDIT_PATH.exists()
assert MANIFEST_PATH.exists()


val = pd.read_csv(
    VAL_PATH
)

audit = pd.read_csv(
    AUDIT_PATH
)

manifest = pd.read_csv(
    MANIFEST_PATH
)


# ----------------------------------------------------------------
# 2. Structural checks
# ----------------------------------------------------------------

assert len(val) == 15
assert len(audit) == 9
assert len(manifest) == 450

assert val["image"].nunique() == 15

assert manifest[
    "model"
].nunique() == 30

assert manifest[
    ["model", "image"]
].drop_duplicates().shape[0] == 450

print(
    "✓ Validation rows:",
    len(val)
)

print(
    "✓ Targeted numerical audit rows:",
    len(audit)
)

print(
    "✓ Full production manifest rows:",
    len(manifest)
)


# ----------------------------------------------------------------
# 3. Check all enrichment endpoints are finite
# ----------------------------------------------------------------

assert np.isfinite(
    val["gaussian_enrichment"]
).all()

assert np.isfinite(
    val["black_enrichment"]
).all()

print(
    "✓ All validation enrichments finite."
)


# ----------------------------------------------------------------
# 4. Check undefined counts
# ----------------------------------------------------------------

gaussian_undefined = int(
    val["gaussian_undefined"].sum()
)

black_undefined = int(
    val["black_undefined"].sum()
)

print(
    "\nGaussian undefined:",
    gaussian_undefined,
    "/ 15"
)

print(
    "Black undefined:",
    black_undefined,
    "/ 15"
)

assert gaussian_undefined == 0
assert black_undefined == 0


# ----------------------------------------------------------------
# 5. Completeness is retained, NOT filtered
# ----------------------------------------------------------------

gaussian_failures = int(
    val[
        "gaussian_completeness_failed"
    ].sum()
)

black_failures = int(
    val[
        "black_completeness_failed"
    ].sum()
)

print(
    "\nGaussian completeness failures:",
    gaussian_failures,
    "/ 15"
)

print(
    "Black completeness failures:",
    black_failures,
    "/ 15"
)

# We intentionally DO NOT assert zero failures.
#
# These are QA fields and remain in the production output.


# ----------------------------------------------------------------
# 6. Targeted high-step stability evidence
# ----------------------------------------------------------------

gaussian_changes = audit[
    "gaussian_relative_enrichment_change_pct"
].dropna()

black_changes = audit[
    "black_relative_enrichment_change_pct"
].dropna()

assert len(
    gaussian_changes
) == 7

assert len(
    black_changes
) == 2

g_mean = float(
    gaussian_changes.mean()
)

g_median = float(
    gaussian_changes.median()
)

g_max = float(
    gaussian_changes.max()
)

b_mean = float(
    black_changes.mean()
)

b_median = float(
    black_changes.median()
)

b_max = float(
    black_changes.max()
)


print(
    "\nTargeted 2000-step sensitivity:"
)

print(
    f"Gaussian mean/median/max: "
    f"{g_mean:.4f}% / "
    f"{g_median:.4f}% / "
    f"{g_max:.4f}%"
)

print(
    f"Black mean/median/max: "
    f"{b_mean:.4f}% / "
    f"{b_median:.4f}% / "
    f"{b_max:.4f}%"
)


# ----------------------------------------------------------------
# 7. Confirm the locked production protocol remains unchanged
# ----------------------------------------------------------------

LOCKED_PROTOCOL = {
    "ig_steps": 50,
    "ig_retry_steps": 100,
    "ig_sigma": 10.0,
    "gaussian_baseline":
        "Gaussian blur of same hair image",
    "black_baseline":
        "raw RGB [0,0,0] before normalization",
    "n_models": 30,
    "n_images_per_model": 15,
    "n_model_image_pairs": 450,
    "n_baselines": 2,
    "planned_ig_evaluations": 900,
}

print(
    "\nLocked production protocol:"
)

for k, v in LOCKED_PROTOCOL.items():

    print(
        f"  {k}: {v}"
    )


# ----------------------------------------------------------------
# 8. Confirm manifest coverage
# ----------------------------------------------------------------

counts = (
    manifest
    .groupby("model")
    .size()
)

assert len(counts) == 30
assert counts.eq(15).all()

rho_counts = (
    manifest
    .groupby("rho")["model"]
    .nunique()
)

assert rho_counts.eq(5).all()

print(
    "\n✓ Every model has exactly 15 images."
)

print(
    "✓ Every rho has exactly 5 models."
)


# ----------------------------------------------------------------
# 9. Final gate decision
# ----------------------------------------------------------------

GATE = (
    len(val) == 15
    and len(audit) == 9
    and len(manifest) == 450
    and gaussian_undefined == 0
    and black_undefined == 0
    and np.isfinite(
        val["gaussian_enrichment"]
    ).all()
    and np.isfinite(
        val["black_enrichment"]
    ).all()
    and counts.eq(15).all()
    and rho_counts.eq(5).all()
)


print("\n" + "=" * 70)

if GATE:

    print(
        "FINAL PRE-FLIGHT GATE: PASS"
    )

    print(
        "\nThe locked 50→100-step T1.4c "
        "production protocol is approved for the "
        "full 30-model evaluation."
    )

else:

    print(
        "FINAL PRE-FLIGHT GATE: FAIL"
    )

    raise RuntimeError(
        "T1.4c pre-flight gate failed."
    )


# ----------------------------------------------------------------
# 10. Save permanent gate record
# ----------------------------------------------------------------

GATE_PATH = (
    BASE_DIR
    /
    "T1_4c_final_preflight_gate.json"
)

gate_record = {
    "protocol":
        "T1.4c_black_baseline_sensitivity_v1",

    "gate":
        "PASS",

    "production_ig_steps":
        50,

    "production_retry_steps":
        100,

    "sigma":
        10.0,

    "n_models":
        30,

    "n_images_per_model":
        15,

    "n_model_image_pairs":
        450,

    "n_baselines":
        2,

    "planned_ig_evaluations":
        900,

    "validation_model":
        "rho_0.0_rep_1",

    "validation_images":
        15,

    "gaussian_completeness_failures_validation":
        gaussian_failures,

    "black_completeness_failures_validation":
        black_failures,

    "gaussian_undefined_validation":
        gaussian_undefined,

    "black_undefined_validation":
        black_undefined,

    "gaussian_targeted_2000step_mean_change_pct":
        g_mean,

    "gaussian_targeted_2000step_median_change_pct":
        g_median,

    "gaussian_targeted_2000step_max_change_pct":
        g_max,

    "black_targeted_2000step_mean_change_pct":
        b_mean,

    "black_targeted_2000step_median_change_pct":
        b_median,

    "black_targeted_2000step_max_change_pct":
        b_max,

    "interpretation":
        "Targeted 2000-step audit showed small "
        "enrichment changes despite some "
        "50-to-100-step completeness failures. "
        "Production protocol therefore remains "
        "locked at 50 steps with 100-step retry."
}

with open(
    GATE_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        gate_record,
        f,
        indent=2
    )

print(
    "\nSaved final gate record:"
)

print(
    GATE_PATH
)

print(
    "\nCELL 16D COMPLETE."
)

CELL 16D — T1.4c FINAL PRE-FLIGHT GATE
✓ Validation rows: 15
✓ Targeted numerical audit rows: 9
✓ Full production manifest rows: 450
✓ All validation enrichments finite.

Gaussian undefined: 0 / 15
Black undefined: 0 / 15

Gaussian completeness failures: 7 / 15
Black completeness failures: 2 / 15

Targeted 2000-step sensitivity:
Gaussian mean/median/max: 0.1695% / 0.0738% / 0.5048%
Black mean/median/max: 0.1214% / 0.1214% / 0.2253%

Locked production protocol:
  ig_steps: 50
  ig_retry_steps: 100
  ig_sigma: 10.0
  gaussian_baseline: Gaussian blur of same hair image
  black_baseline: raw RGB [0,0,0] before normalization
  n_models: 30
  n_images_per_model: 15
  n_model_image_pairs: 450
  n_baselines: 2
  planned_ig_evaluations: 900

✓ Every model has exactly 15 images.
✓ Every rho has exactly 5 models.

FINAL PRE-FLIGHT GATE: PASS

The locked 50→100-step T1.4c production protocol is approved for the full 30-model evaluation.

Saved final gate record:
/kaggle/working/t1_4c_one_model_val

In [24]:
# ================================================================
# CELL 17 — T1.4c FULL 30-MODEL PRODUCTION RUN
#
# 30 models × 15 images × 2 baselines = 900 IG evaluations
#
# LOCKED:
#   IG steps = 50
#   retry    = 100
#   sigma    = 10
#   Gaussian baseline = blurred same hair image
#   Black baseline    = raw RGB [0,0,0] before normalization
#
# Results are checkpointed after EVERY MODEL.
# ================================================================

from pathlib import Path
import hashlib
import json
import importlib.util
import math
import gc

import numpy as np
import pandas as pd
import torch


print("=" * 70)
print("CELL 17 — T1.4c FULL PRODUCTION RUN")
print("=" * 70)


# ----------------------------------------------------------------
# 1. LOCKED PATHS
# ----------------------------------------------------------------

PROJECT_CANDIDATES = list(
    Path("/kaggle/input/datasets").glob(
        "*/isic-final-project"
    )
)

assert PROJECT_CANDIDATES

preferred = [
    p for p in PROJECT_CANDIDATES
    if "tejassh" in str(p).lower()
]

PROJECT_ROOT = (
    preferred[0]
    if preferred
    else PROJECT_CANDIDATES[0]
)

ANALYSIS_PATH = (
    PROJECT_ROOT
    / "step2W_analysis_code_freeze"
    / "analysis_pipeline.py"
)

GENERATOR_PATH = (
    PROJECT_ROOT
    / "step1J_generator_recovery_lock"
    / "recovered_original_hair_generator.py"
)

MANIFEST_PATH = Path(
    "/kaggle/working/"
    "t1_4c_full_evaluation_lock/"
    "T1_4c_full_evaluation_manifest.csv"
)

HAIR_PATH = Path(
    "/kaggle/working/"
    "t1_4c_xai_cache_500/"
    "hair_500.npy"
)

MASK_PATH = Path(
    "/kaggle/working/"
    "t1_4c_xai_cache_500/"
    "mask_500.npy"
)

AREA_PATH = Path(
    "/kaggle/working/"
    "t1_4c_xai_cache_500/"
    "mask_area_500.npy"
)

ROOT_A = Path(
    "/kaggle/input/notebooks/"
    "tejasseshadriiiii/"
    "ml-open-lab-30m-trained/"
    "training_runs"
)

OUT_DIR = Path(
    "/kaggle/working/"
    "t1_4c_full_production"
)

OUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

RESULT_PATH = (
    OUT_DIR
    /
    "T1_4c_full_production_results.csv"
)

RUN_STATE_PATH = (
    OUT_DIR
    /
    "T1_4c_full_production_run_state.json"
)


assert ANALYSIS_PATH.exists()
assert GENERATOR_PATH.exists()
assert MANIFEST_PATH.exists()
assert HAIR_PATH.exists()
assert MASK_PATH.exists()
assert AREA_PATH.exists()
assert ROOT_A.exists()

print("\n✓ All production paths verified.")


# ----------------------------------------------------------------
# 2. HASH FROZEN ANALYSIS SOURCE
# ----------------------------------------------------------------

def sha256_file(path):

    h = hashlib.sha256()

    with open(
        path,
        "rb"
    ) as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):

            h.update(chunk)

    return h.hexdigest()


analysis_hash = sha256_file(
    ANALYSIS_PATH
)

generator_hash = sha256_file(
    GENERATOR_PATH
)

print(
    "Analysis SHA256:",
    analysis_hash
)

print(
    "Generator SHA256:",
    generator_hash
)


# ----------------------------------------------------------------
# 3. LOAD FROZEN ANALYSIS PIPELINE
# ----------------------------------------------------------------

spec = importlib.util.spec_from_file_location(
    "analysis_pipeline_t14c_full",
    str(ANALYSIS_PATH)
)

ap = importlib.util.module_from_spec(
    spec
)

spec.loader.exec_module(ap)

assert hasattr(
    ap,
    "ig_enrichment"
)

ig_enrichment = ap.ig_enrichment


# ----------------------------------------------------------------
# 4. VERIFY LOCKED IG PARAMETERS
# ----------------------------------------------------------------

IG_STEPS = int(
    ap.IG_STEPS
)

IG_RETRY_STEPS = int(
    ap.IG_RETRY_STEPS
)

IG_SIGMA = float(
    ap.IG_SIGMA
)

assert IG_STEPS == 50
assert IG_RETRY_STEPS == 100
assert IG_SIGMA == 10.0

print(
    "\n✓ Frozen IG configuration verified:"
)

print(
    "  steps:",
    IG_STEPS
)

print(
    "  retry:",
    IG_RETRY_STEPS
)

print(
    "  sigma:",
    IG_SIGMA
)


# ----------------------------------------------------------------
# 5. LOAD LOCKED MANIFEST
# ----------------------------------------------------------------

manifest = pd.read_csv(
    MANIFEST_PATH
)

assert len(manifest) == 450

assert (
    manifest["model"].nunique()
    == 30
)

assert (
    manifest[
        ["model", "image"]
    ]
    .drop_duplicates()
    .shape[0]
    == 450
)

counts = (
    manifest
    .groupby("model")
    .size()
)

assert counts.eq(15).all()

print(
    "\n✓ 450-row production manifest verified."
)


# ----------------------------------------------------------------
# 6. LOAD CACHE
# ----------------------------------------------------------------

hair_cache = np.load(
    HAIR_PATH,
    mmap_mode="r"
)

mask_cache = np.load(
    MASK_PATH,
    mmap_mode="r"
)

area_cache = np.load(
    AREA_PATH,
    mmap_mode="r"
)

assert hair_cache.shape == (
    500,
    224,
    224,
    3
)

assert mask_cache.shape == (
    500,
    224,
    224
)

assert area_cache.shape == (
    500,
)

print(
    "✓ Working cache verified."
)


# ----------------------------------------------------------------
# 7. CUDA
# ----------------------------------------------------------------

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

assert device.type == "cuda"

print(
    "✓ CUDA:",
    torch.cuda.get_device_name(0)
)


# ----------------------------------------------------------------
# 8. BLACK-BASELINE IG
# ----------------------------------------------------------------

def black_baseline_ig(
    model,
    hair_u8,
    mask_bool,
    device,
    steps=IG_STEPS,
    retry_steps=IG_RETRY_STEPS,
    chunk=25,
):

    mean = torch.tensor(
        ap._MEAN,
        device=device
    ).view(
        1,
        3,
        1,
        1
    )

    std = torch.tensor(
        ap._STD,
        device=device
    ).view(
        1,
        3,
        1,
        1
    )

    model.eval()

    hair_u8 = np.array(
        hair_u8,
        dtype=np.uint8,
        order="C",
        copy=True
    )

    x = (
        torch.from_numpy(
            hair_u8
        )
        .to(device)
        .permute(
            2,
            0,
            1
        )
        .float()
        .unsqueeze(0)
        / 255.0
    )

    # ------------------------------------------------------------
    # RAW RGB BLACK BASELINE
    # BEFORE ImageNet normalization.
    # ------------------------------------------------------------

    base = torch.zeros_like(
        x
    )

    xn = (
        x - mean
    ) / std

    bn = (
        base - mean
    ) / std

    mask_bool = np.array(
        mask_bool,
        dtype=bool,
        order="C",
        copy=True
    )

    m = torch.from_numpy(
        mask_bool
    ).to(device).bool()

    def zf(t):

        o = model(t)

        return (
            o[:, 1]
            -
            o[:, 0]
        )

    def run(n):

        acc = torch.zeros_like(
            xn
        )

        alphas = (
            torch.arange(
                1,
                n + 1,
                device=device,
                dtype=torch.float32
            )
            /
            n
        )

        for s in range(
            0,
            n,
            chunk
        ):

            a = (
                alphas[
                    s:s + chunk
                ]
                .view(
                    -1,
                    1,
                    1,
                    1
                )
            )

            pts = (
                bn
                +
                a
                *
                (xn - bn)
            ).requires_grad_(True)

            g = torch.autograd.grad(
                zf(pts).sum(),
                pts
            )[0]

            acc += g.sum(
                0,
                keepdim=True
            )

        ig = (
            (xn - bn)
            *
            acc
            /
            n
        )

        with torch.no_grad():

            delta = float(
                (
                    zf(xn)
                    -
                    zf(bn)
                ).item()
            )

        err = abs(
            float(
                ig.sum().item()
            )
            -
            delta
        )

        return (
            ig,
            delta,
            err
        )

    ig, delta, err = run(
        steps
    )

    used = steps

    tol = max(
        ap.IG_TOL_REL
        *
        abs(delta),
        ap.IG_TAU_ABS
    )

    if err > tol:

        ig, delta, err = run(
            retry_steps
        )

        used = retry_steps

    with torch.no_grad():

        a = (
            ig.abs()
            .sum(1)[0]
        )

        total = float(
            a.sum().item()
        )

        area_frac = float(
            m.float()
            .mean()
            .item()
        )

        undefined = (
            total <= ap.XAI_EPS
            or
            area_frac == 0.0
        )

        if undefined:

            enrichment = float(
                "nan"
            )

        else:

            enrichment = float(
                (
                    a[m].sum().item()
                    /
                    total
                )
                /
                area_frac
            )

        signed_in = float(
            ig.sum(1)[0][m]
            .sum()
            .item()
        )

    return {
        "enrichment":
            enrichment,

        "undefined":
            bool(undefined),

        "completeness_error":
            float(err),

        "delta":
            float(delta),

        "tolerance":
            float(tol),

        "steps_used":
            int(used),

        "completeness_failed":
            bool(
                err > tol
            ),

        "signed_attribution_inside_mask":
            signed_in,

        "mask_area_fraction":
            area_frac,
    }


print(
    "✓ Black-baseline evaluator ready."
)


# ----------------------------------------------------------------
# 9. RESULT SCHEMA
# ----------------------------------------------------------------

RESULT_COLUMNS = [
    "model",
    "rho",
    "replicate",
    "checkpoint",
    "cache_row",
    "image",
    "lesion_id",
    "mask_area",

    "gaussian_enrichment",
    "black_enrichment",

    "gaussian_completeness_error",
    "black_completeness_error",

    "gaussian_delta",
    "black_delta",

    "gaussian_tolerance",
    "black_tolerance",

    "gaussian_steps_used",
    "black_steps_used",

    "gaussian_completeness_failed",
    "black_completeness_failed",

    "gaussian_undefined",
    "black_undefined",

    "gaussian_signed_inside",
    "black_signed_inside",

    "mask_area_fraction",
]


# ----------------------------------------------------------------
# 10. RESUME SUPPORT
# ----------------------------------------------------------------

if RESULT_PATH.exists():

    existing = pd.read_csv(
        RESULT_PATH
    )

    assert set(
        RESULT_COLUMNS
    ).issubset(
        existing.columns
    )

    completed_models = set(
        existing["model"].astype(str)
    )

    results = (
        existing[
            RESULT_COLUMNS
        ]
        .to_dict(
            "records"
        )
    )

    print(
        "\nExisting production results found."
    )

    print(
        "Completed models:",
        len(completed_models),
        "/ 30"
    )

else:

    existing = pd.DataFrame(
        columns=RESULT_COLUMNS
    )

    completed_models = set()

    results = []

    print(
        "\nNo previous production result found."
    )


# ----------------------------------------------------------------
# 11. MODEL LOOP
# ----------------------------------------------------------------

model_names = (
    manifest[
        [
            "model",
            "rho",
            "replicate",
            "checkpoint"
        ]
    ]
    .drop_duplicates()
    .sort_values(
        [
            "rho",
            "replicate"
        ]
    )
    .reset_index(drop=True)
)


print(
    "\nModels scheduled:",
    len(model_names)
)

print(
    "Models already complete:",
    len(completed_models)
)


for model_idx, model_row in model_names.iterrows():

    model_name = str(
        model_row["model"]
    )

    if model_name in completed_models:

        print(
            f"\n[{model_idx+1:02d}/30] "
            f"{model_name} — SKIPPED "
            f"(already complete)"
        )

        continue


    print(
        "\n" + "=" * 70
    )

    print(
        f"[{model_idx+1:02d}/30] "
        f"STARTING {model_name}"
    )

    print(
        "rho:",
        float(model_row["rho"]),
        "| replicate:",
        int(model_row["replicate"])
    )

    print(
        "=" * 70
    )


    # ------------------------------------------------------------
    # Load model
    # ------------------------------------------------------------

    checkpoint_path = Path(
        str(
            model_row[
                "checkpoint"
            ]
        )
    )

    assert checkpoint_path.exists()

    import torchvision
    import torch.nn as nn

    model = torchvision.models.resnet18(
        weights=None
    )

    model.fc = nn.Linear(
        model.fc.in_features,
        2
    )

    checkpoint = torch.load(
        checkpoint_path,
        map_location="cpu"
    )

    missing, unexpected = (
        model.load_state_dict(
            checkpoint,
            strict=True
        )
    )

    assert not missing
    assert not unexpected

    model = model.to(
        device
    )

    model.eval()

    print(
        "Checkpoint loaded."
    )


    # ------------------------------------------------------------
    # Model-specific 15-image manifest
    # ------------------------------------------------------------

    model_rows = (
        manifest[
            manifest["model"]
            ==
            model_name
        ]
        .sort_values(
            "image_local_index"
        )
        .reset_index(drop=True)
    )

    assert len(model_rows) == 15


    # ------------------------------------------------------------
    # Evaluate all 15 images
    # ------------------------------------------------------------

    model_results = []

    for image_idx, row in model_rows.iterrows():

        cache_row = int(
            row["cache_row"]
        )

        image_id = str(
            row["image"]
        )

        lesion_id = str(
            row["lesion_id"]
        )

        locked_area = int(
            row["mask_area"]
        )

        hair_u8 = np.array(
            hair_cache[
                cache_row
            ],
            dtype=np.uint8,
            order="C",
            copy=True
        )

        mask_u8 = np.array(
            mask_cache[
                cache_row
            ],
            dtype=np.uint8,
            order="C",
            copy=True
        )

        mask_bool = (
            mask_u8 > 0
        )

        computed_area = int(
            mask_bool.sum()
        )

        assert (
            computed_area
            ==
            locked_area
        ), (
            f"Mask-area mismatch: "
            f"{model_name} / {image_id}"
        )


        # --------------------------------------------------------
        # Gaussian — EXACT frozen production function
        # --------------------------------------------------------

        gaussian = ig_enrichment(
            model,
            hair_u8,
            mask_bool,
            device
        )


        # --------------------------------------------------------
        # Black — same IG except baseline
        # --------------------------------------------------------

        black = black_baseline_ig(
            model,
            hair_u8,
            mask_bool,
            device
        )


        # --------------------------------------------------------
        # Do NOT filter failures.
        # Store every observation.
        # --------------------------------------------------------

        record = {
            "model":
                model_name,

            "rho":
                float(
                    row["rho"]
                ),

            "replicate":
                int(
                    row["replicate"]
                ),

            "checkpoint":
                str(
                    checkpoint_path
                ),

            "cache_row":
                cache_row,

            "image":
                image_id,

            "lesion_id":
                lesion_id,

            "mask_area":
                computed_area,

            "gaussian_enrichment":
                gaussian[
                    "enrichment"
                ],

            "black_enrichment":
                black[
                    "enrichment"
                ],

            "gaussian_completeness_error":
                gaussian[
                    "completeness_error"
                ],

            "black_completeness_error":
                black[
                    "completeness_error"
                ],

            "gaussian_delta":
                gaussian[
                    "delta"
                ],

            "black_delta":
                black[
                    "delta"
                ],

            "gaussian_tolerance":
                gaussian[
                    "tolerance"
                ],

            "black_tolerance":
                black[
                    "tolerance"
                ],

            "gaussian_steps_used":
                gaussian[
                    "steps_used"
                ],

            "black_steps_used":
                black[
                    "steps_used"
                ],

            "gaussian_completeness_failed":
                gaussian[
                    "completeness_failed"
                ],

            "black_completeness_failed":
                black[
                    "completeness_failed"
                ],

            "gaussian_undefined":
                gaussian[
                    "undefined"
                ],

            "black_undefined":
                black[
                    "undefined"
                ],

            "gaussian_signed_inside":
                gaussian[
                    "signed_attribution_inside_mask"
                ],

            "black_signed_inside":
                black[
                    "signed_attribution_inside_mask"
                ],

            "mask_area_fraction":
                gaussian[
                    "mask_area_fraction"
                ],
        }


        model_results.append(
            record
        )

        results.append(
            record
        )


        print(
            f"  [{image_idx+1:02d}/15] "
            f"{image_id} | "
            f"G={gaussian['enrichment']:.6f} | "
            f"B={black['enrichment']:.6f} | "
            f"Gfail={gaussian['completeness_failed']} | "
            f"Bfail={black['completeness_failed']}"
        )


    # ------------------------------------------------------------
    # Model-level checkpoint save
    # ------------------------------------------------------------

    result_df = pd.DataFrame(
        results
    )

    result_df = (
        result_df[
            RESULT_COLUMNS
        ]
        .drop_duplicates(
            subset=[
                "model",
                "image"
            ],
            keep="last"
        )
        .sort_values(
            [
                "rho",
                "replicate",
                "image"
            ]
        )
        .reset_index(drop=True)
    )

    result_df.to_csv(
        RESULT_PATH,
        index=False
    )

    completed_models.add(
        model_name
    )

    state = {
        "protocol":
            "T1.4c_black_baseline_sensitivity_v1",

        "status":
            "RUNNING",

        "completed_models":
            sorted(
                completed_models
            ),

        "n_completed_models":
            len(
                completed_models
            ),

        "n_completed_rows":
            len(
                result_df
            ),

        "expected_models":
            30,

        "expected_rows":
            450,

        "expected_ig_evaluations":
            900,

        "ig_steps":
            50,

        "ig_retry_steps":
            100,

        "sigma":
            10.0,

        "analysis_sha256":
            analysis_hash,

        "generator_sha256":
            generator_hash,
    }

    with open(
        RUN_STATE_PATH,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            state,
            f,
            indent=2
        )


    print(
        f"\n✓ {model_name} COMPLETE"
    )

    print(
        "Saved rows:",
        len(result_df)
    )

    print(
        "Saved to:",
        RESULT_PATH
    )


    # ------------------------------------------------------------
    # GPU memory cleanup
    # ------------------------------------------------------------

    del model
    del checkpoint

    gc.collect()

    torch.cuda.empty_cache()


# ----------------------------------------------------------------
# 12. FINAL RUN VALIDATION
# ----------------------------------------------------------------

final_df = pd.read_csv(
    RESULT_PATH
)

assert len(
    final_df
) == 450, (
    f"Expected 450 rows, "
    f"found {len(final_df)}"
)

assert (
    final_df[
        ["model", "image"]
    ]
    .drop_duplicates()
    .shape[0]
    ==
    450
)

assert (
    final_df["model"].nunique()
    ==
    30
)

assert (
    final_df["image"].nunique()
    ==
    15
)

model_counts = (
    final_df
    .groupby("model")
    .size()
)

assert model_counts.eq(15).all()


# ----------------------------------------------------------------
# 13. Baseline completeness / undefined summary
# ----------------------------------------------------------------

gaussian_failures = int(
    final_df[
        "gaussian_completeness_failed"
    ].sum()
)

black_failures = int(
    final_df[
        "black_completeness_failed"
    ].sum()
)

gaussian_undefined = int(
    final_df[
        "gaussian_undefined"
    ].sum()
)

black_undefined = int(
    final_df[
        "black_undefined"
    ].sum()
)


# ----------------------------------------------------------------
# 14. Add paired enrichment diagnostics
# ----------------------------------------------------------------

final_df[
    "abs_enrichment_difference"
] = (
    final_df[
        "gaussian_enrichment"
    ]
    -
    final_df[
        "black_enrichment"
    ]
).abs()

final_df[
    "relative_enrichment_difference_pct"
] = (
    final_df[
        "abs_enrichment_difference"
    ]
    /
    final_df[
        "gaussian_enrichment"
    ].abs()
    *
    100.0
)

final_df[
    "black_to_gaussian_enrichment_ratio"
] = (
    final_df[
        "black_enrichment"
    ]
    /
    final_df[
        "gaussian_enrichment"
    ]
)

final_df.to_csv(
    RESULT_PATH,
    index=False
)


# ----------------------------------------------------------------
# 15. Mark run complete
# ----------------------------------------------------------------

state = {
    "protocol":
        "T1.4c_black_baseline_sensitivity_v1",

    "status":
        "COMPLETE",

    "n_models":
        30,

    "n_images_per_model":
        15,

    "n_model_image_pairs":
        450,

    "n_baselines":
        2,

    "n_ig_evaluations":
        900,

    "rows":
        len(final_df),

    "gaussian_completeness_failures":
        gaussian_failures,

    "black_completeness_failures":
        black_failures,

    "gaussian_undefined":
        gaussian_undefined,

    "black_undefined":
        black_undefined,

    "analysis_sha256":
        analysis_hash,

    "generator_sha256":
        generator_hash,

    "result_path":
        str(RESULT_PATH),
}

with open(
    RUN_STATE_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        state,
        f,
        indent=2
    )


# ----------------------------------------------------------------
# 16. FINAL REPORT
# ----------------------------------------------------------------

print("\n" + "=" * 70)
print("CELL 17 — FULL T1.4c PRODUCTION COMPLETE")
print("=" * 70)

print(
    "Models:",
    final_df["model"].nunique()
)

print(
    "Images/model:",
    final_df.groupby(
        "model"
    ).size().min(),
    "to",
    final_df.groupby(
        "model"
    ).size().max()
)

print(
    "Model-image pairs:",
    len(final_df)
)

print(
    "IG evaluations:",
    len(final_df) * 2
)

print(
    "\nGaussian completeness failures:",
    gaussian_failures,
    "/ 450"
)

print(
    "Black completeness failures:",
    black_failures,
    "/ 450"
)

print(
    "Gaussian undefined:",
    gaussian_undefined,
    "/ 450"
)

print(
    "Black undefined:",
    black_undefined,
    "/ 450"
)

print(
    "\nMean Gaussian enrichment:",
    f"{final_df['gaussian_enrichment'].mean():.6f}"
)

print(
    "Mean Black enrichment:",
    f"{final_df['black_enrichment'].mean():.6f}"
)

print(
    "Mean absolute enrichment difference:",
    f"{final_df['abs_enrichment_difference'].mean():.6f}"
)

print(
    "Mean relative enrichment difference:",
    f"{final_df['relative_enrichment_difference_pct'].mean():.4f}%"
)

print(
    "\nRESULT FILE:",
    RESULT_PATH
)

print(
    "RUN STATE:",
    RUN_STATE_PATH
)

print(
    "\nT1.4c FULL PRODUCTION RUN COMPLETE."
)

CELL 17 — T1.4c FULL PRODUCTION RUN

✓ All production paths verified.
Analysis SHA256: bcba1d25782f5770fbe629b1d0281f87a4bfde29b343c731bec7cacdb87bcea5
Generator SHA256: 32be8092c85458bed31a547f2ba6c57c3e1c2cf257d7cb0f76bd28eebc7b26c2

✓ Frozen IG configuration verified:
  steps: 50
  retry: 100
  sigma: 10.0

✓ 450-row production manifest verified.
✓ Working cache verified.
✓ CUDA: Tesla T4
✓ Black-baseline evaluator ready.

No previous production result found.

Models scheduled: 30
Models already complete: 0

[01/30] STARTING rho_0.0_rep_1
rho: 0.0 | replicate: 1
Checkpoint loaded.
  [01/15] ISIC_0014089_downsampled | G=6.075407 | B=0.051157 | Gfail=False | Bfail=False
  [02/15] ISIC_0071629 | G=8.278198 | B=0.166949 | Gfail=False | Bfail=False
  [03/15] ISIC_0030763 | G=7.130098 | B=0.039012 | Gfail=True | Bfail=False
  [04/15] ISIC_0064735 | G=6.695275 | B=0.088737 | Gfail=False | Bfail=True
  [05/15] ISIC_0055108 | G=3.910236 | B=0.112081 | Gfail=True | Bfail=False
  [06/15] ISIC_

In [25]:
# ============================================================
# CELL 18A — T1.4c PRODUCTION RESULT LOAD + INTEGRITY CHECK
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import json

RESULT_DIR = Path("/kaggle/working/t1_4c_full_production")
RESULT_CSV = RESULT_DIR / "T1_4c_full_production_results.csv"
RUN_STATE = RESULT_DIR / "T1_4c_full_production_run_state.json"

assert RESULT_CSV.exists(), f"Missing production result: {RESULT_CSV}"
assert RUN_STATE.exists(), f"Missing run state: {RUN_STATE}"

df = pd.read_csv(RESULT_CSV)

print("=" * 70)
print("T1.4c — PRODUCTION RESULT INTEGRITY")
print("=" * 70)

print("Result path:", RESULT_CSV)
print("Rows:", len(df))
print("Columns:", len(df.columns))
print("\nColumns:")
print(df.columns.tolist())

print("\nFirst 3 rows:")
display(df.head(3))

# Expected production structure
assert len(df) == 450, f"Expected 450 rows, found {len(df)}"

required = [
    "model",
    "rho",
    "replicate",
    "image_id",
]

missing = [c for c in required if c not in df.columns]
assert not missing, f"Missing required columns: {missing}"

# Identify enrichment / QA columns robustly
gaussian_candidates = [
    "gaussian_enrichment",
    "enrichment_gaussian",
    "enrichment",
]

black_candidates = [
    "black_enrichment",
    "enrichment_black",
]

gauss_col = next((c for c in gaussian_candidates if c in df.columns), None)
black_col = next((c for c in black_candidates if c in df.columns), None)

print("\nGaussian enrichment column:", gauss_col)
print("Black enrichment column:", black_col)

assert gauss_col is not None, "Could not identify Gaussian enrichment column"
assert black_col is not None, "Could not identify black enrichment column"

# Required numeric fields
for c in ["rho", "replicate", gauss_col, black_col]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

assert df["rho"].isin([0.0, 0.2, 0.4, 0.6, 0.8, 1.0]).all()

# Expected 30 models × 15 images
print("\nUnique models:", df["model"].nunique())
print("Unique images:", df["image_id"].nunique())
print("Rows per model:")
print(df.groupby("model").size().value_counts().sort_index())

# Finite checks
print("\nFinite Gaussian enrichment:",
      np.isfinite(df[gauss_col]).sum(), "/", len(df))

print("Finite black enrichment:",
      np.isfinite(df[black_col]).sum(), "/", len(df))

assert np.isfinite(df[gauss_col]).all()
assert np.isfinite(df[black_col]).all()

# Rho × model balance
balance = df.groupby(["rho", "model"]).size()

print("\nRows per rho:")
print(df.groupby("rho").size())

print("\nRows per model/rho:")
print(balance.value_counts().sort_index())

assert balance.eq(15).all()

print("\nPASS — 450/450 production observations loaded and structurally valid.")

T1.4c — PRODUCTION RESULT INTEGRITY
Result path: /kaggle/working/t1_4c_full_production/T1_4c_full_production_results.csv
Rows: 450
Columns: 28

Columns:
['model', 'rho', 'replicate', 'checkpoint', 'cache_row', 'image', 'lesion_id', 'mask_area', 'gaussian_enrichment', 'black_enrichment', 'gaussian_completeness_error', 'black_completeness_error', 'gaussian_delta', 'black_delta', 'gaussian_tolerance', 'black_tolerance', 'gaussian_steps_used', 'black_steps_used', 'gaussian_completeness_failed', 'black_completeness_failed', 'gaussian_undefined', 'black_undefined', 'gaussian_signed_inside', 'black_signed_inside', 'mask_area_fraction', 'abs_enrichment_difference', 'relative_enrichment_difference_pct', 'black_to_gaussian_enrichment_ratio']

First 3 rows:


,model,rho,replicate,checkpoint,cache_row,image,lesion_id,mask_area,gaussian_enrichment,black_enrichment,...,gaussian_completeness_failed,black_completeness_failed,gaussian_undefined,black_undefined,gaussian_signed_inside,black_signed_inside,mask_area_fraction,abs_enrichment_difference,relative_enrichment_difference_pct,black_to_gaussian_enrichment_ratio
0,rho_0.0_rep_1,0.0,1,/kaggle/input/notebooks/tejasseshadriiiii/ml-o...,7,ISIC_0014089_downsampled,MSK4_0010395,483,6.075407,0.051157,...,False,False,False,False,-3.093883,0.144137,0.009626,6.024249,99.157964,0.008420
1,rho_0.0_rep_1,0.0,1,/kaggle/input/notebooks/tejasseshadriiiii/ml-o...,480,ISIC_0027057,HAM_0002021,558,4.369648,0.064560,...,False,False,False,False,-2.825740,0.021604,0.011121,4.305089,98.522546,0.014775
2,rho_0.0_rep_1,0.0,1,/kaggle/input/notebooks/tejasseshadriiiii/ml-o...,359,ISIC_0027517,HAM_0002906,310,8.296631,0.100778,...,False,False,False,False,-1.273960,0.543946,0.006178,8.195853,98.785313,0.012147


AssertionError: Missing required columns: ['image_id']

In [26]:
# ============================================================
# CELL 18A-FIX — EXACT PRODUCTION CSV SCHEMA
# ============================================================

print("Columns in T1.4c production CSV:\n")

for i, c in enumerate(df.columns):
    print(f"{i:02d}: {c}")

print("\nShape:", df.shape)

print("\nDtypes:")
print(df.dtypes)

print("\nFirst row:")
display(df.iloc[[0]].T)

Columns in T1.4c production CSV:

00: model
01: rho
02: replicate
03: checkpoint
04: cache_row
05: image
06: lesion_id
07: mask_area
08: gaussian_enrichment
09: black_enrichment
10: gaussian_completeness_error
11: black_completeness_error
12: gaussian_delta
13: black_delta
14: gaussian_tolerance
15: black_tolerance
16: gaussian_steps_used
17: black_steps_used
18: gaussian_completeness_failed
19: black_completeness_failed
20: gaussian_undefined
21: black_undefined
22: gaussian_signed_inside
23: black_signed_inside
24: mask_area_fraction
25: abs_enrichment_difference
26: relative_enrichment_difference_pct
27: black_to_gaussian_enrichment_ratio

Shape: (450, 28)

Dtypes:
model                                  object
rho                                   float64
replicate                               int64
checkpoint                             object
cache_row                               int64
image                                  object
lesion_id                              object
m

,0
model,rho_0.0_rep_1
rho,0.0
replicate,1
checkpoint,/kaggle/input/notebooks/tejasseshadriiiii/ml-o...
cache_row,7
image,ISIC_0014089_downsampled
lesion_id,MSK4_0010395
mask_area,483
gaussian_enrichment,6.075407
black_enrichment,0.051157


In [27]:
# ============================================================
# CELL 18A — T1.4c PRODUCTION RESULT FINAL INTEGRITY GATE
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import json

RESULT_DIR = Path("/kaggle/working/t1_4c_full_production")
RESULT_CSV = RESULT_DIR / "T1_4c_full_production_results.csv"
RUN_STATE = RESULT_DIR / "T1_4c_full_production_run_state.json"

assert RESULT_CSV.exists(), f"Missing production result: {RESULT_CSV}"
assert RUN_STATE.exists(), f"Missing run state: {RUN_STATE}"

df = pd.read_csv(RESULT_CSV)

# ------------------------------------------------------------
# Expected schema
# ------------------------------------------------------------

required = [
    "model",
    "rho",
    "replicate",
    "checkpoint",
    "cache_row",
    "image",
    "lesion_id",
    "mask_area",
    "gaussian_enrichment",
    "black_enrichment",
    "gaussian_completeness_error",
    "black_completeness_error",
    "gaussian_delta",
    "black_delta",
    "gaussian_tolerance",
    "black_tolerance",
    "gaussian_steps_used",
    "black_steps_used",
    "gaussian_completeness_failed",
    "black_completeness_failed",
    "gaussian_undefined",
    "black_undefined",
    "gaussian_signed_inside",
    "black_signed_inside",
    "mask_area_fraction",
    "abs_enrichment_difference",
    "relative_enrichment_difference_pct",
    "black_to_gaussian_enrichment_ratio",
]

missing = [c for c in required if c not in df.columns]

print("=" * 72)
print("T1.4c — FINAL PRODUCTION RESULT INTEGRITY GATE")
print("=" * 72)

print("Result:", RESULT_CSV)
print("Shape:", df.shape)
print("Missing required columns:", missing)

assert not missing, f"Missing required columns: {missing}"
assert len(df) == 450, f"Expected 450 rows, found {len(df)}"

# ------------------------------------------------------------
# Identity / balance
# ------------------------------------------------------------

assert df["model"].nunique() == 30
assert df["image"].nunique() == 15
assert df["lesion_id"].nunique() == 15

expected_rhos = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
assert sorted(df["rho"].unique().tolist()) == expected_rhos

model_counts = df.groupby("model").size()
assert model_counts.eq(15).all()

rho_counts = df.groupby("rho").size()
assert rho_counts.eq(75).all()

model_rho_counts = df.groupby(["rho", "model"]).size()
assert model_rho_counts.eq(15).all()

# ------------------------------------------------------------
# Finite numerical fields
# ------------------------------------------------------------

numeric_fields = [
    "gaussian_enrichment",
    "black_enrichment",
    "gaussian_completeness_error",
    "black_completeness_error",
    "gaussian_delta",
    "black_delta",
    "gaussian_tolerance",
    "black_tolerance",
    "gaussian_signed_inside",
    "black_signed_inside",
    "mask_area_fraction",
    "abs_enrichment_difference",
    "relative_enrichment_difference_pct",
    "black_to_gaussian_enrichment_ratio",
]

for c in numeric_fields:
    assert np.isfinite(df[c]).all(), f"Non-finite values found in {c}"

# ------------------------------------------------------------
# Undefined checks
# ------------------------------------------------------------

print("\nUndefined observations:")
print("  Gaussian:", int(df["gaussian_undefined"].sum()), "/ 450")
print("  Black:   ", int(df["black_undefined"].sum()), "/ 450")

assert df["gaussian_undefined"].sum() == 0
assert df["black_undefined"].sum() == 0

# ------------------------------------------------------------
# Completeness QA
# ------------------------------------------------------------

g_fail = int(df["gaussian_completeness_failed"].sum())
b_fail = int(df["black_completeness_failed"].sum())

print("\nCompleteness QA:")
print(f"  Gaussian failures: {g_fail} / 450 ({100*g_fail/450:.2f}%)")
print(f"  Black failures:    {b_fail} / 450 ({100*b_fail/450:.2f}%)")

# ------------------------------------------------------------
# Rho balance
# ------------------------------------------------------------

print("\nRows by rho:")
print(rho_counts)

print("\nModels per rho:")
print(df.groupby("rho")["model"].nunique())

# ------------------------------------------------------------
# Headline production values
# ------------------------------------------------------------

print("\nHeadline enrichment:")
print(f"  Gaussian mean: {df['gaussian_enrichment'].mean():.6f}")
print(f"  Black mean:    {df['black_enrichment'].mean():.6f}")
print(f"  Mean absolute difference: "
      f"{df['abs_enrichment_difference'].mean():.6f}")
print(f"  Mean relative difference: "
      f"{df['relative_enrichment_difference_pct'].mean():.4f}%")

print("\nPASS — T1.4c 450-row production table is structurally valid.")
print("No production values were modified or recomputed.")

T1.4c — FINAL PRODUCTION RESULT INTEGRITY GATE
Result: /kaggle/working/t1_4c_full_production/T1_4c_full_production_results.csv
Shape: (450, 28)
Missing required columns: []

Undefined observations:
  Gaussian: 0 / 450
  Black:    0 / 450

Completeness QA:
  Gaussian failures: 105 / 450 (23.33%)
  Black failures:    162 / 450 (36.00%)

Rows by rho:
rho
0.0    75
0.2    75
0.4    75
0.6    75
0.8    75
1.0    75
dtype: int64

Models per rho:
rho
0.0    5
0.2    5
0.4    5
0.6    5
0.8    5
1.0    5
Name: model, dtype: int64

Headline enrichment:
  Gaussian mean: 9.808741
  Black mean:    0.215778
  Mean absolute difference: 9.592963
  Mean relative difference: 97.0821%

PASS — T1.4c 450-row production table is structurally valid.
No production values were modified or recomputed.


In [28]:
# ============================================================
# CELL 18B — T1.4c BASELINE × RHO DESCRIPTIVE ANALYSIS
# ============================================================

import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1. Overall paired comparison
# ------------------------------------------------------------

print("=" * 72)
print("T1.4c — GAUSSIAN vs BLACK BASELINE")
print("=" * 72)

print("\nOverall enrichment:")
print(f"Gaussian mean   = {df['gaussian_enrichment'].mean():.6f}")
print(f"Gaussian median = {df['gaussian_enrichment'].median():.6f}")
print(f"Black mean      = {df['black_enrichment'].mean():.6f}")
print(f"Black median    = {df['black_enrichment'].median():.6f}")

print("\nOverall paired difference:")
paired_diff = df["gaussian_enrichment"] - df["black_enrichment"]

print(f"Mean difference   = {paired_diff.mean():.6f}")
print(f"Median difference = {paired_diff.median():.6f}")
print(f"SD difference     = {paired_diff.std(ddof=1):.6f}")

print("\nPaired direction:")
print(
    "Gaussian > Black:",
    int((df["gaussian_enrichment"] > df["black_enrichment"]).sum()),
    "/",
    len(df)
)
print(
    "Gaussian < Black:",
    int((df["gaussian_enrichment"] < df["black_enrichment"]).sum()),
    "/",
    len(df)
)
print(
    "Gaussian = Black:",
    int((df["gaussian_enrichment"] == df["black_enrichment"]).sum()),
    "/",
    len(df)
)

# ------------------------------------------------------------
# 2. Rho-stratified descriptive table
# ------------------------------------------------------------

rows = []

for rho, g in df.groupby("rho", sort=True):

    g_enr = g["gaussian_enrichment"]
    b_enr = g["black_enrichment"]

    rows.append({
        "rho": rho,

        "n": len(g),

        "gaussian_mean": g_enr.mean(),
        "gaussian_sd": g_enr.std(ddof=1),
        "gaussian_median": g_enr.median(),

        "black_mean": b_enr.mean(),
        "black_sd": b_enr.std(ddof=1),
        "black_median": b_enr.median(),

        "mean_difference": (g_enr - b_enr).mean(),

        "mean_ratio_black_gaussian":
            (b_enr / g_enr).replace([np.inf, -np.inf], np.nan).mean(),

        "gaussian_failures":
            int(g["gaussian_completeness_failed"].sum()),

        "black_failures":
            int(g["black_completeness_failed"].sum()),
    })

rho_table = pd.DataFrame(rows)

print("\n" + "=" * 72)
print("RHO-STRATIFIED ENRICHMENT")
print("=" * 72)

display(
    rho_table.style.format({
        "rho": "{:.1f}",
        "gaussian_mean": "{:.4f}",
        "gaussian_sd": "{:.4f}",
        "gaussian_median": "{:.4f}",
        "black_mean": "{:.4f}",
        "black_sd": "{:.4f}",
        "black_median": "{:.4f}",
        "mean_difference": "{:.4f}",
        "mean_ratio_black_gaussian": "{:.4f}",
    })
)

# ------------------------------------------------------------
# 3. Per-model means
# ------------------------------------------------------------

model_table = (
    df.groupby(["rho", "model", "replicate"], as_index=False)
      .agg(
          gaussian_mean=("gaussian_enrichment", "mean"),
          black_mean=("black_enrichment", "mean"),
          gaussian_failure_count=("gaussian_completeness_failed", "sum"),
          black_failure_count=("black_completeness_failed", "sum"),
          n_images=("image", "size"),
      )
)

model_table["difference"] = (
    model_table["gaussian_mean"] -
    model_table["black_mean"]
)

model_table["black_to_gaussian_ratio"] = (
    model_table["black_mean"] /
    model_table["gaussian_mean"]
)

print("\n" + "=" * 72)
print("PER-MODEL ENRICHMENT")
print("=" * 72)

display(
    model_table.style.format({
        "rho": "{:.1f}",
        "gaussian_mean": "{:.4f}",
        "black_mean": "{:.4f}",
        "difference": "{:.4f}",
        "black_to_gaussian_ratio": "{:.4f}",
    })
)

print("\nPASS — descriptive baseline × rho analysis completed.")

T1.4c — GAUSSIAN vs BLACK BASELINE

Overall enrichment:
Gaussian mean   = 9.808741
Gaussian median = 8.794926
Black mean      = 0.215778
Black median    = 0.157487

Overall paired difference:
Mean difference   = 9.592963
Median difference = 8.551841
SD difference     = 5.158844

Paired direction:
Gaussian > Black: 450 / 450
Gaussian < Black: 0 / 450
Gaussian = Black: 0 / 450

RHO-STRATIFIED ENRICHMENT


,rho,n,gaussian_mean,gaussian_sd,gaussian_median,black_mean,black_sd,black_median,mean_difference,mean_ratio_black_gaussian,gaussian_failures,black_failures
0,0.0,75,5.7889,2.4146,5.9210,0.0927,0.0514,0.0820,5.6962,0.0252,33,18
1,0.2,75,7.9722,3.5322,8.0092,0.1507,0.0932,0.1201,7.8215,0.0270,22,32
2,0.4,75,8.6620,3.8901,8.5666,0.1964,0.1264,0.1699,8.4656,0.0310,24,35
3,0.6,75,10.1004,4.4015,9.5173,0.2057,0.1445,0.1730,9.8947,0.0272,13,27
4,0.8,75,11.7698,5.0795,10.7905,0.2684,0.1903,0.2045,11.5015,0.0307,6,28
5,1.0,75,14.5592,6.0801,14.4717,0.3808,0.2951,0.2641,14.1783,0.0341,7,22



PER-MODEL ENRICHMENT


,rho,model,replicate,gaussian_mean,black_mean,gaussian_failure_count,black_failure_count,n_images,difference,black_to_gaussian_ratio
0,0.0,rho_0.0_rep_1,1,5.4328,0.0820,7,2,15,5.3508,0.0151
1,0.0,rho_0.0_rep_2,2,5.4392,0.0885,10,3,15,5.3508,0.0163
2,0.0,rho_0.0_rep_3,3,5.9954,0.1031,6,4,15,5.8922,0.0172
3,0.0,rho_0.0_rep_4,4,6.2955,0.0938,8,5,15,6.2017,0.0149
4,0.0,rho_0.0_rep_5,5,5.7816,0.0961,2,4,15,5.6855,0.0166
5,0.2,rho_0.2_rep_1,1,7.3392,0.1417,7,9,15,7.1975,0.0193
6,0.2,rho_0.2_rep_2,2,7.7890,0.1429,2,5,15,7.6461,0.0183
7,0.2,rho_0.2_rep_3,3,7.8751,0.1641,4,5,15,7.7110,0.0208
8,0.2,rho_0.2_rep_4,4,8.8231,0.1676,4,4,15,8.6555,0.0190
9,0.2,rho_0.2_rep_5,5,8.0348,0.1374,5,9,15,7.8974,0.0171



PASS — descriptive baseline × rho analysis completed.


In [29]:
# ============================================================
# CELL 18C — T1.4c MODEL-LEVEL DOSE-RESPONSE INFERENCE
#
# Experimental unit:
#   trained model (30 total)
#
# Six rho levels × five independent model replicates
#
# Primary descriptive/inferential quantities:
#   - Spearman rho
#   - permutation p-value
#   - linear slope
#   - bootstrap 95% CI
#
# No Holm correction:
# T1.4c is a sensitivity analysis, not the locked primary
# H1/H2 confirmatory family.
# ============================================================

import numpy as np
import pandas as pd
from scipy.stats import spearmanr

# ------------------------------------------------------------
# 1. Construct one observation per trained model
# ------------------------------------------------------------

model_df = (
    df.groupby(["model", "rho", "replicate"], as_index=False)
      .agg(
          gaussian_enrichment=("gaussian_enrichment", "mean"),
          black_enrichment=("black_enrichment", "mean"),
          gaussian_failures=("gaussian_completeness_failed", "sum"),
          black_failures=("black_completeness_failed", "sum"),
          n_images=("image", "size"),
      )
)

model_df["paired_difference"] = (
    model_df["gaussian_enrichment"] -
    model_df["black_enrichment"]
)

assert len(model_df) == 30
assert model_df["n_images"].eq(15).all()

# ------------------------------------------------------------
# 2. Spearman helper
# ------------------------------------------------------------

def spearman_permutation_test(
    x,
    y,
    n_perm=20000,
    seed=20260925,
):
    """
    Permutation test for association between fixed x and y.

    x is the observed rho assignment.
    y is the model-level endpoint.

    Permutation preserves the endpoint values and randomly
    reassigns the rho labels across the 30 model observations.
    """

    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    observed_r, _ = spearmanr(x, y)

    rng = np.random.default_rng(seed)

    perm_r = np.empty(n_perm, dtype=float)

    for i in range(n_perm):
        perm_x = rng.permutation(x)
        perm_r[i], _ = spearmanr(perm_x, y)

    # One-sided test for positive dose-response
    p_one_sided = (np.sum(perm_r >= observed_r) + 1) / (n_perm + 1)

    # Two-sided sensitivity value
    p_two_sided = (
        np.sum(np.abs(perm_r) >= abs(observed_r)) + 1
    ) / (n_perm + 1)

    return {
        "r_s": float(observed_r),
        "p_one_sided": float(p_one_sided),
        "p_two_sided": float(p_two_sided),
        "null_mean": float(np.mean(perm_r)),
        "null_sd": float(np.std(perm_r, ddof=1)),
        "null_min": float(np.min(perm_r)),
        "null_max": float(np.max(perm_r)),
    }

# ------------------------------------------------------------
# 3. Linear slope helper
# ------------------------------------------------------------

def linear_slope(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    slope, intercept = np.polyfit(x, y, 1)

    return float(slope), float(intercept)

# ------------------------------------------------------------
# 4. Bootstrap slope CI
# ------------------------------------------------------------

def bootstrap_slope_ci(
    x,
    y,
    n_boot=20000,
    seed=20260925,
):
    """
    Cluster-aware bootstrap:
    resample trained models, not individual image rows.
    """

    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    rng = np.random.default_rng(seed)

    n = len(x)
    slopes = np.empty(n_boot, dtype=float)

    for i in range(n_boot):
        idx = rng.integers(0, n, size=n)
        slopes[i] = np.polyfit(x[idx], y[idx], 1)[0]

    lo, hi = np.percentile(slopes, [2.5, 97.5])

    return {
        "slope": float(np.polyfit(x, y, 1)[0]),
        "ci_low": float(lo),
        "ci_high": float(hi),
    }

# ------------------------------------------------------------
# 5. Analyze Gaussian, black, and paired difference
# ------------------------------------------------------------

results = []

endpoints = {
    "Gaussian enrichment": model_df["gaussian_enrichment"].to_numpy(),
    "Black enrichment": model_df["black_enrichment"].to_numpy(),
    "Gaussian minus black": model_df["paired_difference"].to_numpy(),
}

x = model_df["rho"].to_numpy()

for endpoint_name, y in endpoints.items():

    perm = spearman_permutation_test(
        x,
        y,
        n_perm=20000,
        seed=20260925,
    )

    slope = bootstrap_slope_ci(
        x,
        y,
        n_boot=20000,
        seed=20260926,
    )

    results.append({
        "endpoint": endpoint_name,
        "n_models": len(y),
        "spearman_r": perm["r_s"],
        "permutation_p_one_sided": perm["p_one_sided"],
        "permutation_p_two_sided": perm["p_two_sided"],
        "linear_slope": slope["slope"],
        "slope_ci_low": slope["ci_low"],
        "slope_ci_high": slope["ci_high"],
        "null_mean": perm["null_mean"],
        "null_sd": perm["null_sd"],
        "null_min": perm["null_min"],
        "null_max": perm["null_max"],
    })

dose_response_results = pd.DataFrame(results)

# ------------------------------------------------------------
# 6. Display
# ------------------------------------------------------------

print("=" * 72)
print("T1.4c — MODEL-LEVEL DOSE-RESPONSE INFERENCE")
print("=" * 72)

display(
    dose_response_results.style.format({
        "spearman_r": "{:.6f}",
        "permutation_p_one_sided": "{:.8g}",
        "permutation_p_two_sided": "{:.8g}",
        "linear_slope": "{:.6f}",
        "slope_ci_low": "{:.6f}",
        "slope_ci_high": "{:.6f}",
        "null_mean": "{:.6f}",
        "null_sd": "{:.6f}",
        "null_min": "{:.6f}",
        "null_max": "{:.6f}",
    })
)

print("\nModel-level data:")
display(
    model_df[
        [
            "rho",
            "model",
            "replicate",
            "gaussian_enrichment",
            "black_enrichment",
            "paired_difference",
        ]
    ].sort_values(["rho", "replicate"])
)

print("\nPASS — model-level dose-response inference completed.")

T1.4c — MODEL-LEVEL DOSE-RESPONSE INFERENCE


,endpoint,n_models,spearman_r,permutation_p_one_sided,permutation_p_two_sided,linear_slope,slope_ci_low,slope_ci_high,null_mean,null_sd,null_min,null_max
0,Gaussian enrichment,30,0.977542,4.99975e-05,4.99975e-05,8.097502,7.232369,8.804265,0.000383,0.187506,-0.679882,0.650567
1,Black enrichment,30,0.961757,4.99975e-05,4.99975e-05,0.257555,0.204590,0.306434,0.000632,0.187723,-0.639292,0.704687
2,Gaussian minus black,30,0.977542,4.99975e-05,4.99975e-05,7.839947,6.990993,8.529182,0.000406,0.187600,-0.675372,0.641547



Model-level data:


,rho,model,replicate,gaussian_enrichment,black_enrichment,paired_difference
0,0.0,rho_0.0_rep_1,1,5.432823,0.082040,5.350783
1,0.0,rho_0.0_rep_2,2,5.439241,0.088464,5.350777
2,0.0,rho_0.0_rep_3,3,5.995367,0.103125,5.892241
3,0.0,rho_0.0_rep_4,4,6.295477,0.093750,6.201727
4,0.0,rho_0.0_rep_5,5,5.781602,0.096084,5.685518
5,0.2,rho_0.2_rep_1,1,7.339236,0.141723,7.197514
6,0.2,rho_0.2_rep_2,2,7.788982,0.142868,7.646114
7,0.2,rho_0.2_rep_3,3,7.875116,0.164078,7.711038
8,0.2,rho_0.2_rep_4,4,8.823067,0.167564,8.655503
9,0.2,rho_0.2_rep_5,5,8.034779,0.137422,7.897358



PASS — model-level dose-response inference completed.


In [30]:
# ============================================================
# CELL 18D — FORMAL GAUSSIAN vs BLACK SLOPE COMPARISON
#
# Paired model-level analysis.
#
# Each trained model has:
#   Gaussian enrichment
#   Black enrichment
#
# Therefore Gaussian and black are NOT independent samples.
#
# Primary comparison:
#   slope(Gaussian) - slope(Black)
#
# Inference:
#   1. Paired model bootstrap CI
#   2. Within-model label-swap permutation test
#
# ============================================================

import numpy as np
import pandas as pd

x = model_df["rho"].to_numpy(dtype=float)
g = model_df["gaussian_enrichment"].to_numpy(dtype=float)
b = model_df["black_enrichment"].to_numpy(dtype=float)

# ------------------------------------------------------------
# 1. Observed slopes
# ------------------------------------------------------------

slope_g = np.polyfit(x, g, 1)[0]
slope_b = np.polyfit(x, b, 1)[0]

observed_slope_difference = slope_g - slope_b

print("=" * 72)
print("T1.4c — FORMAL BASELINE SLOPE COMPARISON")
print("=" * 72)

print(f"Gaussian slope = {slope_g:.8f}")
print(f"Black slope    = {slope_b:.8f}")
print(f"Slope difference (G - B) = {observed_slope_difference:.8f}")

print(
    f"\nBlack/Gaussian slope ratio = "
    f"{slope_b / slope_g:.6f}"
)

print(
    f"Gaussian/Black slope ratio = "
    f"{slope_g / slope_b:.6f}"
)

# ------------------------------------------------------------
# 2. Paired bootstrap
# ------------------------------------------------------------

N_BOOT = 20000
BOOT_SEED = 20260927

rng = np.random.default_rng(BOOT_SEED)

n = len(x)
boot_diff = np.empty(N_BOOT, dtype=float)

for i in range(N_BOOT):

    idx = rng.integers(0, n, size=n)

    boot_x = x[idx]
    boot_g = g[idx]
    boot_b = b[idx]

    bg = np.polyfit(boot_x, boot_g, 1)[0]
    bb = np.polyfit(boot_x, boot_b, 1)[0]

    boot_diff[i] = bg - bb

boot_ci_low, boot_ci_high = np.percentile(
    boot_diff,
    [2.5, 97.5]
)

print("\nPaired bootstrap:")
print(f"  B = {N_BOOT}")
print(f"  95% CI = [{boot_ci_low:.8f}, {boot_ci_high:.8f}]")

# ------------------------------------------------------------
# 3. Within-model label-swap permutation test
# ------------------------------------------------------------
#
# Under the null that Gaussian and black have the same
# dose-response relationship, the baseline labels are
# exchangeable within each paired model.
#
# For every permutation:
#   randomly swap G/B within each model
#   calculate slope difference
#
# This preserves:
#   - rho
#   - model identity
#   - pairing
#   - marginal values within each pair
#
# ------------------------------------------------------------

N_PERM = 20000
PERM_SEED = 20260928

rng = np.random.default_rng(PERM_SEED)

perm_diff = np.empty(N_PERM, dtype=float)

for i in range(N_PERM):

    swap = rng.random(n) < 0.5

    perm_g = np.where(swap, b, g)
    perm_b = np.where(swap, g, b)

    pg = np.polyfit(x, perm_g, 1)[0]
    pb = np.polyfit(x, perm_b, 1)[0]

    perm_diff[i] = pg - pb

# Two-sided permutation p-value
p_two_sided = (
    np.sum(np.abs(perm_diff) >= abs(observed_slope_difference))
    + 1
) / (N_PERM + 1)

# One-sided p-value for Gaussian slope > black slope
p_one_sided = (
    np.sum(perm_diff >= observed_slope_difference)
    + 1
) / (N_PERM + 1)

# ------------------------------------------------------------
# 4. Permutation distribution diagnostics
# ------------------------------------------------------------

print("\nWithin-model label-swap permutation:")
print(f"  B = {N_PERM}")
print(f"  Null mean = {perm_diff.mean():.8f}")
print(f"  Null SD   = {perm_diff.std(ddof=1):.8f}")
print(f"  Null min  = {perm_diff.min():.8f}")
print(f"  Null max  = {perm_diff.max():.8f}")

print(f"\n  One-sided p = {p_one_sided:.8g}")
print(f"  Two-sided p = {p_two_sided:.8g}")

# ------------------------------------------------------------
# 5. Bootstrap probability of positive slope difference
# ------------------------------------------------------------

prob_positive = np.mean(boot_diff > 0)

print("\nBootstrap:")
print(
    f"  P(bootstrap slope difference > 0) = "
    f"{prob_positive:.6f}"
)

# ------------------------------------------------------------
# 6. Final compact result
# ------------------------------------------------------------

comparison_table = pd.DataFrame([{
    "gaussian_slope": slope_g,
    "black_slope": slope_b,
    "slope_difference": observed_slope_difference,
    "black_to_gaussian_slope_ratio": slope_b / slope_g,
    "gaussian_to_black_slope_ratio": slope_g / slope_b,
    "bootstrap_ci_low": boot_ci_low,
    "bootstrap_ci_high": boot_ci_high,
    "permutation_p_one_sided": p_one_sided,
    "permutation_p_two_sided": p_two_sided,
    "bootstrap_probability_difference_positive": prob_positive,
}])

print("\n" + "=" * 72)
print("FINAL SLOPE COMPARISON")
print("=" * 72)

display(
    comparison_table.style.format({
        "gaussian_slope": "{:.6f}",
        "black_slope": "{:.6f}",
        "slope_difference": "{:.6f}",
        "black_to_gaussian_slope_ratio": "{:.6f}",
        "gaussian_to_black_slope_ratio": "{:.3f}",
        "bootstrap_ci_low": "{:.6f}",
        "bootstrap_ci_high": "{:.6f}",
        "permutation_p_one_sided": "{:.8g}",
        "permutation_p_two_sided": "{:.8g}",
        "bootstrap_probability_difference_positive": "{:.6f}",
    })
)

print("\nPASS — formal paired baseline slope comparison completed.")

T1.4c — FORMAL BASELINE SLOPE COMPARISON
Gaussian slope = 8.09750165
Black slope    = 0.25755507
Slope difference (G - B) = 7.83994659

Black/Gaussian slope ratio = 0.031807
Gaussian/Black slope ratio = 31.439885

Paired bootstrap:
  B = 20000
  95% CI = [6.98990662, 8.53014786]

Within-model label-swap permutation:
  B = 20000
  Null mean = -0.00968343
  Null SD   = 5.65293032
  Null min  = -19.33618109
  Null max  = 21.61308345

  One-sided p = 0.08679566
  Two-sided p = 0.1700915

Bootstrap:
  P(bootstrap slope difference > 0) = 1.000000

FINAL SLOPE COMPARISON


,gaussian_slope,black_slope,slope_difference,black_to_gaussian_slope_ratio,gaussian_to_black_slope_ratio,bootstrap_ci_low,bootstrap_ci_high,permutation_p_one_sided,permutation_p_two_sided,bootstrap_probability_difference_positive
0,8.097502,0.257555,7.839947,0.031807,31.440,6.989907,8.530148,0.08679566,0.1700915,1.000000



PASS — formal paired baseline slope comparison completed.


In [31]:
# ============================================================
# CELL 18E — T1.4c CORRECTED PAIRED SLOPE-DIFFERENCE TEST
#
# Primary paired quantity:
#     D_i = Gaussian_i - Black_i
#
# Question:
#     Is the dose-response slope of D_i > 0?
#
# Null:
#     No systematic baseline-dependent difference in the
#     dose-response slope.
#
# Permutation:
#     Within-model sign flipping of D_i.
#
# This preserves the paired Gaussian/black structure and does
# not require treating Gaussian and black as independent.
# ============================================================

import numpy as np
import pandas as pd

x = model_df["rho"].to_numpy(dtype=float)

g = model_df["gaussian_enrichment"].to_numpy(dtype=float)
b = model_df["black_enrichment"].to_numpy(dtype=float)

d = g - b

# ------------------------------------------------------------
# 1. Observed slope of paired difference
# ------------------------------------------------------------

observed_difference_slope = np.polyfit(x, d, 1)[0]

print("=" * 72)
print("T1.4c — PAIRED DIFFERENCE SLOPE TEST")
print("=" * 72)

print(f"Observed slope of (Gaussian - Black): "
      f"{observed_difference_slope:.8f}")

# ------------------------------------------------------------
# 2. Sign-flip permutation
# ------------------------------------------------------------

N_PERM = 20000
SEED = 20260929

rng = np.random.default_rng(SEED)

n = len(d)

perm_slopes = np.empty(N_PERM, dtype=float)

for i in range(N_PERM):

    signs = rng.choice(
        np.array([-1.0, 1.0]),
        size=n
    )

    perm_d = d * signs

    perm_slopes[i] = np.polyfit(
        x,
        perm_d,
        1
    )[0]

# ------------------------------------------------------------
# 3. One- and two-sided p-values
# ------------------------------------------------------------

p_one = (
    np.sum(perm_slopes >= observed_difference_slope) + 1
) / (N_PERM + 1)

p_two = (
    np.sum(
        np.abs(perm_slopes) >=
        abs(observed_difference_slope)
    ) + 1
) / (N_PERM + 1)

# ------------------------------------------------------------
# 4. Null diagnostics
# ------------------------------------------------------------

print("\nSign-flip permutation:")
print(f"  B = {N_PERM}")
print(f"  Null mean = {perm_slopes.mean():.8f}")
print(f"  Null SD   = {perm_slopes.std(ddof=1):.8f}")
print(f"  Null min  = {perm_slopes.min():.8f}")
print(f"  Null max  = {perm_slopes.max():.8f}")

print("\nP-values:")
print(f"  One-sided p = {p_one:.8g}")
print(f"  Two-sided p = {p_two:.8g}")

# ------------------------------------------------------------
# 5. Relationship between paired difference and rho
# ------------------------------------------------------------

from scipy.stats import spearmanr

rs, _ = spearmanr(x, d)

print(f"\nSpearman rho for paired difference = {rs:.8f}")

# ------------------------------------------------------------
# 6. Bootstrap distribution already generated in 18D can be
#    regenerated here independently for a clean final record.
# ------------------------------------------------------------

N_BOOT = 20000
BOOT_SEED = 20260930

rng = np.random.default_rng(BOOT_SEED)

boot_slopes = np.empty(N_BOOT, dtype=float)

for i in range(N_BOOT):

    idx = rng.integers(0, n, size=n)

    boot_slopes[i] = np.polyfit(
        x[idx],
        d[idx],
        1
    )[0]

ci_low, ci_high = np.percentile(
    boot_slopes,
    [2.5, 97.5]
)

print("\nPaired bootstrap:")
print(f"  B = {N_BOOT}")
print(f"  95% CI = [{ci_low:.8f}, {ci_high:.8f}]")
print(f"  P(slope > 0) = {np.mean(boot_slopes > 0):.6f}")

# ------------------------------------------------------------
# 7. Compact final record
# ------------------------------------------------------------

paired_slope_test = pd.DataFrame([{
    "n_models": n,
    "observed_difference_slope": observed_difference_slope,
    "spearman_r_difference": rs,
    "signflip_p_one_sided": p_one,
    "signflip_p_two_sided": p_two,
    "bootstrap_ci_low": ci_low,
    "bootstrap_ci_high": ci_high,
    "bootstrap_probability_positive": np.mean(
        boot_slopes > 0
    ),
}])

print("\n" + "=" * 72)
print("FINAL PAIRED DIFFERENCE RESULT")
print("=" * 72)

display(
    paired_slope_test.style.format({
        "observed_difference_slope": "{:.8f}",
        "spearman_r_difference": "{:.8f}",
        "signflip_p_one_sided": "{:.8g}",
        "signflip_p_two_sided": "{:.8g}",
        "bootstrap_ci_low": "{:.8f}",
        "bootstrap_ci_high": "{:.8f}",
        "bootstrap_probability_positive": "{:.6f}",
    })
)

print("\nPASS — corrected paired slope-difference analysis completed.")

T1.4c — PAIRED DIFFERENCE SLOPE TEST
Observed slope of (Gaussian - Black): 7.83994659

Sign-flip permutation:
  B = 20000
  Null mean = -0.00286573
  Null SD   = 5.61299622
  Null min  = -19.30548276
  Null max  = 19.96915495

P-values:
  One-sided p = 0.082645868
  Two-sided p = 0.16734163

Spearman rho for paired difference = 0.97754155

Paired bootstrap:
  B = 20000
  95% CI = [6.99794437, 8.53522104]
  P(slope > 0) = 1.000000

FINAL PAIRED DIFFERENCE RESULT


,n_models,observed_difference_slope,spearman_r_difference,signflip_p_one_sided,signflip_p_two_sided,bootstrap_ci_low,bootstrap_ci_high,bootstrap_probability_positive
0,30,7.83994659,0.97754155,0.082645868,0.16734163,6.99794437,8.53522104,1.000000



PASS — corrected paired slope-difference analysis completed.


In [32]:
# ============================================================
# CELL 18F — T1.4c COMPLETENESS-FAILURE SENSITIVITY ANALYSIS
#
# SECONDARY ONLY
#
# Primary analysis (18C) used all finite observations.
# Here we exclude observations that failed the relevant
# baseline's IG completeness criterion.
#
# Gaussian sensitivity:
#     exclude gaussian_completeness_failed == True
#
# Black sensitivity:
#     exclude black_completeness_failed == True
#
# Paired difference sensitivity:
#     retain only image/model observations where BOTH
#     baselines passed completeness.
#
# Model-level aggregation is retained.
# ============================================================

import numpy as np
import pandas as pd
from scipy.stats import spearmanr

print("=" * 72)
print("T1.4c — COMPLETENESS-FAILURE SENSITIVITY ANALYSIS")
print("=" * 72)

# ------------------------------------------------------------
# Helper: model-level dose-response
# ------------------------------------------------------------

def sensitivity_model_analysis(
    data,
    enrichment_col,
    label,
    seed=20260931,
):
    """
    Aggregate image-level observations to model level,
    then calculate Spearman and linear slope.

    Important:
    model remains the experimental unit.
    """

    m = (
        data.groupby(
            ["model", "rho", "replicate"],
            as_index=False
        )
        .agg(
            enrichment=(enrichment_col, "mean"),
            n_images=("image", "size"),
        )
    )

    # Only models with at least one retained image
    m = m[m["n_images"] > 0].copy()

    x = m["rho"].to_numpy(dtype=float)
    y = m["enrichment"].to_numpy(dtype=float)

    rs, _ = spearmanr(x, y)
    slope = np.polyfit(x, y, 1)[0]

    # Bootstrap model-level slope
    rng = np.random.default_rng(seed)
    n = len(m)

    boot = np.empty(20000, dtype=float)

    for i in range(20000):
        idx = rng.integers(0, n, size=n)
        boot[i] = np.polyfit(
            x[idx],
            y[idx],
            1
        )[0]

    ci_low, ci_high = np.percentile(
        boot,
        [2.5, 97.5]
    )

    return {
        "baseline": label,
        "n_image_rows": len(data),
        "n_models": len(m),
        "min_images_per_model": int(m["n_images"].min()),
        "max_images_per_model": int(m["n_images"].max()),
        "spearman_r": float(rs),
        "linear_slope": float(slope),
        "bootstrap_ci_low": float(ci_low),
        "bootstrap_ci_high": float(ci_high),
        "bootstrap_probability_positive": float(
            np.mean(boot > 0)
        ),
        "model_table": m,
    }


# ------------------------------------------------------------
# 1. Gaussian completeness-pass subset
# ------------------------------------------------------------

gaussian_pass = df[
    ~df["gaussian_completeness_failed"]
].copy()

# ------------------------------------------------------------
# 2. Black completeness-pass subset
# ------------------------------------------------------------

black_pass = df[
    ~df["black_completeness_failed"]
].copy()

# ------------------------------------------------------------
# 3. Both-baselines-pass paired subset
# ------------------------------------------------------------

both_pass = df[
    (~df["gaussian_completeness_failed"]) &
    (~df["black_completeness_failed"])
].copy()

print("\nRetained observations:")
print(
    f"Gaussian pass: "
    f"{len(gaussian_pass)} / {len(df)} "
    f"({100*len(gaussian_pass)/len(df):.2f}%)"
)

print(
    f"Black pass:    "
    f"{len(black_pass)} / {len(df)} "
    f"({100*len(black_pass)/len(df):.2f}%)"
)

print(
    f"Both pass:     "
    f"{len(both_pass)} / {len(df)} "
    f"({100*len(both_pass)/len(df):.2f}%)"
)

# ------------------------------------------------------------
# 4. Run Gaussian sensitivity
# ------------------------------------------------------------

g_result = sensitivity_model_analysis(
    gaussian_pass,
    "gaussian_enrichment",
    "Gaussian",
    seed=20260931,
)

# ------------------------------------------------------------
# 5. Run black sensitivity
# ------------------------------------------------------------

b_result = sensitivity_model_analysis(
    black_pass,
    "black_enrichment",
    "Black",
    seed=20260932,
)

# ------------------------------------------------------------
# 6. Paired difference sensitivity
# ------------------------------------------------------------

both_pass = both_pass.copy()

both_pass["paired_difference"] = (
    both_pass["gaussian_enrichment"] -
    both_pass["black_enrichment"]
)

d_result = sensitivity_model_analysis(
    both_pass,
    "paired_difference",
    "Gaussian − Black",
    seed=20260933,
)

# ------------------------------------------------------------
# 7. Summary table
# ------------------------------------------------------------

summary = pd.DataFrame([
    {
        k: v
        for k, v in g_result.items()
        if k != "model_table"
    },
    {
        k: v
        for k, v in b_result.items()
        if k != "model_table"
    },
    {
        k: v
        for k, v in d_result.items()
        if k != "model_table"
    },
])

print("\n" + "=" * 72)
print("COMPLETENESS-PASS SENSITIVITY RESULTS")
print("=" * 72)

display(
    summary.style.format({
        "spearman_r": "{:.6f}",
        "linear_slope": "{:.6f}",
        "bootstrap_ci_low": "{:.6f}",
        "bootstrap_ci_high": "{:.6f}",
        "bootstrap_probability_positive": "{:.6f}",
    })
)

# ------------------------------------------------------------
# 8. Rho-specific retention
# ------------------------------------------------------------

print("\n" + "=" * 72)
print("RETENTION BY RHO")
print("=" * 72)

retention = pd.DataFrame({
    "rho": sorted(df["rho"].unique()),
    "total": [
        int((df["rho"] == r).sum())
        for r in sorted(df["rho"].unique())
    ],
    "gaussian_pass": [
        int(
            ((df["rho"] == r) &
             (~df["gaussian_completeness_failed"])).sum()
        )
        for r in sorted(df["rho"].unique())
    ],
    "black_pass": [
        int(
            ((df["rho"] == r) &
             (~df["black_completeness_failed"])).sum()
        )
        for r in sorted(df["rho"].unique())
    ],
    "both_pass": [
        int(
            ((df["rho"] == r) &
             (~df["gaussian_completeness_failed"]) &
             (~df["black_completeness_failed"])).sum()
        )
        for r in sorted(df["rho"].unique())
    ],
})

display(retention)

print("\nPASS — completeness-failure sensitivity analysis completed.")
print("These results are SECONDARY and do not replace the all-observation analysis.")

T1.4c — COMPLETENESS-FAILURE SENSITIVITY ANALYSIS

Retained observations:
Gaussian pass: 345 / 450 (76.67%)
Black pass:    288 / 450 (64.00%)
Both pass:     240 / 450 (53.33%)

COMPLETENESS-PASS SENSITIVITY RESULTS


,baseline,n_image_rows,n_models,min_images_per_model,max_images_per_model,spearman_r,linear_slope,bootstrap_ci_low,bootstrap_ci_high,bootstrap_probability_positive
0,Gaussian,345,30,5,14,0.945972,8.095152,7.146013,8.961318,1.000000
1,Black,288,30,6,13,0.950482,0.330318,0.246080,0.411556,1.000000
2,Gaussian − Black,240,30,5,12,0.957247,7.866020,6.521254,8.946100,1.000000



RETENTION BY RHO


,rho,total,gaussian_pass,black_pass,both_pass
0,0.0,75,42,57,36
1,0.2,75,53,43,36
2,0.4,75,51,40,30
3,0.6,75,62,48,43
4,0.8,75,69,47,45
5,1.0,75,68,53,50



PASS — completeness-failure sensitivity analysis completed.
These results are SECONDARY and do not replace the all-observation analysis.


In [33]:
# ============================================================
# CELL 18G — T1.4c FINAL EVIDENCE PACKAGE / FREEZE
# ============================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd

FINAL_DIR = Path("/kaggle/working/T1_4c_FINAL_EVIDENCE")
FINAL_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 1. Primary model-level results from Cell 18C
# ------------------------------------------------------------

primary_results = dose_response_results.copy()

# ------------------------------------------------------------
# 2. Formal paired slope comparison from Cell 18E
# ------------------------------------------------------------

paired_result = paired_slope_test.copy()

# ------------------------------------------------------------
# 3. Completeness sensitivity from Cell 18F
# ------------------------------------------------------------

sensitivity_results = summary.copy()
retention_results = retention.copy()

# ------------------------------------------------------------
# 4. Production QA summary
# ------------------------------------------------------------

qa_summary = pd.DataFrame([{
    "production_rows": len(df),
    "models": df["model"].nunique(),
    "rho_levels": df["rho"].nunique(),
    "images": df["image"].nunique(),

    "gaussian_undefined": int(
        df["gaussian_undefined"].sum()
    ),
    "black_undefined": int(
        df["black_undefined"].sum()
    ),

    "gaussian_completeness_failures": int(
        df["gaussian_completeness_failed"].sum()
    ),
    "black_completeness_failures": int(
        df["black_completeness_failed"].sum()
    ),

    "gaussian_pass_rows": int(
        (~df["gaussian_completeness_failed"]).sum()
    ),
    "black_pass_rows": int(
        (~df["black_completeness_failed"]).sum()
    ),
    "both_pass_rows": int(
        (
            (~df["gaussian_completeness_failed"]) &
            (~df["black_completeness_failed"])
        ).sum()
    ),

    "gaussian_mean_enrichment": float(
        df["gaussian_enrichment"].mean()
    ),
    "black_mean_enrichment": float(
        df["black_enrichment"].mean()
    ),
    "mean_absolute_difference": float(
        df["abs_enrichment_difference"].mean()
    ),
    "mean_relative_difference_pct": float(
        df["relative_enrichment_difference_pct"].mean()
    ),
}])

# ------------------------------------------------------------
# 5. Save CSV artifacts
# ------------------------------------------------------------

primary_path = FINAL_DIR / "T1_4c_primary_model_dose_response.csv"
paired_path = FINAL_DIR / "T1_4c_paired_slope_comparison.csv"
sensitivity_path = FINAL_DIR / "T1_4c_completeness_sensitivity.csv"
retention_path = FINAL_DIR / "T1_4c_completeness_retention_by_rho.csv"
qa_path = FINAL_DIR / "T1_4c_production_QA_summary.csv"

primary_results.to_csv(primary_path, index=False)
paired_result.to_csv(paired_path, index=False)
sensitivity_results.to_csv(sensitivity_path, index=False)
retention_results.to_csv(retention_path, index=False)
qa_summary.to_csv(qa_path, index=False)

# ------------------------------------------------------------
# 6. Final interpretation record
# ------------------------------------------------------------

final_record = {
    "protocol": "T1.4c_black_baseline_sensitivity_v1",

    "production_design": {
        "models": 30,
        "rho_levels": [0.0, 0.2, 0.4, 0.6, 0.8, 1.0],
        "replicates_per_rho": 5,
        "images_per_model": 15,
        "model_image_pairs": 450,
        "ig_evaluations": 900,
    },

    "production_QA": {
        "gaussian_undefined": 0,
        "black_undefined": 0,
        "gaussian_completeness_failures": 105,
        "black_completeness_failures": 162,
    },

    "primary_model_level": {
        "gaussian_spearman_r": float(
            primary_results.loc[
                primary_results["endpoint"] ==
                "Gaussian enrichment",
                "spearman_r"
            ].iloc[0]
        ),
        "black_spearman_r": float(
            primary_results.loc[
                primary_results["endpoint"] ==
                "Black enrichment",
                "spearman_r"
            ].iloc[0]
        ),
        "gaussian_slope": float(
            primary_results.loc[
                primary_results["endpoint"] ==
                "Gaussian enrichment",
                "linear_slope"
            ].iloc[0]
        ),
        "black_slope": float(
            primary_results.loc[
                primary_results["endpoint"] ==
                "Black enrichment",
                "linear_slope"
            ].iloc[0]
        ),
    },

    "paired_slope_comparison": {
        "slope_difference": float(
            paired_result["observed_difference_slope"].iloc[0]
        ),
        "bootstrap_ci_low": float(
            paired_result["bootstrap_ci_low"].iloc[0]
        ),
        "bootstrap_ci_high": float(
            paired_result["bootstrap_ci_high"].iloc[0]
        ),
        "signflip_p_one_sided": float(
            paired_result["signflip_p_one_sided"].iloc[0]
        ),
        "signflip_p_two_sided": float(
            paired_result["signflip_p_two_sided"].iloc[0]
        ),
    },

    "sensitivity_analysis": {
        "gaussian_slope": float(
            sensitivity_results.loc[
                sensitivity_results["baseline"] == "Gaussian",
                "linear_slope"
            ].iloc[0]
        ),
        "black_slope": float(
            sensitivity_results.loc[
                sensitivity_results["baseline"] == "Black",
                "linear_slope"
            ].iloc[0]
        ),
        "difference_slope": float(
            sensitivity_results.loc[
                sensitivity_results["baseline"] ==
                "Gaussian − Black",
                "linear_slope"
            ].iloc[0]
        ),
        "both_pass_rows": int(
            qa_summary["both_pass_rows"].iloc[0]
        ),
    },

    "interpretation_status": [
        "Gaussian enrichment shows a strong positive dose-response.",
        "Black-baseline enrichment also shows a positive dose-response.",
        "Black baseline produces a much smaller enrichment magnitude.",
        "The Gaussian-minus-black slope difference has a positive bootstrap interval.",
        "The paired sign-flip permutation test does not reach conventional alpha=0.05 significance.",
        "Completeness-pass sensitivity analysis preserves the positive dose-response patterns.",
        "T1.4c does not establish baseline-independent XAI localization.",
        "T1.4c does not by itself establish hair-specific spatial localization."
    ],

    "analysis_status": "FROZEN_AFTER_18F",
}

json_path = FINAL_DIR / "T1_4c_FINAL_ANALYSIS_RECORD.json"

with open(json_path, "w") as f:
    json.dump(final_record, f, indent=2)

# ------------------------------------------------------------
# 7. Print final freeze summary
# ------------------------------------------------------------

print("=" * 72)
print("T1.4c — FINAL ANALYSIS FREEZE")
print("=" * 72)

print("\nEvidence directory:")
print(FINAL_DIR)

print("\nSaved artifacts:")
for p in sorted(FINAL_DIR.iterdir()):
    print(" ", p.name)

print("\nPrimary dose-response:")
display(primary_results)

print("\nPaired slope comparison:")
display(paired_result)

print("\nCompleteness sensitivity:")
display(sensitivity_results)

print("\nProduction QA:")
display(qa_summary)

print("\nSTATUS: T1.4c ANALYSIS FROZEN")

T1.4c — FINAL ANALYSIS FREEZE

Evidence directory:
/kaggle/working/T1_4c_FINAL_EVIDENCE

Saved artifacts:
  T1_4c_FINAL_ANALYSIS_RECORD.json
  T1_4c_completeness_retention_by_rho.csv
  T1_4c_completeness_sensitivity.csv
  T1_4c_paired_slope_comparison.csv
  T1_4c_primary_model_dose_response.csv
  T1_4c_production_QA_summary.csv

Primary dose-response:


,endpoint,n_models,spearman_r,permutation_p_one_sided,permutation_p_two_sided,linear_slope,slope_ci_low,slope_ci_high,null_mean,null_sd,null_min,null_max
0,Gaussian enrichment,30,0.977542,0.00005,0.00005,8.097502,7.232369,8.804265,0.000383,0.187506,-0.679882,0.650567
1,Black enrichment,30,0.961757,0.00005,0.00005,0.257555,0.204590,0.306434,0.000632,0.187723,-0.639292,0.704687
2,Gaussian minus black,30,0.977542,0.00005,0.00005,7.839947,6.990993,8.529182,0.000406,0.187600,-0.675372,0.641547



Paired slope comparison:


,n_models,observed_difference_slope,spearman_r_difference,signflip_p_one_sided,signflip_p_two_sided,bootstrap_ci_low,bootstrap_ci_high,bootstrap_probability_positive
0,30,7.839947,0.977542,0.082646,0.167342,6.997944,8.535221,1.0



Completeness sensitivity:


,baseline,n_image_rows,n_models,min_images_per_model,max_images_per_model,spearman_r,linear_slope,bootstrap_ci_low,bootstrap_ci_high,bootstrap_probability_positive
0,Gaussian,345,30,5,14,0.945972,8.095152,7.146013,8.961318,1.0
1,Black,288,30,6,13,0.950482,0.330318,0.246080,0.411556,1.0
2,Gaussian − Black,240,30,5,12,0.957247,7.866020,6.521254,8.946100,1.0



Production QA:


,production_rows,models,rho_levels,images,gaussian_undefined,black_undefined,gaussian_completeness_failures,black_completeness_failures,gaussian_pass_rows,black_pass_rows,both_pass_rows,gaussian_mean_enrichment,black_mean_enrichment,mean_absolute_difference,mean_relative_difference_pct
0,450,30,6,15,0,0,105,162,345,288,240,9.808741,0.215778,9.592963,97.082126



STATUS: T1.4c ANALYSIS FROZEN
